# NB_P2_13: App-Relevant CRVSE Training Without ECG-Fitness

## Research question

The notebook tests whether a live-compatible CRVSE PhysFormer can improve heart-rate estimation for app-relevant still/seated webcam rPPG conditions when ECG-Fitness is excluded from model selection.

## App scope

The experiment treats the live demo as a still/seated webcam rPPG research demo.

The experiment does not claim robustness for exercise, high-motion recordings, recovery physiology, or general high-HR monitoring.

ECG-Fitness is excluded because it is a small exercise/high-motion dataset and does not represent the intended first app scope. Its previous results remain scientifically useful as a stress-test finding, but it is not used as an NB13 checkpoint-adoption gate.

## Datasets used for NB13

The app-relevant training, validation, and held-out test pipeline uses:

- MCD-rPPG
- UBFC-rPPG
- UBFC-Phys

The notebook excludes ECG-Fitness from training, validation, held-out testing, and model selection.

## Model branches

The notebook tests two model-development branches:

1. Transfer learning from the current frozen multichannel CRVSE PhysFormer checkpoint.
2. Scratch training of the same CRVSE PhysFormer architecture on the app-relevant source-FPS tensors.

Both branches must use the same app-relevant train, validation, and held-out test split.

Transfer learning tests whether the existing checkpoint already contains useful general rPPG features that can adapt to the app scope.

Scratch training tests whether the app-relevant datasets are sufficient to learn a cleaner still/seated model without inheriting the previous checkpoint's broader-domain trade-offs.

## First success criterion

Before any training starts, the notebook must reproduce the three-dataset source-FPS frozen baseline.

Expected app-relevant rows:

```text
finetune_train: 12,503
finetune_val:    2,839
finetune_test:   2,828
total:          18,170

In [1]:
import json, math, random, sys, h5py, torch, os, copy
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")

LIVE_COMPATIBLE_DIR = Path(
    "/kaggle/input/datasets/cezarytubacki/live-compatible-crvse/Live_Compatible_CRVSE"
)
RPPG_ENSEMBLE_DIR = Path(
    "/kaggle/input/datasets/cezarytubacki/rppg-ensemble/rPPG ensemble"
)

NB13_WORKING_DIR = KAGGLE_WORKING / "crvse_nb13_app_relevant_without_ecg_fitness"
NB13_WORKING_DIR.mkdir(parents=True, exist_ok=True)

APP_DATASETS = ["mcd_rppg", "ubfc_rppg", "ubfc_phys"]
EXCLUDED_DATASET = "ecg_fitness"
APP_ROLES = ["finetune_train", "finetune_val", "finetune_test"]
SOURCEFPS_MODE = "training_buffer_source_fps"

EXPECTED_APP_ROLE_COUNTS = {
    "finetune_train": 12503,
    "finetune_val": 2839,
    "finetune_test": 2828,
}

EXPECTED_SOURCEFPS_APP_MAE = {
    "finetune_train": 5.8236,
    "finetune_val": 6.0275,
    "finetune_test": 5.7696,
}

BASELINE_TOLERANCE_BPM = 0.03


def resolve_artifact(filename: str, preferred_path: Path) -> Path:
    """Resolve an artifact path, preferring the known Kaggle dataset layout."""
    if preferred_path.exists():
        return preferred_path

    candidates = []
    for root in [KAGGLE_INPUT, KAGGLE_WORKING]:
        if root.exists():
            candidates.extend(path for path in root.rglob(filename) if path.is_file())

    candidates = sorted({path.resolve() for path in candidates})

    if len(candidates) == 1:
        return candidates[0]

    if len(candidates) == 0:
        raise FileNotFoundError(f"Could not find required artifact: {filename}")

    raise RuntimeError(
        "Found multiple candidates for artifact "
        f"{filename}. Candidate paths:\n" + "\n".join(str(path) for path in candidates)
    )


def parse_bool(value) -> bool:
    """Parse bool-like values loaded from CSV."""
    if isinstance(value, bool):
        return value
    if pd.isna(value):
        return False
    return str(value).strip().lower() in {"true", "1", "yes", "y"}


def q90(values: pd.Series) -> float:
    """Return the 90th percentile."""
    return float(values.quantile(0.90))


def summarize_errors(data: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    """Summarize frozen baseline errors for NB13 app-relevant rows."""
    summary = (
        data.groupby(group_cols, dropna=False)
        .agg(
            rows=("model_abs_error", "size"),
            target_hr_mean=("target_hr_mean", "mean"),
            model_mae_mean=("model_abs_error", "mean"),
            model_mae_median=("model_abs_error", "median"),
            model_mae_p90=("model_abs_error", q90),
            signed_error_mean=("model_signed_error", "mean"),
            severe_error_15bpm_rate=("model_abs_error", lambda values: (values >= 15.0).mean()),
            spectral_valid_rows=("spectral_abs_error", lambda values: values.notna().sum()),
            spectral_mae_mean=("spectral_abs_error", "mean"),
        )
        .reset_index()
    )

    numeric_cols = summary.select_dtypes(include=[np.number]).columns
    summary[numeric_cols] = summary[numeric_cols].round(4)
    return summary


MANIFEST_PATH = resolve_artifact(
    "live_finetune_manifest.csv",
    LIVE_COMPATIBLE_DIR / "live_finetune_manifest.csv",
)
BASELINE_SUMMARY_PATH = resolve_artifact(
    "live_finetune_frozen_baseline_summary.csv",
    LIVE_COMPATIBLE_DIR / "live_finetune_frozen_baseline_summary.csv",
)
BASELINE_PREDICTIONS_PATH = resolve_artifact(
    "live_finetune_frozen_baseline_predictions.csv",
    LIVE_COMPATIBLE_DIR / "live_finetune_frozen_baseline_predictions.csv",
)
CHECKPOINT_PATH = resolve_artifact(
    "CRVSETransformer_Ensemble_physformer_multichannel_best.pt",
    LIVE_COMPATIBLE_DIR / "CRVSETransformer_Ensemble_physformer_multichannel_best.pt",
)

H5_PATHS = {
    "mcd_rppg": resolve_artifact(
        "mcd_rppg_ensemble.h5",
        RPPG_ENSEMBLE_DIR / "mcd_rppg_ensemble.h5",
    ),
    "ubfc_rppg": resolve_artifact(
        "ubfc_rppg_ensemble.h5",
        RPPG_ENSEMBLE_DIR / "ubfc_rppg_ensemble.h5",
    ),
    "ubfc_phys": resolve_artifact(
        "ubfc_phys_ensemble.h5",
        RPPG_ENSEMBLE_DIR / "ubfc_phys_ensemble.h5",
    ),
}

artifact_report = pd.DataFrame(
    [
        {"artifact": "manifest", "path": str(MANIFEST_PATH), "exists": MANIFEST_PATH.exists()},
        {"artifact": "baseline_summary", "path": str(BASELINE_SUMMARY_PATH), "exists": BASELINE_SUMMARY_PATH.exists()},
        {"artifact": "baseline_predictions", "path": str(BASELINE_PREDICTIONS_PATH), "exists": BASELINE_PREDICTIONS_PATH.exists()},
        {"artifact": "checkpoint", "path": str(CHECKPOINT_PATH), "exists": CHECKPOINT_PATH.exists()},
        *[
            {"artifact": f"h5_{name}", "path": str(path), "exists": path.exists()}
            for name, path in H5_PATHS.items()
        ],
    ]
)

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"h5py: {h5py.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")
print(f"NB13 working directory: {NB13_WORKING_DIR}")

print("\nArtifact report:")
display(artifact_report)

manifest = pd.read_csv(MANIFEST_PATH)
manifest["include_in_finetune_bool"] = manifest["include_in_finetune"].map(parse_bool)
manifest["include_in_eval_bool"] = manifest["include_in_eval"].map(parse_bool)

app_manifest = manifest[
    manifest["dataset"].isin(APP_DATASETS)
    & manifest["window_role"].isin(APP_ROLES)
].copy()

app_role_counts = (
    app_manifest["window_role"]
    .value_counts()
    .rename_axis("window_role")
    .reset_index(name="actual_rows")
)

expected_counts = pd.DataFrame(
    [
        {"window_role": role, "expected_rows": expected}
        for role, expected in EXPECTED_APP_ROLE_COUNTS.items()
    ]
)

app_role_check = expected_counts.merge(app_role_counts, on="window_role", how="left")
app_role_check["actual_rows"] = app_role_check["actual_rows"].fillna(0).astype(int)
app_role_check["matches_expected"] = (
    app_role_check["actual_rows"] == app_role_check["expected_rows"]
)

dataset_role_counts = (
    app_manifest.groupby(["dataset", "window_role"], dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values(["dataset", "window_role"])
)

ecg_policy_check = {
    "ecg_rows_in_nb13_app_manifest": int((app_manifest["dataset"] == EXCLUDED_DATASET).sum()),
    "ecg_fitness_finetune_rows_in_full_manifest": int(
        manifest[
            (manifest["dataset"] == EXCLUDED_DATASET)
            & manifest["include_in_finetune_bool"]
        ].shape[0]
    ),
    "nb13_app_rows": int(len(app_manifest)),
}

print("\nNB13 app-scope role check:")
display(app_role_check)

print("\nNB13 rows by dataset and role:")
display(dataset_role_counts)

print("\nECG-Fitness exclusion policy check:")
print(json.dumps(ecg_policy_check, indent=2))

if not app_role_check["matches_expected"].all():
    raise AssertionError("NB13 app-relevant role counts do not match expected counts.")

if ecg_policy_check["ecg_rows_in_nb13_app_manifest"] != 0:
    raise AssertionError("ECG-Fitness leaked into the NB13 app-relevant manifest.")

if ecg_policy_check["ecg_fitness_finetune_rows_in_full_manifest"] != 0:
    raise AssertionError("ECG-Fitness appears in fine-tuning rows in the full manifest.")

baseline_predictions = pd.read_csv(BASELINE_PREDICTIONS_PATH, low_memory=False)

for col in [
    "target_hr_mean",
    "model_hr",
    "model_abs_error",
    "spectral_abs_error",
    "spectral_spread_bpm",
]:
    baseline_predictions[col] = pd.to_numeric(baseline_predictions[col], errors="coerce")

sourcefps_app = baseline_predictions[
    (baseline_predictions["dataset"].isin(APP_DATASETS))
    & (baseline_predictions["window_role"].isin(APP_ROLES))
    & (baseline_predictions["preprocessing_mode"] == SOURCEFPS_MODE)
    & (baseline_predictions["preprocessing_status"] == "ok")
].copy()

sourcefps_app["model_signed_error"] = (
    sourcefps_app["model_hr"] - sourcefps_app["target_hr_mean"]
)

baseline_by_role = summarize_errors(sourcefps_app, ["window_role"])
baseline_by_dataset_role = summarize_errors(sourcefps_app, ["dataset", "window_role"])

expected_mae = pd.DataFrame(
    [
        {"window_role": role, "expected_model_mae_mean": mae}
        for role, mae in EXPECTED_SOURCEFPS_APP_MAE.items()
    ]
)

baseline_role_check = baseline_by_role.merge(expected_mae, on="window_role", how="left")
baseline_role_check["mae_delta_vs_expected"] = (
    baseline_role_check["model_mae_mean"]
    - baseline_role_check["expected_model_mae_mean"]
).round(4)
baseline_role_check["matches_expected"] = (
    baseline_role_check["mae_delta_vs_expected"].abs() <= BASELINE_TOLERANCE_BPM
)

print("\nNB13 frozen source-FPS baseline by role:")
display(baseline_role_check)

print("\nNB13 frozen source-FPS baseline by dataset and role:")
display(baseline_by_dataset_role.sort_values(["window_role", "dataset"]))

if not baseline_role_check["matches_expected"].all():
    raise AssertionError("Frozen source-FPS app baseline does not match expected NB13 baseline.")

NB13_CONTEXT = {
    "seed": SEED,
    "app_datasets": APP_DATASETS,
    "excluded_dataset": EXCLUDED_DATASET,
    "sourcefps_mode": SOURCEFPS_MODE,
    "manifest_path": str(MANIFEST_PATH),
    "baseline_predictions_path": str(BASELINE_PREDICTIONS_PATH),
    "checkpoint_path": str(CHECKPOINT_PATH),
    "working_dir": str(NB13_WORKING_DIR),
}

print("\nPASS: NB13 app-relevant scope and frozen source-FPS baseline are reproduced.")
print(json.dumps(NB13_CONTEXT, indent=2))

Python: 3.12.13
NumPy: 2.0.2
Pandas: 2.3.3
h5py: 3.16.0
PyTorch: 2.10.0+cu128
CUDA available: True
Device name: Tesla T4
NB13 working directory: /kaggle/working/crvse_nb13_app_relevant_without_ecg_fitness

Artifact report:


,artifact,path,exists
0,manifest,/kaggle/input/datasets/cezarytubacki/live-comp...,True
1,baseline_summary,/kaggle/input/datasets/cezarytubacki/live-comp...,True
2,baseline_predictions,/kaggle/input/datasets/cezarytubacki/live-comp...,True
3,checkpoint,/kaggle/input/datasets/cezarytubacki/live-comp...,True
4,h5_mcd_rppg,/kaggle/input/datasets/cezarytubacki/rppg-ense...,True
5,h5_ubfc_rppg,/kaggle/input/datasets/cezarytubacki/rppg-ense...,True
6,h5_ubfc_phys,/kaggle/input/datasets/cezarytubacki/rppg-ense...,True



NB13 app-scope role check:


,window_role,expected_rows,actual_rows,matches_expected
0,finetune_train,12503,12503,True
1,finetune_val,2839,2839,True
2,finetune_test,2828,2828,True



NB13 rows by dataset and role:


...


ECG-Fitness exclusion policy check:
{
  "ecg_rows_in_nb13_app_manifest": 0,
  "ecg_fitness_finetune_rows_in_full_manifest": 0,
  "nb13_app_rows": 18170
}

NB13 frozen source-FPS baseline by role:


,window_role,rows,target_hr_mean,model_mae_mean,model_mae_median,model_mae_p90,signed_error_mean,severe_error_15bpm_rate,spectral_valid_rows,spectral_mae_mean,expected_model_mae_mean,mae_delta_vs_expected,matches_expected
0,finetune_test,2828,79.7822,5.7696,3.6274,13.6984,1.4174,0.0831,2820,10.9565,5.7696,0.0,True
1,finetune_train,12503,80.2370,5.8236,3.7437,13.7192,1.5658,0.0846,12483,11.3589,5.8236,0.0,True
2,finetune_val,2839,80.1617,6.0275,3.8533,14.0701,1.4724,0.0891,2833,10.5942,6.0275,0.0,True



NB13 frozen source-FPS baseline by dataset and role:


,dataset,window_role,rows,target_hr_mean,model_mae_mean,model_mae_median,model_mae_p90,signed_error_mean,severe_error_15bpm_rate,spectral_valid_rows,spectral_mae_mean
0,mcd_rppg,finetune_test,2094,79.8533,4.8847,3.1502,10.9700,0.9242,0.0525,2089,10.2928
3,ubfc_phys,finetune_test,658,77.4205,8.8584,5.6735,20.2862,3.1980,0.1900,655,13.9428
6,ubfc_rppg,finetune_test,76,98.2693,3.4085,3.1453,6.0835,-0.4102,0.0000,76,3.4633
1,mcd_rppg,finetune_train,7446,79.0648,4.6876,3.1639,10.3722,1.2645,0.0431,7428,10.5998
4,ubfc_phys,finetune_train,4595,80.3055,7.7819,5.4248,18.5928,2.3192,0.1567,4593,13.2336
7,ubfc_rppg,finetune_train,462,98.4492,4.6548,3.1447,9.9189,-1.0723,0.0368,462,4.9265
2,mcd_rppg,finetune_val,1743,79.1086,5.1325,3.3746,11.5368,1.2656,0.0545,1739,10.7898
5,ubfc_phys,finetune_val,986,80.0024,7.7629,4.8461,18.2725,2.2826,0.1531,984,10.9255
8,ubfc_rppg,finetune_val,110,98.2772,4.6533,2.4438,12.3774,-2.5126,0.0636,110,4.5386



PASS: NB13 app-relevant scope and frozen source-FPS baseline are reproduced.
{
  "seed": 42,
  "app_datasets": [
    "mcd_rppg",
    "ubfc_rppg",
    "ubfc_phys"
  ],
  "excluded_dataset": "ecg_fitness",
  "sourcefps_mode": "training_buffer_source_fps",
  "manifest_path": "/kaggle/input/datasets/cezarytubacki/live-compatible-crvse/Live_Compatible_CRVSE/live_finetune_manifest.csv",
  "baseline_predictions_path": "/kaggle/input/datasets/cezarytubacki/live-compatible-crvse/Live_Compatible_CRVSE/live_finetune_frozen_baseline_predictions.csv",
  "checkpoint_path": "/kaggle/input/datasets/cezarytubacki/live-compatible-crvse/Live_Compatible_CRVSE/CRVSETransformer_Ensemble_physformer_multichannel_best.pt",
  "working_dir": "/kaggle/working/crvse_nb13_app_relevant_without_ecg_fitness"
}


## NB13 Candidate Decision Policy

This section defines how NB13 candidate checkpoints are judged before any training starts.

The frozen source-FPS CRVSE PhysFormer remains the reference checkpoint.

The notebook compares three model states:

1. The frozen source-FPS baseline.
2. A transfer-learning candidate initialized from the frozen checkpoint.
3. A scratch-trained CRVSE PhysFormer candidate initialized from random weights.

The selection target is app-relevant still/seated webcam rPPG performance on MCD-rPPG, UBFC-rPPG, and UBFC-Phys.

A candidate is not accepted only because validation MAE improves. The candidate must also preserve held-out test behavior, per-dataset behavior, p90 error, severe-error rate, and signed bias.

UBFC-Phys receives special attention because it is the weakest app-relevant dataset in the frozen baseline. A candidate that improves the aggregate mean while worsening UBFC-Phys is treated as a trade-off, not an app checkpoint.

In [2]:
NB13_DECISION_THRESHOLDS = {
    "minimum_val_gain_bpm": 0.20,
    "minimum_test_gain_bpm": 0.10,
    "maximum_dataset_test_regression_bpm": 0.30,
    "maximum_ubfc_phys_test_regression_bpm": 0.10,
    "maximum_p90_regression_bpm": 0.25,
    "maximum_severe_error_rate_increase": 0.01,
    "maximum_abs_signed_bias_increase_bpm": 0.25,
    "severe_error_threshold_bpm": 15.0,
    "descriptive_high_hr_threshold_bpm": 100.0,
}


def require_nb13_baseline_context() -> None:
    """Verify that the NB13 scope and frozen baseline cell has been run."""
    required_names = [
        "sourcefps_app",
        "APP_DATASETS",
        "APP_ROLES",
        "NB13_WORKING_DIR",
        "NB13_CONTEXT",
    ]

    missing = [name for name in required_names if name not in globals()]

    if missing:
        raise NameError(
            "Run the NB13 scope and frozen baseline cell before this cell. "
            f"Missing names: {missing}"
        )


def prepare_nb13_error_table(
    predictions: pd.DataFrame,
    experiment_name: str,
    target_col: str,
    pred_col: str,
    role_col: str = "window_role",
) -> pd.DataFrame:
    """Prepare a standard NB13 error table for frozen, transfer, or scratch predictions."""
    required_cols = ["dataset", role_col, target_col, pred_col]
    missing_cols = [col for col in required_cols if col not in predictions.columns]

    if missing_cols:
        raise KeyError(
            f"Prediction table is missing required columns: {missing_cols}. "
            f"Available columns: {list(predictions.columns)}"
        )

    table = predictions.copy()
    table["experiment"] = experiment_name
    table["window_role"] = table[role_col].astype(str)
    table["target_hr"] = pd.to_numeric(table[target_col], errors="coerce")
    table["pred_hr"] = pd.to_numeric(table[pred_col], errors="coerce")
    table["signed_error"] = table["pred_hr"] - table["target_hr"]
    table["abs_error"] = table["signed_error"].abs()

    table["is_severe_error_15bpm"] = (
        table["abs_error"] >= NB13_DECISION_THRESHOLDS["severe_error_threshold_bpm"]
    )
    table["is_high_hr_ge_100"] = (
        table["target_hr"] >= NB13_DECISION_THRESHOLDS["descriptive_high_hr_threshold_bpm"]
    )

    table = table[
        table["dataset"].isin(APP_DATASETS)
        & table["window_role"].isin(APP_ROLES)
    ].copy()

    if table["target_hr"].isna().any() or table["pred_hr"].isna().any():
        raise ValueError("NB13 error table contains missing target or prediction values.")

    return table


def summarize_nb13_errors(error_table: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    """Summarize NB13 candidate errors by role or dataset-role."""
    summary = (
        error_table
        .groupby(group_cols, dropna=False)
        .agg(
            rows=("abs_error", "size"),
            target_hr_mean=("target_hr", "mean"),
            pred_hr_mean=("pred_hr", "mean"),
            mae_mean=("abs_error", "mean"),
            mae_median=("abs_error", "median"),
            mae_p90=("abs_error", lambda values: values.quantile(0.90)),
            mae_p95=("abs_error", lambda values: values.quantile(0.95)),
            signed_error_mean=("signed_error", "mean"),
            abs_signed_bias=("signed_error", lambda values: abs(values.mean())),
            severe_error_15bpm_rate=("is_severe_error_15bpm", "mean"),
            high_hr_ge_100_rows=("is_high_hr_ge_100", "sum"),
        )
        .reset_index()
    )

    numeric_cols = summary.select_dtypes(include=[np.number]).columns
    summary[numeric_cols] = summary[numeric_cols].round(4)
    return summary


def metric_from_role(summary: pd.DataFrame, role: str, metric: str) -> float:
    """Extract one metric from a role-level summary table."""
    rows = summary.loc[summary["window_role"] == role, metric]

    if len(rows) != 1:
        raise ValueError(f"Expected one row for role={role}, metric={metric}; found {len(rows)}.")

    return float(rows.iloc[0])


def compare_candidate_to_frozen(
    candidate_error_table: pd.DataFrame,
    candidate_name: str,
    frozen_error_table: pd.DataFrame,
) -> dict:
    """Compare one NB13 candidate against the frozen source-FPS baseline."""
    candidate_role = summarize_nb13_errors(candidate_error_table, ["window_role"])
    frozen_role = summarize_nb13_errors(frozen_error_table, ["window_role"])

    candidate_dataset_role = summarize_nb13_errors(candidate_error_table, ["dataset", "window_role"])
    frozen_dataset_role = summarize_nb13_errors(frozen_error_table, ["dataset", "window_role"])

    role_compare = candidate_role.merge(
        frozen_role,
        on="window_role",
        suffixes=("_candidate", "_frozen"),
    )

    dataset_role_compare = candidate_dataset_role.merge(
        frozen_dataset_role,
        on=["dataset", "window_role"],
        suffixes=("_candidate", "_frozen"),
    )

    for metric in [
        "mae_mean",
        "mae_median",
        "mae_p90",
        "mae_p95",
        "signed_error_mean",
        "abs_signed_bias",
        "severe_error_15bpm_rate",
    ]:
        role_compare[f"delta_{metric}"] = (
            role_compare[f"{metric}_candidate"] - role_compare[f"{metric}_frozen"]
        ).round(4)

        dataset_role_compare[f"delta_{metric}"] = (
            dataset_role_compare[f"{metric}_candidate"] - dataset_role_compare[f"{metric}_frozen"]
        ).round(4)

    test_dataset_compare = dataset_role_compare[
        dataset_role_compare["window_role"] == "finetune_test"
    ].copy()

    ubfc_phys_test = test_dataset_compare[
        test_dataset_compare["dataset"] == "ubfc_phys"
    ]

    decision_row = {
        "candidate": candidate_name,
        "val_delta_mae_mean": metric_from_role(role_compare, "finetune_val", "delta_mae_mean"),
        "test_delta_mae_mean": metric_from_role(role_compare, "finetune_test", "delta_mae_mean"),
        "val_delta_mae_p90": metric_from_role(role_compare, "finetune_val", "delta_mae_p90"),
        "test_delta_mae_p90": metric_from_role(role_compare, "finetune_test", "delta_mae_p90"),
        "test_delta_severe_error_rate": metric_from_role(
            role_compare,
            "finetune_test",
            "delta_severe_error_15bpm_rate",
        ),
        "test_delta_abs_signed_bias": metric_from_role(
            role_compare,
            "finetune_test",
            "delta_abs_signed_bias",
        ),
        "max_dataset_test_delta_mae": float(test_dataset_compare["delta_mae_mean"].max()),
        "ubfc_phys_test_delta_mae": float(ubfc_phys_test["delta_mae_mean"].iloc[0]),
    }

    decision_row["decision_status"] = assign_nb13_candidate_status(decision_row)

    return {
        "decision_row": decision_row,
        "role_compare": role_compare,
        "dataset_role_compare": dataset_role_compare,
        "candidate_role_summary": candidate_role,
        "candidate_dataset_role_summary": candidate_dataset_role,
    }


def assign_nb13_candidate_status(decision_row: dict) -> str:
    """Assign an NB13 candidate status from predefined decision thresholds."""
    val_gain = -float(decision_row["val_delta_mae_mean"])
    test_gain = -float(decision_row["test_delta_mae_mean"])

    improves_validation = val_gain >= NB13_DECISION_THRESHOLDS["minimum_val_gain_bpm"]
    improves_test = test_gain >= NB13_DECISION_THRESHOLDS["minimum_test_gain_bpm"]

    guardrails_pass = (
        decision_row["max_dataset_test_delta_mae"]
        <= NB13_DECISION_THRESHOLDS["maximum_dataset_test_regression_bpm"]
        and decision_row["ubfc_phys_test_delta_mae"]
        <= NB13_DECISION_THRESHOLDS["maximum_ubfc_phys_test_regression_bpm"]
        and decision_row["test_delta_mae_p90"]
        <= NB13_DECISION_THRESHOLDS["maximum_p90_regression_bpm"]
        and decision_row["test_delta_severe_error_rate"]
        <= NB13_DECISION_THRESHOLDS["maximum_severe_error_rate_increase"]
        and decision_row["test_delta_abs_signed_bias"]
        <= NB13_DECISION_THRESHOLDS["maximum_abs_signed_bias_increase_bpm"]
    )

    if improves_validation and improves_test and guardrails_pass:
        return "adoption_candidate_for_review"

    if improves_validation and improves_test:
        return "analysis_only_tradeoff"

    if improves_validation or improves_test:
        return "weak_or_incomplete_candidate"

    return "reject"


require_nb13_baseline_context()

FROZEN_NB13_ERROR_TABLE = prepare_nb13_error_table(
    predictions=sourcefps_app,
    experiment_name="frozen_sourcefps_baseline",
    target_col="target_hr_mean",
    pred_col="model_hr",
)

FROZEN_NB13_ROLE_SUMMARY = summarize_nb13_errors(
    FROZEN_NB13_ERROR_TABLE,
    ["window_role"],
)

FROZEN_NB13_DATASET_ROLE_SUMMARY = summarize_nb13_errors(
    FROZEN_NB13_ERROR_TABLE,
    ["dataset", "window_role"],
)

decision_policy_table = pd.DataFrame(
    [
        {"threshold": key, "value": value}
        for key, value in NB13_DECISION_THRESHOLDS.items()
    ]
)

policy_path = NB13_WORKING_DIR / "nb13_candidate_decision_policy.json"
policy_payload = {
    "context": NB13_CONTEXT,
    "thresholds": NB13_DECISION_THRESHOLDS,
    "frozen_role_summary": json.loads(FROZEN_NB13_ROLE_SUMMARY.to_json(orient="records")),
    "frozen_dataset_role_summary": json.loads(
        FROZEN_NB13_DATASET_ROLE_SUMMARY.to_json(orient="records")
    ),
}

policy_path.write_text(json.dumps(policy_payload, indent=2), encoding="utf-8")

print("NB13 decision thresholds:")
display(decision_policy_table)

print("\nFrozen NB13 role summary:")
display(FROZEN_NB13_ROLE_SUMMARY)

print("\nFrozen NB13 dataset-role summary:")
display(FROZEN_NB13_DATASET_ROLE_SUMMARY.sort_values(["window_role", "dataset"]))

print(f"\nSaved decision policy: {policy_path}")
print("PASS: NB13 decision policy is defined before transfer or scratch training.")

NB13 decision thresholds:


,threshold,value
0,minimum_val_gain_bpm,0.20
1,minimum_test_gain_bpm,0.10
2,maximum_dataset_test_regression_bpm,0.30
3,maximum_ubfc_phys_test_regression_bpm,0.10
4,maximum_p90_regression_bpm,0.25
5,maximum_severe_error_rate_increase,0.01
6,maximum_abs_signed_bias_increase_bpm,0.25
7,severe_error_threshold_bpm,15.00
8,descriptive_high_hr_threshold_bpm,100.00



Frozen NB13 role summary:


,window_role,rows,target_hr_mean,pred_hr_mean,mae_mean,mae_median,mae_p90,mae_p95,signed_error_mean,abs_signed_bias,severe_error_15bpm_rate,high_hr_ge_100_rows
0,finetune_test,2828,79.7822,81.1995,5.7696,3.6274,13.6984,19.0461,1.4174,1.4174,0.0831,244
1,finetune_train,12503,80.2370,81.8028,5.8236,3.7437,13.7192,18.9426,1.5658,1.5658,0.0846,1129
2,finetune_val,2839,80.1617,81.6342,6.0275,3.8533,14.0701,19.2319,1.4724,1.4724,0.0891,168



Frozen NB13 dataset-role summary:


,dataset,window_role,rows,target_hr_mean,pred_hr_mean,mae_mean,mae_median,mae_p90,mae_p95,signed_error_mean,abs_signed_bias,severe_error_15bpm_rate,high_hr_ge_100_rows
0,mcd_rppg,finetune_test,2094,79.8533,80.7775,4.8847,3.1502,10.9700,15.5490,0.9242,0.9242,0.0525,192
3,ubfc_phys,finetune_test,658,77.4205,80.6185,8.8584,5.6735,20.2862,25.6733,3.1980,3.1980,0.1900,7
6,ubfc_rppg,finetune_test,76,98.2693,97.8591,3.4085,3.1453,6.0835,6.8394,-0.4102,0.4102,0.0000,45
1,mcd_rppg,finetune_train,7446,79.0648,80.3294,4.6876,3.1639,10.3722,13.9969,1.2645,1.2645,0.0431,569
4,ubfc_phys,finetune_train,4595,80.3055,82.6246,7.7819,5.4248,18.5928,23.4842,2.3192,2.3192,0.1567,367
7,ubfc_rppg,finetune_train,462,98.4492,97.3769,4.6548,3.1447,9.9189,13.1687,-1.0723,1.0723,0.0368,193
2,mcd_rppg,finetune_val,1743,79.1086,80.3743,5.1325,3.3746,11.5368,15.4784,1.2656,1.2656,0.0545,79
5,ubfc_phys,finetune_val,986,80.0024,82.2850,7.7629,4.8461,18.2725,24.2656,2.2826,2.2826,0.1531,19
8,ubfc_rppg,finetune_val,110,98.2772,95.7647,4.6533,2.4438,12.3774,15.4017,-2.5126,2.5126,0.0636,70



Saved decision policy: /kaggle/working/crvse_nb13_app_relevant_without_ecg_fitness/nb13_candidate_decision_policy.json
PASS: NB13 decision policy is defined before transfer or scratch training.


## Source-FPS Tensor Cache Verification

This section verifies the materialized source-FPS tensor caches used by NB13.

The same tensors will be used for both transfer learning and scratch training.

The purpose of this section is to confirm:

- train, validation, and held-out test tensor files exist,
- tensor shapes match the CRVSE PhysFormer input contract,
- metadata rows align with tensor rows,
- target labels are finite and physiologically plausible,
- ECG-Fitness is absent from all NB13 training and model-selection tensors,
- and dataset/role counts match the app-relevant manifest.

This is still an audit step. No model training starts in this section.

In [3]:
TENSOR_ARTIFACTS = {
    "finetune_train": {
        "tensor_filename": "finetune_train_sourcefps_tensors.npz",
        "metadata_filename": "finetune_train_sourcefps_metadata.csv",
        "expected_rows": 12503,
    },
    "finetune_val": {
        "tensor_filename": "finetune_val_sourcefps_tensors.npz",
        "metadata_filename": "finetune_val_sourcefps_metadata.csv",
        "expected_rows": 2839,
    },
    "finetune_test": {
        "tensor_filename": "finetune_test_sourcefps_tensors.npz",
        "metadata_filename": "finetune_test_sourcefps_metadata.csv",
        "expected_rows": 2828,
    },
}


def load_sourcefps_tensor_cache(role: str, spec: dict) -> dict:
    """Load one source-FPS tensor cache and metadata table."""
    tensor_path = resolve_artifact(
        spec["tensor_filename"],
        LIVE_COMPATIBLE_DIR / spec["tensor_filename"],
    )
    metadata_path = resolve_artifact(
        spec["metadata_filename"],
        LIVE_COMPATIBLE_DIR / spec["metadata_filename"],
    )

    loaded = np.load(tensor_path)

    missing_arrays = {"x", "y"} - set(loaded.files)
    if missing_arrays:
        raise KeyError(f"{tensor_path} is missing arrays: {sorted(missing_arrays)}")

    x = loaded["x"].astype(np.float32, copy=False)
    y = loaded["y"].astype(np.float32, copy=False)
    metadata = pd.read_csv(metadata_path)

    return {
        "role": role,
        "tensor_path": tensor_path,
        "metadata_path": metadata_path,
        "x": x,
        "y": y,
        "metadata": metadata,
        "expected_rows": int(spec["expected_rows"]),
    }


def summarize_sourcefps_tensor_cache(cache_entry: dict) -> dict:
    """Summarize one loaded NB13 tensor cache."""
    role = cache_entry["role"]
    x = cache_entry["x"]
    y = cache_entry["y"]
    metadata = cache_entry["metadata"]
    expected_rows = cache_entry["expected_rows"]

    dataset_counts = (
        metadata["dataset"].value_counts().to_dict()
        if "dataset" in metadata.columns
        else {}
    )

    role_values = (
        sorted(metadata["window_role"].dropna().unique().tolist())
        if "window_role" in metadata.columns
        else []
    )

    ecg_rows = (
        int((metadata["dataset"] == EXCLUDED_DATASET).sum())
        if "dataset" in metadata.columns
        else -1
    )

    summary = {
        "role": role,
        "expected_rows": expected_rows,
        "x_shape": tuple(x.shape),
        "y_shape": tuple(y.shape),
        "metadata_rows": int(len(metadata)),
        "x_dtype": str(x.dtype),
        "y_dtype": str(y.dtype),
        "x_finite": bool(np.isfinite(x).all()),
        "y_finite": bool(np.isfinite(y).all()),
        "row_count_ok": bool(x.shape[0] == y.shape[0] == len(metadata) == expected_rows),
        "shape_ok": bool(x.ndim == 3 and x.shape[1] == 3 and x.shape[2] == 240),
        "target_hr_min": float(np.min(y)),
        "target_hr_mean": float(np.mean(y)),
        "target_hr_max": float(np.max(y)),
        "role_values": role_values,
        "role_ok": role_values == [role],
        "dataset_counts": dataset_counts,
        "ecg_rows": ecg_rows,
        "ecg_excluded_ok": ecg_rows == 0,
    }

    if "target_hr_mean" in metadata.columns:
        metadata_target = pd.to_numeric(metadata["target_hr_mean"], errors="coerce").to_numpy()
        if len(metadata_target) == len(y):
            summary["max_abs_y_vs_metadata_target"] = float(
                np.nanmax(np.abs(y - metadata_target))
            )

    return summary


NB13_TENSOR_CACHE = {
    role: load_sourcefps_tensor_cache(role, spec)
    for role, spec in TENSOR_ARTIFACTS.items()
}

tensor_cache_summary = pd.DataFrame(
    [
        summarize_sourcefps_tensor_cache(cache_entry)
        for cache_entry in NB13_TENSOR_CACHE.values()
    ]
)

tensor_dataset_role_counts = []

for role, cache_entry in NB13_TENSOR_CACHE.items():
    metadata = cache_entry["metadata"].copy()

    if "dataset" in metadata.columns and "window_role" in metadata.columns:
        counts = (
            metadata
            .groupby(["dataset", "window_role"], dropna=False)
            .size()
            .reset_index(name="rows")
        )
        counts["cache_role"] = role
        tensor_dataset_role_counts.append(counts)

tensor_dataset_role_counts = (
    pd.concat(tensor_dataset_role_counts, ignore_index=True)
    if tensor_dataset_role_counts
    else pd.DataFrame()
)

print("NB13 source-FPS tensor cache summary:")
display(tensor_cache_summary)

print("\nNB13 tensor cache rows by dataset and role:")
display(tensor_dataset_role_counts.sort_values(["cache_role", "dataset", "window_role"]))

failed_checks = []

for row in tensor_cache_summary.to_dict(orient="records"):
    role = row["role"]

    for check_name in [
        "x_finite",
        "y_finite",
        "row_count_ok",
        "shape_ok",
        "role_ok",
        "ecg_excluded_ok",
    ]:
        if not bool(row[check_name]):
            failed_checks.append(f"{role}: {check_name}")

    if row["target_hr_min"] < 40.0 or row["target_hr_max"] > 180.0:
        failed_checks.append(f"{role}: target HR outside expected 40-180 BPM range")

    if "max_abs_y_vs_metadata_target" in row and row["max_abs_y_vs_metadata_target"] > 0.02:
        failed_checks.append(
            f"{role}: tensor y differs from metadata target_hr_mean by more than 0.02 BPM"
        )

if failed_checks:
    raise AssertionError("Tensor cache verification failed:\n" + "\n".join(failed_checks))

tensor_summary_path = NB13_WORKING_DIR / "nb13_sourcefps_tensor_cache_summary.csv"
tensor_counts_path = NB13_WORKING_DIR / "nb13_sourcefps_tensor_dataset_role_counts.csv"

tensor_cache_summary.to_csv(tensor_summary_path, index=False)
tensor_dataset_role_counts.to_csv(tensor_counts_path, index=False)

print(f"\nSaved tensor cache summary: {tensor_summary_path}")
print(f"Saved tensor dataset-role counts: {tensor_counts_path}")
print("PASS: NB13 source-FPS tensor caches are ready for transfer and scratch branches.")

NB13 source-FPS tensor cache summary:


,role,expected_rows,x_shape,y_shape,metadata_rows,x_dtype,y_dtype,x_finite,y_finite,row_count_ok,shape_ok,target_hr_min,target_hr_mean,target_hr_max,role_values,role_ok,dataset_counts,ecg_rows,ecg_excluded_ok,max_abs_y_vs_metadata_target
0,finetune_train,12503,"(12503, 3, 240)","(12503,)",12503,float32,float32,True,True,True,True,40.056313,80.237045,127.000000,[finetune_train],True,"{'mcd_rppg': 7446, 'ubfc_phys': 4595, 'ubfc_rp...",0,True,1.421085e-14
1,finetune_val,2839,"(2839, 3, 240)","(2839,)",2839,float32,float32,True,True,True,True,40.296436,80.161743,127.000000,[finetune_val],True,"{'mcd_rppg': 1743, 'ubfc_phys': 986, 'ubfc_rpp...",0,True,1.421085e-14
2,finetune_test,2828,"(2828, 3, 240)","(2828,)",2828,float32,float32,True,True,True,True,47.631721,79.782181,129.071884,[finetune_test],True,"{'mcd_rppg': 2094, 'ubfc_phys': 658, 'ubfc_rpp...",0,True,1.421085e-14



NB13 tensor cache rows by dataset and role:


,dataset,window_role,rows,cache_role
6,mcd_rppg,finetune_test,2094,finetune_test
7,ubfc_phys,finetune_test,658,finetune_test
8,ubfc_rppg,finetune_test,76,finetune_test
0,mcd_rppg,finetune_train,7446,finetune_train
1,ubfc_phys,finetune_train,4595,finetune_train
2,ubfc_rppg,finetune_train,462,finetune_train
3,mcd_rppg,finetune_val,1743,finetune_val
4,ubfc_phys,finetune_val,986,finetune_val
5,ubfc_rppg,finetune_val,110,finetune_val



Saved tensor cache summary: /kaggle/working/crvse_nb13_app_relevant_without_ecg_fitness/nb13_sourcefps_tensor_cache_summary.csv
Saved tensor dataset-role counts: /kaggle/working/crvse_nb13_app_relevant_without_ecg_fitness/nb13_sourcefps_tensor_dataset_role_counts.csv
PASS: NB13 source-FPS tensor caches are ready for transfer and scratch branches.


## Shared Dataset And DataLoader Check

This section creates the shared PyTorch datasets and dataloaders used by both NB13 model branches.

The transfer-learning branch and scratch-training branch must use the same source-FPS tensors, labels, and metadata.

This section verifies:

- tensors can be wrapped as PyTorch datasets,
- train, validation, and held-out test loaders produce the expected shapes,
- labels are finite,
- batches preserve the CRVSE PhysFormer input contract,
- and dataset metadata remains available for later per-dataset evaluation.

No model training starts in this section.

In [4]:
BATCH_SIZE = 256
EVAL_BATCH_SIZE = 512
NUM_WORKERS = os.cpu_count()


class NB13TensorDataset(Dataset):
    """Dataset backed by NB13 source-FPS tensors and metadata."""

    def __init__(self, role: str, cache_entry: dict) -> None:
        self.role = role
        self.x = cache_entry["x"]
        self.y = cache_entry["y"]
        self.metadata = cache_entry["metadata"].reset_index(drop=True).copy()

        if len(self.x) != len(self.y):
            raise ValueError(f"{role}: x/y row mismatch: {len(self.x)} vs {len(self.y)}")

        if len(self.x) != len(self.metadata):
            raise ValueError(
                f"{role}: tensor/metadata row mismatch: {len(self.x)} vs {len(self.metadata)}"
            )

    def __len__(self) -> int:
        return int(len(self.x))

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor, int]:
        x = torch.from_numpy(self.x[index]).float()
        y = torch.tensor(self.y[index], dtype=torch.float32)
        return x, y, int(index)


def make_train_loader(dataset: Dataset, batch_size: int = BATCH_SIZE) -> DataLoader:
    """Create the standard shuffled training DataLoader."""
    generator = torch.Generator()
    generator.manual_seed(SEED)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        generator=generator,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
    )


def make_eval_loader(dataset: Dataset, batch_size: int = EVAL_BATCH_SIZE) -> DataLoader:
    """Create a deterministic evaluation DataLoader."""
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
    )


NB13_DATASETS = {
    role: NB13TensorDataset(role=role, cache_entry=cache_entry)
    for role, cache_entry in NB13_TENSOR_CACHE.items()
}

NB13_LOADERS = {
    "train": make_train_loader(NB13_DATASETS["finetune_train"]),
    "val": make_eval_loader(NB13_DATASETS["finetune_val"]),
    "test": make_eval_loader(NB13_DATASETS["finetune_test"]),
}

loader_rows = []

for loader_name, loader in NB13_LOADERS.items():
    x_batch, y_batch, index_batch = next(iter(loader))

    loader_rows.append(
        {
            "loader": loader_name,
            "dataset_rows": len(loader.dataset),
            "batch_x_shape": tuple(x_batch.shape),
            "batch_y_shape": tuple(y_batch.shape),
            "batch_index_shape": tuple(index_batch.shape),
            "x_dtype": str(x_batch.dtype),
            "y_dtype": str(y_batch.dtype),
            "x_finite": bool(torch.isfinite(x_batch).all().item()),
            "y_finite": bool(torch.isfinite(y_batch).all().item()),
            "y_min": float(y_batch.min().item()),
            "y_mean": float(y_batch.mean().item()),
            "y_max": float(y_batch.max().item()),
        }
    )

loader_check = pd.DataFrame(loader_rows)

print("NB13 DataLoader batch check:")
display(loader_check)

failed_loader_checks = []

for row in loader_check.to_dict(orient="records"):
    loader = row["loader"]

    if row["batch_x_shape"][1:] != (3, 240):
        failed_loader_checks.append(f"{loader}: expected batch tensor channels/time (3, 240)")

    if not row["x_finite"]:
        failed_loader_checks.append(f"{loader}: x batch contains non-finite values")

    if not row["y_finite"]:
        failed_loader_checks.append(f"{loader}: y batch contains non-finite values")

    if row["y_min"] < 40.0 or row["y_max"] > 180.0:
        failed_loader_checks.append(f"{loader}: y batch outside expected 40-180 BPM range")

if failed_loader_checks:
    raise AssertionError("DataLoader check failed:\n" + "\n".join(failed_loader_checks))

dataset_metadata_summary = []

for role, dataset in NB13_DATASETS.items():
    metadata = dataset.metadata

    dataset_metadata_summary.append(
        {
            "role": role,
            "rows": len(dataset),
            "datasets": sorted(metadata["dataset"].unique().tolist()),
            "window_roles": sorted(metadata["window_role"].unique().tolist()),
            "target_hr_mean": float(np.mean(dataset.y)),
            "target_hr_min": float(np.min(dataset.y)),
            "target_hr_max": float(np.max(dataset.y)),
        }
    )

dataset_metadata_summary = pd.DataFrame(dataset_metadata_summary)

print("\nNB13 dataset metadata summary:")
display(dataset_metadata_summary)

print("PASS: Shared NB13 datasets and dataloaders are ready for transfer and scratch branches.")

NB13 DataLoader batch check:


,loader,dataset_rows,batch_x_shape,batch_y_shape,batch_index_shape,x_dtype,y_dtype,x_finite,y_finite,y_min,y_mean,y_max
0,train,12503,"(256, 3, 240)","(256,)","(256,)",torch.float32,torch.float32,True,True,46.667923,80.740753,127.000000
1,val,2839,"(512, 3, 240)","(512,)","(512,)",torch.float32,torch.float32,True,True,59.251266,83.150345,101.463158
2,test,2828,"(512, 3, 240)","(512,)","(512,)",torch.float32,torch.float32,True,True,61.668060,85.193321,107.084930



NB13 dataset metadata summary:


,role,rows,datasets,window_roles,target_hr_mean,target_hr_min,target_hr_max
0,finetune_train,12503,"[mcd_rppg, ubfc_phys, ubfc_rppg]",[finetune_train],80.237045,40.056313,127.000000
1,finetune_val,2839,"[mcd_rppg, ubfc_phys, ubfc_rppg]",[finetune_val],80.161743,40.296436,127.000000
2,finetune_test,2828,"[mcd_rppg, ubfc_phys, ubfc_rppg]",[finetune_test],79.782181,47.631721,129.071884


PASS: Shared NB13 datasets and dataloaders are ready for transfer and scratch branches.


## CRVSE PhysFormer Checkpoint Smoke Test

This section rebuilds the CRVSE PhysFormer architecture used by the current app checkpoint.

The purpose is to verify that NB13 can:

- instantiate the checkpoint-compatible architecture,
- load the frozen multichannel PhysFormer weights,
- run inference on NB13 source-FPS tensors,
- produce one prediction per input window,
- and keep the same input contract used by the live demo.

This section does not train the model. It only verifies that the frozen checkpoint can run on the NB13 app-relevant tensor loaders.

In [5]:
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding for transformer time tokens."""

    def __init__(self, d_model: int, max_len: int = 300, dropout: float = 0.1) -> None:
        super().__init__()

        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term[: pe[:, 1::2].shape[1]])

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.size(1) > self.pe.size(1):
            raise ValueError(
                f"Input sequence length {x.size(1)} exceeds positional encoding max length "
                f"{self.pe.size(1)}."
            )

        return self.dropout(x + self.pe[:, : x.size(1), :])


class TransformerEncoderLayerCustom(nn.Module):
    """Custom pre-norm transformer encoder layer used by the CRVSE checkpoint."""

    def __init__(
        self,
        d_model: int,
        n_heads: int,
        dim_feedforward: int = 256,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()

        if d_model % n_heads != 0:
            raise ValueError(f"d_model={d_model} must be divisible by n_heads={n_heads}.")

        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.scale = self.head_dim ** -0.5

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        self.ff = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def _attention(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, time_steps, _ = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(batch_size, time_steps, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, time_steps, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, time_steps, self.n_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)

        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(batch_size, time_steps, -1)

        return self.out_proj(out)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.dropout(self._attention(self.norm1(x)))
        x = x + self.dropout(self.ff(self.norm2(x)))
        return x


class CRVSEPhysFormer(nn.Module):
    """CNN, FFT, and Transformer model for source-FPS rPPG HR estimation."""

    def __init__(
        self,
        in_channels: int = 3,
        cnn_channels: int = 16,
        freq_channels: int = 64,
        n_heads: int = 4,
        n_layers: int = 4,
        dim_feedforward: int = 256,
        dropout: float = 0.11331939348791525,
        hr_min: float = 40.0,
        hr_max: float = 180.0,
        target_frames: int = 240,
        max_positional_length: int = 300,
    ) -> None:
        super().__init__()

        self.in_channels = in_channels
        self.cnn_channels = cnn_channels
        self.freq_channels = freq_channels
        self.n_heads = n_heads
        self.n_layers = n_layers
        self.dim_feedforward = dim_feedforward
        self.dropout_value = dropout
        self.hr_min = hr_min
        self.hr_max = hr_max
        self.target_frames = target_frames

        self.d_model = cnn_channels + freq_channels

        if self.d_model % n_heads != 0:
            raise ValueError(f"d_model={self.d_model} must be divisible by n_heads={n_heads}.")

        self.encoder = nn.Sequential(
            nn.Conv1d(in_channels, cnn_channels // 2, kernel_size=7, padding=3),
            nn.BatchNorm1d(cnn_channels // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(cnn_channels // 2, cnn_channels, kernel_size=5, padding=2),
            nn.BatchNorm1d(cnn_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(cnn_channels, cnn_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(cnn_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        n_fft_bins = target_frames // 2 + 1

        self.freq_proj = nn.Sequential(
            nn.Linear(n_fft_bins, freq_channels * 4),
            nn.ReLU(),
            nn.Linear(freq_channels * 4, freq_channels),
        )

        self.pos_enc = PositionalEncoding(
            d_model=self.d_model,
            max_len=max_positional_length,
            dropout=dropout,
        )

        self.transformer_layers = nn.ModuleList(
            [
                TransformerEncoderLayerCustom(
                    d_model=self.d_model,
                    n_heads=n_heads,
                    dim_feedforward=dim_feedforward,
                    dropout=dropout,
                )
                for _ in range(n_layers)
            ]
        )

        self.head = nn.Sequential(
            nn.Linear(self.d_model, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 3:
            raise ValueError(f"Expected input shape (batch, channels, time), got {tuple(x.shape)}.")

        if x.shape[1] != self.in_channels:
            raise ValueError(f"Expected {self.in_channels} input channels, got {x.shape[1]}.")

        if x.shape[2] != self.target_frames:
            raise ValueError(f"Expected {self.target_frames} frames, got {x.shape[2]}.")

        time_features = self.encoder(x).transpose(1, 2)

        fft_mag = torch.abs(torch.fft.rfft(x, dim=-1))
        fft_mag = torch.mean(fft_mag, dim=1)
        freq_features = self.freq_proj(fft_mag)
        freq_features = freq_features.unsqueeze(1).expand(-1, time_features.size(1), -1)

        features = torch.cat([time_features, freq_features], dim=-1)
        features = self.pos_enc(features)

        for layer in self.transformer_layers:
            features = layer(features)

        pooled = features.mean(dim=1)
        raw = self.head(pooled).squeeze(-1)

        return torch.clamp(raw, min=self.hr_min, max=self.hr_max)


def count_trainable_parameters(model: nn.Module) -> int:
    """Return the number of trainable parameters."""
    return int(sum(param.numel() for param in model.parameters() if param.requires_grad))


def load_frozen_checkpoint(path: Path) -> dict:
    """Load the frozen CRVSE checkpoint."""
    checkpoint = torch.load(path, map_location="cpu")

    if not isinstance(checkpoint, dict):
        raise TypeError(f"Expected checkpoint dictionary, got {type(checkpoint)}")

    if "model_state" not in checkpoint:
        raise KeyError("Checkpoint is missing required key: model_state")

    return checkpoint


def build_physformer_from_checkpoint(checkpoint: dict) -> CRVSEPhysFormer:
    """Build a CRVSEPhysFormer instance from checkpoint metadata."""
    best_params = checkpoint.get("best_params", {}) or {}

    model = CRVSEPhysFormer(
        in_channels=int(checkpoint.get("in_channels", 3)),
        cnn_channels=int(best_params.get("cnn_channels", 16)),
        freq_channels=int(best_params.get("freq_channels", 64)),
        n_heads=int(best_params.get("n_heads", 4)),
        n_layers=int(best_params.get("n_layers", 4)),
        dim_feedforward=int(best_params.get("dim_feedforward", 256)),
        dropout=float(best_params.get("dropout", 0.11331939348791525)),
        hr_min=40.0,
        hr_max=180.0,
        target_frames=240,
        max_positional_length=300,
    )

    model.load_state_dict(checkpoint["model_state"], strict=True)
    return model


FROZEN_CHECKPOINT = load_frozen_checkpoint(CHECKPOINT_PATH)
FROZEN_MODEL = build_physformer_from_checkpoint(FROZEN_CHECKPOINT).to(DEVICE)
FROZEN_MODEL.eval()

checkpoint_report = {
    "checkpoint_path": str(CHECKPOINT_PATH),
    "checkpoint_input_mode": FROZEN_CHECKPOINT.get("input_mode"),
    "checkpoint_in_channels": FROZEN_CHECKPOINT.get("in_channels"),
    "checkpoint_best_val_mae": FROZEN_CHECKPOINT.get("best_val_mae"),
    "checkpoint_best_n_epochs": FROZEN_CHECKPOINT.get("best_n_epochs"),
    "best_params": FROZEN_CHECKPOINT.get("best_params"),
    "trainable_parameters": count_trainable_parameters(FROZEN_MODEL),
    "device": str(DEVICE),
}

x_batch, y_batch, index_batch = next(iter(NB13_LOADERS["val"]))
x_batch = x_batch.to(DEVICE)

with torch.no_grad():
    pred_batch = FROZEN_MODEL(x_batch).detach().cpu()

smoke_report = {
    "input_shape": tuple(x_batch.shape),
    "target_shape": tuple(y_batch.shape),
    "prediction_shape": tuple(pred_batch.shape),
    "prediction_min": float(pred_batch.min().item()),
    "prediction_mean": float(pred_batch.mean().item()),
    "prediction_max": float(pred_batch.max().item()),
    "prediction_finite": bool(torch.isfinite(pred_batch).all().item()),
}

print("Frozen checkpoint report:")
print(json.dumps(checkpoint_report, indent=2))

print("\nFrozen model batch smoke report:")
print(json.dumps(smoke_report, indent=2))

if smoke_report["prediction_shape"] != tuple(y_batch.shape):
    raise AssertionError("Frozen model prediction shape does not match target shape.")

if not smoke_report["prediction_finite"]:
    raise AssertionError("Frozen model produced non-finite predictions.")

print("\nPASS: Frozen CRVSE PhysFormer checkpoint runs on NB13 source-FPS tensors.")


Frozen checkpoint report:
{
  "checkpoint_path": "/kaggle/input/datasets/cezarytubacki/live-compatible-crvse/Live_Compatible_CRVSE/CRVSETransformer_Ensemble_physformer_multichannel_best.pt",
  "checkpoint_input_mode": "multichannel",
  "checkpoint_in_channels": 3,
  "checkpoint_best_val_mae": 6.900240182876587,
  "checkpoint_best_n_epochs": 50,
  "best_params": {
    "lr": 0.0006523895417699133,
    "weight_decay": 0.0008816216939148644,
    "dropout": 0.11331939348791525,
    "huber_delta": 6.544152619447903,
    "cnn_channels": 16,
    "freq_channels": 64,
    "n_heads": 4,
    "n_layers": 4,
    "ff_mult": 4
  },
  "trainable_parameters": 322145,
  "device": "cuda"
}

Frozen model batch smoke report:
{
  "input_shape": [
    512,
    3,
    240
  ],
  "target_shape": [
    512
  ],
  "prediction_shape": [
    512
  ],
  "prediction_min": 40.0,
  "prediction_mean": 73.54159545898438,
  "prediction_max": 117.61675262451172,
  "prediction_finite": true
}

PASS: Frozen CRVSE PhysFormer 

## Frozen Checkpoint Reproduction On NB13 Tensors

This section evaluates the frozen CRVSE PhysFormer on all NB13 source-FPS train, validation, and held-out test tensors.

The notebook first rebuilds the app-aligned PhysFormer architecture, loads the frozen checkpoint with strict weight matching, and then compares notebook predictions against the saved frozen source-FPS baseline.

This check protects the experiment from architecture drift, tensor-order mismatch, metadata mismatch, or checkpoint-loading mistakes.

If the predictions do not match the saved baseline closely, transfer learning and scratch training should not start.

In [6]:
FROZEN_REPRODUCTION_TOLERANCE_BPM = 0.05

NB13_EVAL_LOADERS_BY_ROLE = {
    role: make_eval_loader(dataset, batch_size=EVAL_BATCH_SIZE)
    for role, dataset in NB13_DATASETS.items()
}


class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding for transformer time tokens."""

    def __init__(self, d_model: int, max_len: int = 300, dropout: float = 0.1) -> None:
        super().__init__()

        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term[: pe[:, 1::2].shape[1]])

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.size(1) > self.pe.size(1):
            raise ValueError(
                f"Input sequence length {x.size(1)} exceeds positional encoding max length "
                f"{self.pe.size(1)}."
            )

        return self.dropout(x + self.pe[:, : x.size(1), :])


class TransformerEncoderLayerCustom(nn.Module):
    """Custom pre-norm transformer encoder layer used by the CRVSE checkpoint."""

    def __init__(
        self,
        d_model: int,
        n_heads: int,
        dim_feedforward: int = 256,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()

        if d_model % n_heads != 0:
            raise ValueError(f"d_model={d_model} must be divisible by n_heads={n_heads}.")

        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.scale = self.head_dim ** -0.5

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        self.ff = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def _attention(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, time_steps, _ = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(batch_size, time_steps, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, time_steps, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, time_steps, self.n_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)

        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(batch_size, time_steps, -1)

        return self.out_proj(out)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.dropout(self._attention(self.norm1(x)))
        x = x + self.dropout(self.ff(self.norm2(x)))
        return x


class CRVSEPhysFormer(nn.Module):
    """App-aligned CNN, FFT, and Transformer model for source-FPS rPPG HR estimation."""

    def __init__(
        self,
        in_channels: int = 3,
        cnn_channels: int = 16,
        freq_channels: int = 64,
        n_heads: int = 4,
        n_layers: int = 4,
        dim_feedforward: int = 256,
        dropout: float = 0.11331939348791525,
        hr_min: float = 40.0,
        hr_max: float = 180.0,
        target_frames: int = 240,
        max_positional_length: int = 300,
    ) -> None:
        super().__init__()

        self.in_channels = in_channels
        self.cnn_channels = cnn_channels
        self.freq_channels = freq_channels
        self.n_heads = n_heads
        self.n_layers = n_layers
        self.dim_feedforward = dim_feedforward
        self.dropout_value = dropout
        self.hr_min = hr_min
        self.hr_max = hr_max
        self.target_frames = target_frames
        self.d_model = cnn_channels + freq_channels

        if self.d_model % n_heads != 0:
            raise ValueError(f"d_model={self.d_model} must be divisible by n_heads={n_heads}.")

        self.encoder = nn.Sequential(
            nn.Conv1d(in_channels, cnn_channels // 2, kernel_size=7, padding=3),
            nn.BatchNorm1d(cnn_channels // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(cnn_channels // 2, cnn_channels, kernel_size=5, padding=2),
            nn.BatchNorm1d(cnn_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(cnn_channels, cnn_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(cnn_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        n_fft_bins = target_frames // 2 + 1

        self.freq_proj = nn.Sequential(
            nn.Linear(n_fft_bins, freq_channels * 4),
            nn.ReLU(),
            nn.Linear(freq_channels * 4, freq_channels),
        )

        self.pos_enc = PositionalEncoding(
            d_model=self.d_model,
            max_len=max_positional_length,
            dropout=dropout,
        )

        self.transformer_layers = nn.ModuleList(
            [
                TransformerEncoderLayerCustom(
                    d_model=self.d_model,
                    n_heads=n_heads,
                    dim_feedforward=dim_feedforward,
                    dropout=dropout,
                )
                for _ in range(n_layers)
            ]
        )

        self.head = nn.Sequential(
            nn.Linear(self.d_model, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 3:
            raise ValueError(f"Expected input shape (batch, channels, time), got {tuple(x.shape)}.")

        if x.shape[1] != self.in_channels:
            raise ValueError(f"Expected {self.in_channels} input channels, got {x.shape[1]}.")

        if x.shape[2] != self.target_frames:
            raise ValueError(f"Expected {self.target_frames} frames, got {x.shape[2]}.")

        time_features = self.encoder(x).transpose(1, 2)

        fft_mag = torch.fft.rfft(x, norm="ortho").abs()
        fft_mag = fft_mag.mean(dim=1)
        freq_features = self.freq_proj(fft_mag)
        freq_features = freq_features.unsqueeze(1).expand(-1, time_features.size(1), -1)

        features = torch.cat([time_features, freq_features], dim=-1)
        features = self.pos_enc(features)

        for layer in self.transformer_layers:
            features = layer(features)

        pooled = features.mean(dim=1)
        out = self.head(pooled).squeeze(-1)

        if not self.training:
            out = out.clamp(self.hr_min, self.hr_max)

        return out


def count_trainable_parameters(model: nn.Module) -> int:
    """Return the number of trainable parameters."""
    return int(sum(param.numel() for param in model.parameters() if param.requires_grad))


def load_checkpoint_file(path: Path) -> dict:
    """Load a checkpoint dictionary from disk."""
    try:
        checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        checkpoint = torch.load(path, map_location="cpu")

    if not isinstance(checkpoint, dict):
        raise TypeError(f"Expected checkpoint dictionary, got {type(checkpoint)}.")

    if "model_state" not in checkpoint:
        raise KeyError("Checkpoint is missing required key: model_state.")

    return checkpoint


def infer_dim_feedforward_from_state_dict(model_state: dict) -> int:
    """Infer transformer feed-forward width from checkpoint tensor shapes."""
    key = "transformer_layers.0.ff.0.weight"

    if key not in model_state:
        raise KeyError(f"Cannot infer dim_feedforward; checkpoint is missing {key!r}.")

    return int(model_state[key].shape[0])


def build_physformer_from_checkpoint(checkpoint: dict) -> CRVSEPhysFormer:
    """Build the app-aligned CRVSE PhysFormer from checkpoint metadata."""
    best_params = checkpoint.get("best_params", {}) or {}
    model_state = checkpoint["model_state"]

    cnn_channels = int(best_params.get("cnn_channels", 16))
    freq_channels = int(best_params.get("freq_channels", 64))
    dim_feedforward = infer_dim_feedforward_from_state_dict(model_state)

    model = CRVSEPhysFormer(
        in_channels=int(checkpoint.get("in_channels", 3)),
        cnn_channels=cnn_channels,
        freq_channels=freq_channels,
        n_heads=int(best_params.get("n_heads", 4)),
        n_layers=int(best_params.get("n_layers", 4)),
        dim_feedforward=dim_feedforward,
        dropout=float(best_params.get("dropout", 0.11331939348791525)),
        hr_min=40.0,
        hr_max=180.0,
        target_frames=240,
        max_positional_length=300,
    )

    model.load_state_dict(model_state, strict=True)
    return model


def evaluate_model_on_nb13_role(
    model: nn.Module,
    role: str,
    loader: DataLoader,
    experiment_name: str,
) -> pd.DataFrame:
    """Run one model over one NB13 role and return row-level predictions."""
    dataset = loader.dataset
    model.eval()

    output_parts = []

    with torch.inference_mode():
        for x_batch, y_batch, index_batch in loader:
            x_batch = x_batch.to(DEVICE, non_blocking=True)
            pred_batch = model(x_batch).detach().cpu().float().numpy()

            index_np = index_batch.cpu().numpy()
            metadata = dataset.metadata.iloc[index_np].reset_index(drop=True).copy()

            metadata["experiment"] = experiment_name
            metadata["role"] = role
            metadata["target_hr_from_tensor"] = y_batch.cpu().numpy().astype(np.float32)
            metadata["notebook_model_hr"] = pred_batch.astype(np.float32)
            metadata["notebook_model_abs_error"] = (
                metadata["notebook_model_hr"] - metadata["target_hr_from_tensor"]
            ).abs()

            output_parts.append(metadata)

    return pd.concat(output_parts, ignore_index=True)


def normalize_prediction_keys(table: pd.DataFrame) -> pd.DataFrame:
    """Normalize row identity columns before joining prediction tables."""
    output = table.copy()

    for col in ["dataset", "subject_id", "recording_id", "window_role"]:
        output[col] = output[col].astype(str)

    for col in ["start_frame", "end_frame"]:
        output[col] = pd.to_numeric(output[col], errors="raise").astype(int)

    return output


FROZEN_CHECKPOINT = load_checkpoint_file(CHECKPOINT_PATH)
FROZEN_MODEL = build_physformer_from_checkpoint(FROZEN_CHECKPOINT).to(DEVICE)
FROZEN_MODEL.eval()

checkpoint_report = {
    "checkpoint_path": str(CHECKPOINT_PATH),
    "checkpoint_input_mode": FROZEN_CHECKPOINT.get("input_mode"),
    "checkpoint_in_channels": FROZEN_CHECKPOINT.get("in_channels"),
    "checkpoint_best_val_mae": FROZEN_CHECKPOINT.get("best_val_mae"),
    "checkpoint_best_n_epochs": FROZEN_CHECKPOINT.get("best_n_epochs"),
    "best_params": FROZEN_CHECKPOINT.get("best_params"),
    "inferred_dim_feedforward": infer_dim_feedforward_from_state_dict(FROZEN_CHECKPOINT["model_state"]),
    "trainable_parameters": count_trainable_parameters(FROZEN_MODEL),
    "architecture_note": "App-aligned PhysFormer with dim_feedforward inferred from checkpoint and torch.fft.rfft(norm='ortho').",
    "device": str(DEVICE),
}

frozen_reproduction_parts = []

for role, loader in NB13_EVAL_LOADERS_BY_ROLE.items():
    role_predictions = evaluate_model_on_nb13_role(
        model=FROZEN_MODEL,
        role=role,
        loader=loader,
        experiment_name="frozen_checkpoint_reproduced_in_nb13",
    )
    frozen_reproduction_parts.append(role_predictions)

FROZEN_NB13_REPRODUCED_PREDICTIONS = pd.concat(
    frozen_reproduction_parts,
    ignore_index=True,
)

join_keys = [
    "dataset",
    "window_role",
    "subject_id",
    "recording_id",
    "start_frame",
    "end_frame",
]

reproduced_join = normalize_prediction_keys(FROZEN_NB13_REPRODUCED_PREDICTIONS)
baseline_join = normalize_prediction_keys(sourcefps_app.copy())

baseline_keep = baseline_join[
    join_keys + ["model_hr", "model_abs_error", "target_hr_mean"]
].rename(
    columns={
        "model_hr": "baseline_model_hr",
        "model_abs_error": "baseline_model_abs_error",
        "target_hr_mean": "baseline_target_hr_mean",
    }
)

frozen_reproduction_compare = reproduced_join.merge(
    baseline_keep,
    on=join_keys,
    how="left",
    validate="one_to_one",
)

frozen_reproduction_compare["abs_diff_model_hr_vs_baseline"] = (
    frozen_reproduction_compare["notebook_model_hr"]
    - frozen_reproduction_compare["baseline_model_hr"]
).abs()

frozen_reproduction_compare["abs_diff_target_vs_baseline"] = (
    frozen_reproduction_compare["target_hr_from_tensor"]
    - frozen_reproduction_compare["baseline_target_hr_mean"]
).abs()

reproduction_summary = (
    frozen_reproduction_compare
    .groupby("window_role", dropna=False)
    .agg(
        rows=("notebook_model_hr", "size"),
        matched_baseline_rows=("baseline_model_hr", lambda values: values.notna().sum()),
        notebook_mae=("notebook_model_abs_error", "mean"),
        baseline_mae=("baseline_model_abs_error", "mean"),
        mean_abs_diff_model_hr=("abs_diff_model_hr_vs_baseline", "mean"),
        max_abs_diff_model_hr=("abs_diff_model_hr_vs_baseline", "max"),
        max_abs_diff_target=("abs_diff_target_vs_baseline", "max"),
    )
    .reset_index()
)

numeric_cols = reproduction_summary.select_dtypes(include=[np.number]).columns
reproduction_summary[numeric_cols] = reproduction_summary[numeric_cols].round(6)

worst_prediction_diffs = (
    frozen_reproduction_compare
    .sort_values("abs_diff_model_hr_vs_baseline", ascending=False)
    [
        join_keys
        + [
            "target_hr_from_tensor",
            "notebook_model_hr",
            "baseline_model_hr",
            "abs_diff_model_hr_vs_baseline",
        ]
    ]
    .head(10)
)

print("Frozen checkpoint report:")
print(json.dumps(checkpoint_report, indent=2))

print("\nFrozen checkpoint reproduction summary:")
display(reproduction_summary)

print("\nLargest prediction differences versus saved baseline:")
display(worst_prediction_diffs)

failed_reproduction_checks = []

if frozen_reproduction_compare["baseline_model_hr"].isna().any():
    missing_count = int(frozen_reproduction_compare["baseline_model_hr"].isna().sum())
    failed_reproduction_checks.append(f"Missing saved baseline matches: {missing_count}")

max_model_diff = float(frozen_reproduction_compare["abs_diff_model_hr_vs_baseline"].max())
max_target_diff = float(frozen_reproduction_compare["abs_diff_target_vs_baseline"].max())

if max_model_diff > FROZEN_REPRODUCTION_TOLERANCE_BPM:
    failed_reproduction_checks.append(
        "Notebook frozen predictions differ from saved baseline by more than "
        f"{FROZEN_REPRODUCTION_TOLERANCE_BPM} BPM. "
        f"Observed max diff: {max_model_diff:.6f} BPM."
    )

if max_target_diff > 0.02:
    failed_reproduction_checks.append(
        "Tensor targets differ from saved baseline target_hr_mean by more than 0.02 BPM. "
        f"Observed max diff: {max_target_diff:.6f} BPM."
    )

if failed_reproduction_checks:
    raise AssertionError(
        "Frozen checkpoint reproduction failed:\n"
        + "\n".join(failed_reproduction_checks)
    )

reproduction_predictions_path = NB13_WORKING_DIR / "nb13_frozen_reproduced_predictions.csv"
reproduction_compare_path = NB13_WORKING_DIR / "nb13_frozen_reproduction_compare.csv"

FROZEN_NB13_REPRODUCED_PREDICTIONS.to_csv(reproduction_predictions_path, index=False)
frozen_reproduction_compare.to_csv(reproduction_compare_path, index=False)

print(f"\nSaved reproduced frozen predictions: {reproduction_predictions_path}")
print(f"Saved frozen reproduction comparison: {reproduction_compare_path}")
print("PASS: NB13 frozen checkpoint reproduces the saved source-FPS baseline.")

Frozen checkpoint report:
{
  "checkpoint_path": "/kaggle/input/datasets/cezarytubacki/live-compatible-crvse/Live_Compatible_CRVSE/CRVSETransformer_Ensemble_physformer_multichannel_best.pt",
  "checkpoint_input_mode": "multichannel",
  "checkpoint_in_channels": 3,
  "checkpoint_best_val_mae": 6.900240182876587,
  "checkpoint_best_n_epochs": 50,
  "best_params": {
    "lr": 0.0006523895417699133,
    "weight_decay": 0.0008816216939148644,
    "dropout": 0.11331939348791525,
    "huber_delta": 6.544152619447903,
    "cnn_channels": 16,
    "freq_channels": 64,
    "n_heads": 4,
    "n_layers": 4,
    "ff_mult": 4
  },
  "inferred_dim_feedforward": 256,
  "trainable_parameters": 322145,
  "architecture_note": "App-aligned PhysFormer with dim_feedforward inferred from checkpoint and torch.fft.rfft(norm='ortho').",
  "device": "cuda"
}

Frozen checkpoint reproduction summary:


,window_role,rows,matched_baseline_rows,notebook_mae,baseline_mae,mean_abs_diff_model_hr,max_abs_diff_model_hr,max_abs_diff_target
0,finetune_test,2828,2828,5.769625,5.769625,0.000009,0.000107,0.0
1,finetune_train,12503,12503,5.823606,5.823606,0.000009,0.000137,0.0
2,finetune_val,2839,2839,6.027502,6.027502,0.000009,0.000092,0.0



Largest prediction differences versus saved baseline:


,dataset,window_role,subject_id,recording_id,start_frame,end_frame,target_hr_from_tensor,notebook_model_hr,baseline_model_hr,abs_diff_model_hr_vs_baseline
1490,mcd_rppg,finetune_train,2732,before,1190,1429,64.441162,78.558884,78.559021,0.000137
9798,ubfc_phys,finetune_train,s35,T3,6020,6301,60.509174,104.911751,104.911613,0.000137
16877,mcd_rppg,finetune_test,7142,before,833,1072,77.166908,83.462303,83.462410,0.000107
3478,mcd_rppg,finetune_train,4731,before,4200,4440,63.162003,72.793121,72.793015,0.000107
16733,mcd_rppg,finetune_test,6031,after,4403,4642,64.261765,63.999489,63.999382,0.000107
11,mcd_rppg,finetune_train,1035,before,1309,1548,66.839890,79.306808,79.306709,0.000099
3834,mcd_rppg,finetune_train,5156,before,238,477,71.951363,98.943703,98.943604,0.000099
12073,ubfc_rppg,finetune_train,subject14,rest,117,351,93.241882,116.033493,116.033394,0.000099
1370,mcd_rppg,finetune_train,2676,after,2499,2738,71.804794,81.382217,81.382118,0.000099
10594,ubfc_phys,finetune_train,s42,T3,4480,4761,76.542091,80.795990,80.795891,0.000099



Saved reproduced frozen predictions: /kaggle/working/crvse_nb13_app_relevant_without_ecg_fitness/nb13_frozen_reproduced_predictions.csv
Saved frozen reproduction comparison: /kaggle/working/crvse_nb13_app_relevant_without_ecg_fitness/nb13_frozen_reproduction_compare.csv
PASS: NB13 frozen checkpoint reproduces the saved source-FPS baseline.


## Shared Training And Evaluation Helpers

This section defines reusable functions for NB13 transfer learning and scratch training.

Both model branches use the same training loop, validation loop, evaluation tables, and candidate-comparison function.

The purpose is to avoid comparing transfer and scratch with different code paths.

No model training starts in this section.

In [7]:
NB13_TRAINING_CONFIG = {
    "loss": "SmoothL1Loss",
    "default_huber_beta": 6.0,
    "default_lr": 5e-5,
    "default_weight_decay": 1e-4,
    "default_grad_clip": 5.0,
    "max_epochs_smoke": 2,
    "max_epochs_main": 20,
    "early_stopping_patience": 5,
}


def make_loss_fn(beta: float = NB13_TRAINING_CONFIG["default_huber_beta"]) -> nn.Module:
    """Create the HR regression loss used for NB13 experiments."""
    return nn.SmoothL1Loss(beta=float(beta))


def set_all_trainable(model: nn.Module, trainable: bool) -> None:
    """Set requires_grad for every model parameter."""
    for param in model.parameters():
        param.requires_grad = bool(trainable)


def set_transfer_trainable_layers(
    model: CRVSEPhysFormer,
    train_encoder: bool = False,
    train_last_transformer_block: bool = True,
    train_head: bool = True,
    train_freq_proj: bool = True,
) -> pd.DataFrame:
    """Configure a conservative transfer-learning parameter policy."""
    set_all_trainable(model, False)

    if train_encoder:
        for param in model.encoder.parameters():
            param.requires_grad = True

    if train_freq_proj:
        for param in model.freq_proj.parameters():
            param.requires_grad = True

    if train_last_transformer_block:
        for param in model.transformer_layers[-1].parameters():
            param.requires_grad = True

    if train_head:
        for param in model.head.parameters():
            param.requires_grad = True

    rows = []

    for module_name, module in [
        ("encoder", model.encoder),
        ("freq_proj", model.freq_proj),
        ("transformer_layers", model.transformer_layers),
        ("head", model.head),
    ]:
        total = sum(param.numel() for param in module.parameters())
        trainable = sum(param.numel() for param in module.parameters() if param.requires_grad)
        rows.append(
            {
                "module": module_name,
                "total_params": int(total),
                "trainable_params": int(trainable),
                "trainable_pct": round(100.0 * trainable / total, 2) if total else 0.0,
            }
        )

    return pd.DataFrame(rows)


def build_transfer_model_from_frozen() -> CRVSEPhysFormer:
    """Build a transfer model initialized from the frozen app checkpoint."""
    model = build_physformer_from_checkpoint(FROZEN_CHECKPOINT)
    return model.to(DEVICE)


def build_scratch_model_like_checkpoint() -> CRVSEPhysFormer:
    """Build a randomly initialized PhysFormer with the frozen checkpoint architecture."""
    best_params = FROZEN_CHECKPOINT.get("best_params", {}) or {}
    dim_feedforward = infer_dim_feedforward_from_state_dict(FROZEN_CHECKPOINT["model_state"])

    model = CRVSEPhysFormer(
        in_channels=int(FROZEN_CHECKPOINT.get("in_channels", 3)),
        cnn_channels=int(best_params.get("cnn_channels", 16)),
        freq_channels=int(best_params.get("freq_channels", 64)),
        n_heads=int(best_params.get("n_heads", 4)),
        n_layers=int(best_params.get("n_layers", 4)),
        dim_feedforward=dim_feedforward,
        dropout=float(best_params.get("dropout", 0.11331939348791525)),
        hr_min=40.0,
        hr_max=180.0,
        target_frames=240,
        max_positional_length=300,
    )

    return model.to(DEVICE)


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_fn: nn.Module,
    grad_clip: float,
) -> dict:
    """Train one epoch and return loss/MAE diagnostics."""
    model.train()

    total_loss = 0.0
    total_abs_error = 0.0
    total_rows = 0
    grad_norms = []

    for x_batch, y_batch, _ in loader:
        x_batch = x_batch.to(DEVICE, non_blocking=True)
        y_batch = y_batch.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        pred = model(x_batch)
        loss = loss_fn(pred, y_batch)

        loss.backward()

        grad_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=float(grad_clip),
        )
        grad_norms.append(float(grad_norm.detach().cpu().item()))

        optimizer.step()

        batch_rows = int(len(y_batch))
        total_rows += batch_rows
        total_loss += float(loss.detach().cpu().item()) * batch_rows
        total_abs_error += float((pred.detach() - y_batch).abs().sum().cpu().item())

    return {
        "train_rows": total_rows,
        "train_loss": total_loss / total_rows,
        "train_mae": total_abs_error / total_rows,
        "grad_norm_mean": float(np.mean(grad_norms)) if grad_norms else np.nan,
        "grad_norm_max": float(np.max(grad_norms)) if grad_norms else np.nan,
    }


def evaluate_mae_on_loader(model: nn.Module, loader: DataLoader) -> dict:
    """Evaluate MAE and error distribution on one loader."""
    model.eval()

    errors = []
    preds = []
    targets = []

    with torch.inference_mode():
        for x_batch, y_batch, _ in loader:
            x_batch = x_batch.to(DEVICE, non_blocking=True)
            y_batch = y_batch.to(DEVICE, non_blocking=True)

            pred = model(x_batch)

            preds.append(pred.detach().cpu().numpy())
            targets.append(y_batch.detach().cpu().numpy())
            errors.append((pred - y_batch).abs().detach().cpu().numpy())

    errors = np.concatenate(errors)
    preds = np.concatenate(preds)
    targets = np.concatenate(targets)

    return {
        "rows": int(len(errors)),
        "mae_mean": float(np.mean(errors)),
        "mae_median": float(np.median(errors)),
        "mae_p90": float(np.quantile(errors, 0.90)),
        "mae_p95": float(np.quantile(errors, 0.95)),
        "pred_mean": float(np.mean(preds)),
        "target_mean": float(np.mean(targets)),
        "signed_error_mean": float(np.mean(preds - targets)),
    }


def run_training_loop(
    model: nn.Module,
    experiment_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    max_epochs: int,
    lr: float,
    weight_decay: float,
    huber_beta: float,
    grad_clip: float,
    patience: int,
) -> dict:
    """Run a supervised HR training loop with validation-based checkpoint selection."""
    loss_fn = make_loss_fn(beta=huber_beta)

    optimizer = AdamW(
        [param for param in model.parameters() if param.requires_grad],
        lr=float(lr),
        weight_decay=float(weight_decay),
    )

    history_rows = []
    best_state = None
    best_val_mae = float("inf")
    best_epoch = 0
    stale_epochs = 0

    for epoch in range(1, int(max_epochs) + 1):
        train_metrics = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            loss_fn=loss_fn,
            grad_clip=grad_clip,
        )
        val_metrics = evaluate_mae_on_loader(model, val_loader)

        improved = val_metrics["mae_mean"] < best_val_mae

        if improved:
            best_val_mae = val_metrics["mae_mean"]
            best_epoch = epoch
            stale_epochs = 0
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
        else:
            stale_epochs += 1

        row = {
            "experiment": experiment_name,
            "epoch": epoch,
            **train_metrics,
            "val_mae": val_metrics["mae_mean"],
            "val_median": val_metrics["mae_median"],
            "val_p90": val_metrics["mae_p90"],
            "val_p95": val_metrics["mae_p95"],
            "val_signed_error_mean": val_metrics["signed_error_mean"],
            "improved": bool(improved),
            "best_epoch_so_far": int(best_epoch),
            "best_val_mae_so_far": float(best_val_mae),
        }
        history_rows.append(row)

        print(
            f"{experiment_name} | epoch {epoch:02d} | "
            f"train MAE {train_metrics['train_mae']:.4f} | "
            f"val MAE {val_metrics['mae_mean']:.4f} | "
            f"best {best_val_mae:.4f} @ epoch {best_epoch}"
        )

        if stale_epochs >= int(patience):
            print(f"Early stopping after epoch {epoch}.")
            break

    if best_state is None:
        raise RuntimeError(f"{experiment_name}: no best state was recorded.")

    model.load_state_dict(best_state, strict=True)

    return {
        "model": model,
        "history": pd.DataFrame(history_rows),
        "best_epoch": int(best_epoch),
        "best_val_mae": float(best_val_mae),
    }


def evaluate_candidate_model(
    model: nn.Module,
    experiment_name: str,
) -> pd.DataFrame:
    """Evaluate a candidate model on NB13 train, validation, and held-out test roles."""
    parts = []

    for role, loader in NB13_EVAL_LOADERS_BY_ROLE.items():
        parts.append(
            evaluate_model_on_nb13_role(
                model=model,
                role=role,
                loader=loader,
                experiment_name=experiment_name,
            )
        )

    return pd.concat(parts, ignore_index=True)


def compare_nb13_candidate_predictions(
    predictions: pd.DataFrame,
    candidate_name: str,
) -> dict:
    """Compare candidate predictions against the frozen NB13 baseline."""
    candidate_errors = prepare_nb13_error_table(
        predictions=predictions,
        experiment_name=candidate_name,
        target_col="target_hr_from_tensor",
        pred_col="notebook_model_hr",
        role_col="window_role",
    )

    return compare_candidate_to_frozen(
        candidate_error_table=candidate_errors,
        candidate_name=candidate_name,
        frozen_error_table=FROZEN_NB13_ERROR_TABLE,
    )


helper_report = {
    "training_config": NB13_TRAINING_CONFIG,
    "train_loader_rows": len(NB13_LOADERS["train"].dataset),
    "val_loader_rows": len(NB13_LOADERS["val"].dataset),
    "test_loader_rows": len(NB13_LOADERS["test"].dataset),
    "device": str(DEVICE),
}

print("NB13 shared training/evaluation helper report:")
print(json.dumps(helper_report, indent=2))
print("PASS: Shared training and evaluation helpers are defined.")

NB13 shared training/evaluation helper report:
{
  "training_config": {
    "loss": "SmoothL1Loss",
    "default_huber_beta": 6.0,
    "default_lr": 5e-05,
    "default_weight_decay": 0.0001,
    "default_grad_clip": 5.0,
    "max_epochs_smoke": 2,
    "max_epochs_main": 20,
    "early_stopping_patience": 5
  },
  "train_loader_rows": 12503,
  "val_loader_rows": 2839,
  "test_loader_rows": 2828,
  "device": "cuda"
}
PASS: Shared training and evaluation helpers are defined.


## Transfer-Learning Smoke Test

This section runs a short transfer-learning smoke test.

The smoke test initializes the model from the frozen app checkpoint and trains only a conservative subset of parameters.

The purpose is not to produce an adoption candidate. The purpose is to verify that the NB13 training loop, validation logic, checkpoint selection, and frozen-baseline comparison work end to end.

The result should be interpreted as implementation readiness only.

In [8]:
TRANSFER_SMOKE_NAME = "transfer_smoke_last_block_freq_head"

transfer_smoke_model = build_transfer_model_from_frozen()

transfer_trainable_report = set_transfer_trainable_layers(
    transfer_smoke_model,
    train_encoder=False,
    train_last_transformer_block=True,
    train_head=True,
    train_freq_proj=True,
)

print("Transfer smoke trainable parameter report:")
display(transfer_trainable_report)

transfer_smoke_result = run_training_loop(
    model=transfer_smoke_model,
    experiment_name=TRANSFER_SMOKE_NAME,
    train_loader=NB13_LOADERS["train"],
    val_loader=NB13_LOADERS["val"],
    max_epochs=NB13_TRAINING_CONFIG["max_epochs_smoke"],
    lr=NB13_TRAINING_CONFIG["default_lr"],
    weight_decay=NB13_TRAINING_CONFIG["default_weight_decay"],
    huber_beta=NB13_TRAINING_CONFIG["default_huber_beta"],
    grad_clip=NB13_TRAINING_CONFIG["default_grad_clip"],
    patience=NB13_TRAINING_CONFIG["early_stopping_patience"],
)

TRANSFER_SMOKE_MODEL = transfer_smoke_result["model"]
TRANSFER_SMOKE_HISTORY = transfer_smoke_result["history"]

print("\nTransfer smoke training history:")
display(TRANSFER_SMOKE_HISTORY)

TRANSFER_SMOKE_PREDICTIONS = evaluate_candidate_model(
    model=TRANSFER_SMOKE_MODEL,
    experiment_name=TRANSFER_SMOKE_NAME,
)

transfer_smoke_comparison = compare_nb13_candidate_predictions(
    predictions=TRANSFER_SMOKE_PREDICTIONS,
    candidate_name=TRANSFER_SMOKE_NAME,
)

TRANSFER_SMOKE_DECISION = pd.DataFrame([transfer_smoke_comparison["decision_row"]])
TRANSFER_SMOKE_ROLE_COMPARE = transfer_smoke_comparison["role_compare"]
TRANSFER_SMOKE_DATASET_ROLE_COMPARE = transfer_smoke_comparison["dataset_role_compare"]

print("\nTransfer smoke decision row:")
display(TRANSFER_SMOKE_DECISION)

print("\nTransfer smoke role comparison:")
display(TRANSFER_SMOKE_ROLE_COMPARE)

print("\nTransfer smoke dataset-role comparison:")
display(
    TRANSFER_SMOKE_DATASET_ROLE_COMPARE.sort_values(["window_role", "dataset"])
)

transfer_smoke_history_path = NB13_WORKING_DIR / "nb13_transfer_smoke_history.csv"
transfer_smoke_predictions_path = NB13_WORKING_DIR / "nb13_transfer_smoke_predictions.csv"
transfer_smoke_decision_path = NB13_WORKING_DIR / "nb13_transfer_smoke_decision.csv"
transfer_smoke_dataset_compare_path = NB13_WORKING_DIR / "nb13_transfer_smoke_dataset_role_compare.csv"
transfer_smoke_checkpoint_path = NB13_WORKING_DIR / "nb13_transfer_smoke_best_state.pt"

TRANSFER_SMOKE_HISTORY.to_csv(transfer_smoke_history_path, index=False)
TRANSFER_SMOKE_PREDICTIONS.to_csv(transfer_smoke_predictions_path, index=False)
TRANSFER_SMOKE_DECISION.to_csv(transfer_smoke_decision_path, index=False)
TRANSFER_SMOKE_DATASET_ROLE_COMPARE.to_csv(transfer_smoke_dataset_compare_path, index=False)

torch.save(
    {
        "experiment": TRANSFER_SMOKE_NAME,
        "model_state": TRANSFER_SMOKE_MODEL.state_dict(),
        "best_epoch": transfer_smoke_result["best_epoch"],
        "best_val_mae": transfer_smoke_result["best_val_mae"],
        "training_config": NB13_TRAINING_CONFIG,
        "trainable_report": transfer_trainable_report.to_dict(orient="records"),
        "decision_row": transfer_smoke_comparison["decision_row"],
        "note": "Smoke-test checkpoint only. Not an app adoption candidate.",
    },
    transfer_smoke_checkpoint_path,
)

print(f"\nSaved transfer smoke history: {transfer_smoke_history_path}")
print(f"Saved transfer smoke predictions: {transfer_smoke_predictions_path}")
print(f"Saved transfer smoke decision: {transfer_smoke_decision_path}")
print(f"Saved transfer smoke dataset comparison: {transfer_smoke_dataset_compare_path}")
print(f"Saved transfer smoke checkpoint: {transfer_smoke_checkpoint_path}")

print("\nPASS: Transfer-learning smoke test completed end to end.")

Transfer smoke trainable parameter report:


,module,total_params,trainable_params,trainable_pct
0,encoder,1696,0,0.0
1,freq_proj,47680,47680,100.0
2,transformer_layers,270144,67536,25.0
3,head,2625,2625,100.0


transfer_smoke_last_block_freq_head | epoch 01 | train MAE 8.6080 | val MAE 5.9951 | best 5.9951 @ epoch 1
transfer_smoke_last_block_freq_head | epoch 02 | train MAE 8.5591 | val MAE 5.9358 | best 5.9358 @ epoch 2

Transfer smoke training history:


,experiment,epoch,train_rows,train_loss,train_mae,grad_norm_mean,grad_norm_max,val_mae,val_median,val_p90,val_p95,val_signed_error_mean,improved,best_epoch_so_far,best_val_mae_so_far
0,transfer_smoke_last_block_freq_head,1,12503,6.082776,8.607970,18.982785,36.544594,5.995136,3.882492,14.114280,19.046434,1.353937,True,1,5.995136
1,transfer_smoke_last_block_freq_head,2,12503,6.039472,8.559076,16.876085,42.904953,5.935812,3.804024,14.165088,19.056429,0.810676,True,2,5.935812



Transfer smoke decision row:


...


Transfer smoke role comparison:


,window_role,rows_candidate,target_hr_mean_candidate,pred_hr_mean_candidate,mae_mean_candidate,mae_median_candidate,mae_p90_candidate,mae_p95_candidate,signed_error_mean_candidate,abs_signed_bias_candidate,...,abs_signed_bias_frozen,severe_error_15bpm_rate_frozen,high_hr_ge_100_rows_frozen,delta_mae_mean,delta_mae_median,delta_mae_p90,delta_mae_p95,delta_signed_error_mean,delta_abs_signed_bias,delta_severe_error_15bpm_rate
0,finetune_test,2828,79.782204,80.667503,5.6466,3.4336,13.0406,18.7436,0.8853,0.8853,...,1.4174,0.0831,244,-0.1230,-0.1938,-0.6578,-0.3025,-0.5321,-0.5321,-0.0018
1,finetune_train,12503,80.237000,81.174301,5.6218,3.6190,13.2297,18.2548,0.9372,0.9372,...,1.5658,0.0846,1129,-0.2018,-0.1247,-0.4895,-0.6878,-0.6286,-0.6286,-0.0061
2,finetune_val,2839,80.161697,80.972397,5.9358,3.8040,14.1651,19.0564,0.8107,0.8107,...,1.4724,0.0891,168,-0.0917,-0.0493,0.0950,-0.1755,-0.6617,-0.6617,0.0018



Transfer smoke dataset-role comparison:


,dataset,window_role,rows_candidate,target_hr_mean_candidate,pred_hr_mean_candidate,mae_mean_candidate,mae_median_candidate,mae_p90_candidate,mae_p95_candidate,signed_error_mean_candidate,...,abs_signed_bias_frozen,severe_error_15bpm_rate_frozen,high_hr_ge_100_rows_frozen,delta_mae_mean,delta_mae_median,delta_mae_p90,delta_mae_p95,delta_signed_error_mean,delta_abs_signed_bias,delta_severe_error_15bpm_rate
0,mcd_rppg,finetune_test,2094,79.853302,80.260696,4.7635,2.9959,10.7291,15.5472,0.4074,...,0.9242,0.0525,192,-0.1212,-0.1543,-0.2409,-0.0018,-0.5168,-0.5168,-0.0004
3,ubfc_phys,finetune_test,658,77.420601,79.985497,8.7002,5.7155,19.8350,25.3721,2.5649,...,3.1980,0.1900,7,-0.1582,0.0420,-0.4512,-0.3012,-0.6331,-0.6331,-0.0061
6,ubfc_rppg,finetune_test,76,98.269302,97.781502,3.5411,3.1712,6.6195,7.7883,-0.4878,...,0.4102,0.0000,45,0.1326,0.0259,0.5360,0.9489,-0.0776,0.0776,0.0000
1,mcd_rppg,finetune_train,7446,79.064796,79.702103,4.4482,3.0213,9.8673,13.3031,0.6372,...,1.2645,0.0431,569,-0.2394,-0.1426,-0.5049,-0.6938,-0.6273,-0.6273,-0.0059
4,ubfc_phys,finetune_train,4595,80.305496,81.962601,7.6168,5.3236,18.0475,22.8044,1.6572,...,2.3192,0.1567,367,-0.1651,-0.1012,-0.5453,-0.6798,-0.6620,-0.6620,-0.0072
7,ubfc_rppg,finetune_train,462,98.449203,97.060799,4.6950,3.2443,10.2604,13.7578,-1.3884,...,1.0723,0.0368,193,0.0402,0.0996,0.3415,0.5891,-0.3161,0.3161,0.0022
2,mcd_rppg,finetune_val,1743,79.108597,79.581703,4.9874,3.3075,11.1019,15.3133,0.4731,...,1.2656,0.0545,79,-0.1451,-0.0671,-0.4349,-0.1651,-0.7925,-0.7925,-0.0006
5,ubfc_phys,finetune_val,986,80.002403,81.817398,7.7319,5.0587,18.2675,23.9994,1.8151,...,2.2826,0.1531,19,-0.0310,0.2126,-0.0050,-0.2662,-0.4675,-0.4675,0.0051
8,ubfc_rppg,finetune_val,110,98.277199,95.434502,4.8643,2.6603,13.7487,16.5366,-2.8427,...,2.5126,0.0636,70,0.2110,0.2165,1.3713,1.1349,-0.3301,0.3301,0.0091



Saved transfer smoke history: /kaggle/working/crvse_nb13_app_relevant_without_ecg_fitness/nb13_transfer_smoke_history.csv
Saved transfer smoke predictions: /kaggle/working/crvse_nb13_app_relevant_without_ecg_fitness/nb13_transfer_smoke_predictions.csv
Saved transfer smoke decision: /kaggle/working/crvse_nb13_app_relevant_without_ecg_fitness/nb13_transfer_smoke_decision.csv
Saved transfer smoke dataset comparison: /kaggle/working/crvse_nb13_app_relevant_without_ecg_fitness/nb13_transfer_smoke_dataset_role_compare.csv
Saved transfer smoke checkpoint: /kaggle/working/crvse_nb13_app_relevant_without_ecg_fitness/nb13_transfer_smoke_best_state.pt

PASS: Transfer-learning smoke test completed end to end.


## Scratch-Training Smoke Test

This section runs a short scratch-training smoke test.

The smoke test initializes the same CRVSE PhysFormer architecture from random weights and trains it on the NB13 app-relevant source-FPS tensors.

The purpose is not to produce an adoption candidate. The purpose is to verify that scratch training can run through the same training loop, validation logic, checkpoint selection, and frozen-baseline comparison used by transfer learning.

The result should be interpreted as implementation readiness only.

In [9]:
SCRATCH_SMOKE_NAME = "scratch_smoke_physformer"

scratch_smoke_model = build_scratch_model_like_checkpoint()
set_all_trainable(scratch_smoke_model, True)

scratch_trainable_report = pd.DataFrame(
    [
        {
            "module": name,
            "total_params": int(sum(param.numel() for param in module.parameters())),
            "trainable_params": int(
                sum(param.numel() for param in module.parameters() if param.requires_grad)
            ),
        }
        for name, module in [
            ("encoder", scratch_smoke_model.encoder),
            ("freq_proj", scratch_smoke_model.freq_proj),
            ("transformer_layers", scratch_smoke_model.transformer_layers),
            ("head", scratch_smoke_model.head),
        ]
    ]
)

scratch_trainable_report["trainable_pct"] = (
    100.0
    * scratch_trainable_report["trainable_params"]
    / scratch_trainable_report["total_params"]
).round(2)

print("Scratch smoke trainable parameter report:")
display(scratch_trainable_report)

scratch_smoke_result = run_training_loop(
    model=scratch_smoke_model,
    experiment_name=SCRATCH_SMOKE_NAME,
    train_loader=NB13_LOADERS["train"],
    val_loader=NB13_LOADERS["val"],
    max_epochs=NB13_TRAINING_CONFIG["max_epochs_smoke"],
    lr=1e-4,
    weight_decay=NB13_TRAINING_CONFIG["default_weight_decay"],
    huber_beta=NB13_TRAINING_CONFIG["default_huber_beta"],
    grad_clip=NB13_TRAINING_CONFIG["default_grad_clip"],
    patience=NB13_TRAINING_CONFIG["early_stopping_patience"],
)

SCRATCH_SMOKE_MODEL = scratch_smoke_result["model"]
SCRATCH_SMOKE_HISTORY = scratch_smoke_result["history"]

print("\nScratch smoke training history:")
display(SCRATCH_SMOKE_HISTORY)

SCRATCH_SMOKE_PREDICTIONS = evaluate_candidate_model(
    model=SCRATCH_SMOKE_MODEL,
    experiment_name=SCRATCH_SMOKE_NAME,
)

scratch_smoke_comparison = compare_nb13_candidate_predictions(
    predictions=SCRATCH_SMOKE_PREDICTIONS,
    candidate_name=SCRATCH_SMOKE_NAME,
)

SCRATCH_SMOKE_DECISION = pd.DataFrame([scratch_smoke_comparison["decision_row"]])
SCRATCH_SMOKE_ROLE_COMPARE = scratch_smoke_comparison["role_compare"]
SCRATCH_SMOKE_DATASET_ROLE_COMPARE = scratch_smoke_comparison["dataset_role_compare"]

print("\nScratch smoke decision row:")
display(SCRATCH_SMOKE_DECISION)

print("\nScratch smoke role comparison:")
display(SCRATCH_SMOKE_ROLE_COMPARE)

print("\nScratch smoke dataset-role comparison:")
display(
    SCRATCH_SMOKE_DATASET_ROLE_COMPARE.sort_values(["window_role", "dataset"])
)

scratch_smoke_history_path = NB13_WORKING_DIR / "nb13_scratch_smoke_history.csv"
scratch_smoke_predictions_path = NB13_WORKING_DIR / "nb13_scratch_smoke_predictions.csv"
scratch_smoke_decision_path = NB13_WORKING_DIR / "nb13_scratch_smoke_decision.csv"
scratch_smoke_dataset_compare_path = NB13_WORKING_DIR / "nb13_scratch_smoke_dataset_role_compare.csv"
scratch_smoke_checkpoint_path = NB13_WORKING_DIR / "nb13_scratch_smoke_best_state.pt"

SCRATCH_SMOKE_HISTORY.to_csv(scratch_smoke_history_path, index=False)
SCRATCH_SMOKE_PREDICTIONS.to_csv(scratch_smoke_predictions_path, index=False)
SCRATCH_SMOKE_DECISION.to_csv(scratch_smoke_decision_path, index=False)
SCRATCH_SMOKE_DATASET_ROLE_COMPARE.to_csv(scratch_smoke_dataset_compare_path, index=False)

torch.save(
    {
        "experiment": SCRATCH_SMOKE_NAME,
        "model_state": SCRATCH_SMOKE_MODEL.state_dict(),
        "best_epoch": scratch_smoke_result["best_epoch"],
        "best_val_mae": scratch_smoke_result["best_val_mae"],
        "training_config": {
            **NB13_TRAINING_CONFIG,
            "smoke_lr": 1e-4,
        },
        "trainable_report": scratch_trainable_report.to_dict(orient="records"),
        "decision_row": scratch_smoke_comparison["decision_row"],
        "note": "Smoke-test checkpoint only. Not an app adoption candidate.",
    },
    scratch_smoke_checkpoint_path,
)

print(f"\nSaved scratch smoke history: {scratch_smoke_history_path}")
print(f"Saved scratch smoke predictions: {scratch_smoke_predictions_path}")
print(f"Saved scratch smoke decision: {scratch_smoke_decision_path}")
print(f"Saved scratch smoke dataset comparison: {scratch_smoke_dataset_compare_path}")
print(f"Saved scratch smoke checkpoint: {scratch_smoke_checkpoint_path}")

print("\nPASS: Scratch-training smoke test completed end to end.")

Scratch smoke trainable parameter report:


,module,total_params,trainable_params,trainable_pct
0,encoder,1696,1696,100.0
1,freq_proj,47680,47680,100.0
2,transformer_layers,270144,270144,100.0
3,head,2625,2625,100.0


scratch_smoke_physformer | epoch 01 | train MAE 77.2380 | val MAE 40.1617 | best 40.1617 @ epoch 1
scratch_smoke_physformer | epoch 02 | train MAE 66.4852 | val MAE 40.1617 | best 40.1617 @ epoch 1

Scratch smoke training history:


,experiment,epoch,train_rows,train_loss,train_mae,grad_norm_mean,grad_norm_max,val_mae,val_median,val_p90,val_p95,val_signed_error_mean,improved,best_epoch_so_far,best_val_mae_so_far
0,scratch_smoke_physformer,1,12503,74.237995,77.237995,11.511238,21.330318,40.161743,41.398399,56.477245,61.47102,-40.161743,True,1,40.161743
1,scratch_smoke_physformer,2,12503,63.485162,66.485162,39.472044,61.912357,40.161743,41.398399,56.477245,61.47102,-40.161743,False,1,40.161743



Scratch smoke decision row:


...


Scratch smoke role comparison:


,window_role,rows_candidate,target_hr_mean_candidate,pred_hr_mean_candidate,mae_mean_candidate,mae_median_candidate,mae_p90_candidate,mae_p95_candidate,signed_error_mean_candidate,abs_signed_bias_candidate,...,abs_signed_bias_frozen,severe_error_15bpm_rate_frozen,high_hr_ge_100_rows_frozen,delta_mae_mean,delta_mae_median,delta_mae_p90,delta_mae_p95,delta_signed_error_mean,delta_abs_signed_bias,delta_severe_error_15bpm_rate
0,finetune_test,2828,79.782204,40.0,39.782200,38.040699,58.8912,64.1066,-39.782200,39.782200,...,1.4174,0.0831,244,34.0126,34.4133,45.1928,45.0605,-41.1996,38.3648,0.9006
1,finetune_train,12503,80.237000,40.0,40.237000,38.941200,58.9065,65.0066,-40.237000,40.237000,...,1.5658,0.0846,1129,34.4134,35.1975,45.1873,46.0640,-41.8028,38.6712,0.9032
2,finetune_val,2839,80.161697,40.0,40.161701,41.398399,56.4772,61.4710,-40.161701,40.161701,...,1.4724,0.0891,168,34.1342,37.5451,42.4071,42.2391,-41.6341,38.6893,0.8736



Scratch smoke dataset-role comparison:


,dataset,window_role,rows_candidate,target_hr_mean_candidate,pred_hr_mean_candidate,mae_mean_candidate,mae_median_candidate,mae_p90_candidate,mae_p95_candidate,signed_error_mean_candidate,...,abs_signed_bias_frozen,severe_error_15bpm_rate_frozen,high_hr_ge_100_rows_frozen,delta_mae_mean,delta_mae_median,delta_mae_p90,delta_mae_p95,delta_signed_error_mean,delta_abs_signed_bias,delta_severe_error_15bpm_rate
0,mcd_rppg,finetune_test,2094,79.853302,40.0,39.853298,38.018600,59.3013,63.6733,-39.853298,...,0.9242,0.0525,192,34.9686,34.8684,48.3313,48.1243,-40.7775,38.9291,0.9317
3,ubfc_phys,finetune_test,658,77.420601,40.0,37.420502,37.492298,49.6468,53.7800,-37.420502,...,3.1980,0.1900,7,28.5621,31.8188,29.3606,28.1067,-40.6185,34.2225,0.7902
6,ubfc_rppg,finetune_test,76,98.269302,40.0,58.269299,66.554001,81.1256,83.6714,-58.269299,...,0.4102,0.0000,45,54.8608,63.4087,75.0421,76.8320,-57.8591,57.8591,1.0000
1,mcd_rppg,finetune_train,7446,79.064796,40.0,39.064800,38.117298,56.9220,64.4151,-39.064800,...,1.2645,0.0431,569,34.3772,34.9534,46.5498,50.4182,-40.3293,37.8003,0.9444
4,ubfc_phys,finetune_train,4595,80.305496,40.0,40.305500,39.345299,58.1041,62.7214,-40.305500,...,2.3192,0.1567,367,32.5236,33.9205,39.5113,39.2372,-42.6247,37.9863,0.8302
7,ubfc_rppg,finetune_train,462,98.449203,40.0,58.449200,57.379200,77.1463,83.6050,-58.449200,...,1.0723,0.0368,193,53.7944,54.2345,67.2274,70.4363,-57.3769,57.3769,0.9632
2,mcd_rppg,finetune_val,1743,79.108597,40.0,39.108601,41.553200,56.4610,59.2806,-39.108601,...,1.2656,0.0545,79,33.9761,38.1786,44.9242,43.8022,-40.3742,37.8430,0.9019
5,ubfc_phys,finetune_val,986,80.002403,40.0,40.002399,40.710701,54.2339,55.9675,-40.002399,...,2.2826,0.1531,19,32.2395,35.8646,35.9614,31.7019,-42.2850,37.7198,0.8165
8,ubfc_rppg,finetune_val,110,98.277199,40.0,58.277199,67.900299,81.0300,84.8673,-58.277199,...,2.5126,0.0636,70,53.6239,65.4565,68.6526,69.4656,-55.7646,55.7646,0.9364



Saved scratch smoke history: /kaggle/working/crvse_nb13_app_relevant_without_ecg_fitness/nb13_scratch_smoke_history.csv
Saved scratch smoke predictions: /kaggle/working/crvse_nb13_app_relevant_without_ecg_fitness/nb13_scratch_smoke_predictions.csv
Saved scratch smoke decision: /kaggle/working/crvse_nb13_app_relevant_without_ecg_fitness/nb13_scratch_smoke_decision.csv
Saved scratch smoke dataset comparison: /kaggle/working/crvse_nb13_app_relevant_without_ecg_fitness/nb13_scratch_smoke_dataset_role_compare.csv
Saved scratch smoke checkpoint: /kaggle/working/crvse_nb13_app_relevant_without_ecg_fitness/nb13_scratch_smoke_best_state.pt

PASS: Scratch-training smoke test completed end to end.


## Main Transfer-Learning Experiment

This section runs the first full NB13 transfer-learning experiment.

The model is initialized from the frozen app checkpoint.

The experiment trains a conservative subset of parameters:

- frequency projection,
- final transformer block,
- prediction head.

The encoder and earlier transformer blocks remain frozen.

The purpose is to test whether the existing checkpoint can adapt to the app-relevant still/seated source-FPS tensor distribution without changing the whole representation.


In [10]:
TRANSFER_MAIN_NAME = "transfer_main_last_block_freq_head_lr5e5"

transfer_main_model = build_transfer_model_from_frozen()

transfer_main_trainable_report = set_transfer_trainable_layers(
    transfer_main_model,
    train_encoder=False,
    train_last_transformer_block=True,
    train_head=True,
    train_freq_proj=True,
)

print("Main transfer trainable parameter report:")
display(transfer_main_trainable_report)

transfer_main_result = run_training_loop(
    model=transfer_main_model,
    experiment_name=TRANSFER_MAIN_NAME,
    train_loader=NB13_LOADERS["train"],
    val_loader=NB13_LOADERS["val"],
    max_epochs=NB13_TRAINING_CONFIG["max_epochs_main"],
    lr=NB13_TRAINING_CONFIG["default_lr"],
    weight_decay=NB13_TRAINING_CONFIG["default_weight_decay"],
    huber_beta=NB13_TRAINING_CONFIG["default_huber_beta"],
    grad_clip=NB13_TRAINING_CONFIG["default_grad_clip"],
    patience=NB13_TRAINING_CONFIG["early_stopping_patience"],
)

TRANSFER_MAIN_MODEL = transfer_main_result["model"]
TRANSFER_MAIN_HISTORY = transfer_main_result["history"]

print("\nMain transfer training history:")
display(TRANSFER_MAIN_HISTORY)

TRANSFER_MAIN_PREDICTIONS = evaluate_candidate_model(
    model=TRANSFER_MAIN_MODEL,
    experiment_name=TRANSFER_MAIN_NAME,
)

transfer_main_comparison = compare_nb13_candidate_predictions(
    predictions=TRANSFER_MAIN_PREDICTIONS,
    candidate_name=TRANSFER_MAIN_NAME,
)

TRANSFER_MAIN_DECISION = pd.DataFrame([transfer_main_comparison["decision_row"]])
TRANSFER_MAIN_ROLE_COMPARE = transfer_main_comparison["role_compare"]
TRANSFER_MAIN_DATASET_ROLE_COMPARE = transfer_main_comparison["dataset_role_compare"]

print("\nMain transfer decision row:")
display(TRANSFER_MAIN_DECISION)

print("\nMain transfer role comparison:")
display(TRANSFER_MAIN_ROLE_COMPARE)

print("\nMain transfer dataset-role comparison:")
display(
    TRANSFER_MAIN_DATASET_ROLE_COMPARE.sort_values(["window_role", "dataset"])
)

transfer_main_history_path = NB13_WORKING_DIR / "nb13_transfer_main_history.csv"
transfer_main_predictions_path = NB13_WORKING_DIR / "nb13_transfer_main_predictions.csv"
transfer_main_decision_path = NB13_WORKING_DIR / "nb13_transfer_main_decision.csv"
transfer_main_dataset_compare_path = NB13_WORKING_DIR / "nb13_transfer_main_dataset_role_compare.csv"
transfer_main_checkpoint_path = NB13_WORKING_DIR / "nb13_transfer_main_best_state.pt"

TRANSFER_MAIN_HISTORY.to_csv(transfer_main_history_path, index=False)
TRANSFER_MAIN_PREDICTIONS.to_csv(transfer_main_predictions_path, index=False)
TRANSFER_MAIN_DECISION.to_csv(transfer_main_decision_path, index=False)
TRANSFER_MAIN_DATASET_ROLE_COMPARE.to_csv(transfer_main_dataset_compare_path, index=False)

torch.save(
    {
        "experiment": TRANSFER_MAIN_NAME,
        "model_state": TRANSFER_MAIN_MODEL.state_dict(),
        "best_epoch": transfer_main_result["best_epoch"],
        "best_val_mae": transfer_main_result["best_val_mae"],
        "training_config": NB13_TRAINING_CONFIG,
        "trainable_report": transfer_main_trainable_report.to_dict(orient="records"),
        "decision_row": transfer_main_comparison["decision_row"],
        "note": "Main NB13 transfer-learning candidate. Requires decision-table review before any app use.",
    },
    transfer_main_checkpoint_path,
)

print(f"\nSaved main transfer history: {transfer_main_history_path}")
print(f"Saved main transfer predictions: {transfer_main_predictions_path}")
print(f"Saved main transfer decision: {transfer_main_decision_path}")
print(f"Saved main transfer dataset comparison: {transfer_main_dataset_compare_path}")
print(f"Saved main transfer checkpoint: {transfer_main_checkpoint_path}")

print("\nPASS: Main transfer-learning experiment completed.")

Main transfer trainable parameter report:


,module,total_params,trainable_params,trainable_pct
0,encoder,1696,0,0.0
1,freq_proj,47680,47680,100.0
2,transformer_layers,270144,67536,25.0
3,head,2625,2625,100.0


transfer_main_last_block_freq_head_lr5e5 | epoch 01 | train MAE 8.6436 | val MAE 5.9241 | best 5.9241 @ epoch 1
transfer_main_last_block_freq_head_lr5e5 | epoch 02 | train MAE 8.5391 | val MAE 5.9570 | best 5.9241 @ epoch 1
transfer_main_last_block_freq_head_lr5e5 | epoch 03 | train MAE 8.5635 | val MAE 6.0294 | best 5.9241 @ epoch 1
transfer_main_last_block_freq_head_lr5e5 | epoch 04 | train MAE 8.4655 | val MAE 5.9038 | best 5.9038 @ epoch 4
transfer_main_last_block_freq_head_lr5e5 | epoch 05 | train MAE 8.5435 | val MAE 5.9480 | best 5.9038 @ epoch 4
transfer_main_last_block_freq_head_lr5e5 | epoch 06 | train MAE 8.4693 | val MAE 5.9211 | best 5.9038 @ epoch 4
transfer_main_last_block_freq_head_lr5e5 | epoch 07 | train MAE 8.4713 | val MAE 5.9647 | best 5.9038 @ epoch 4
transfer_main_last_block_freq_head_lr5e5 | epoch 08 | train MAE 8.5714 | val MAE 6.0647 | best 5.9038 @ epoch 4
transfer_main_last_block_freq_head_lr5e5 | epoch 09 | train MAE 8.5271 | val MAE 5.9651 | best 5.9038 @ 

,experiment,epoch,train_rows,train_loss,train_mae,grad_norm_mean,grad_norm_max,val_mae,val_median,val_p90,val_p95,val_signed_error_mean,improved,best_epoch_so_far,best_val_mae_so_far
0,transfer_main_last_block_freq_head_lr5e5,1,12503,6.119200,8.643608,18.029429,35.313885,5.924100,3.801048,14.076461,18.824179,0.913936,True,1,5.924100
1,transfer_main_last_block_freq_head_lr5e5,2,12503,6.025935,8.539095,17.887398,43.895088,5.957026,3.855118,14.164666,18.892466,0.973203,False,1,5.924100
2,transfer_main_last_block_freq_head_lr5e5,3,12503,6.047940,8.563492,17.867383,48.385082,6.029412,3.929569,14.185478,19.017250,1.595197,False,1,5.924100
3,transfer_main_last_block_freq_head_lr5e5,4,12503,5.954383,8.465495,17.033598,34.919708,5.903828,3.812233,14.070310,18.867279,0.844978,True,4,5.903828
4,transfer_main_last_block_freq_head_lr5e5,5,12503,6.032529,8.543451,18.762254,48.713390,5.948047,3.844666,14.220149,19.016270,1.337607,False,4,5.903828
5,transfer_main_last_block_freq_head_lr5e5,6,12503,5.964404,8.469290,18.383653,43.730797,5.921099,3.797768,14.147566,19.135893,1.139239,False,4,5.903828
6,transfer_main_last_block_freq_head_lr5e5,7,12503,5.962994,8.471254,15.724218,34.546062,5.964741,3.890026,14.110260,19.073204,1.286364,False,4,5.903828
7,transfer_main_last_block_freq_head_lr5e5,8,12503,6.050385,8.571437,17.886010,33.956280,6.064707,3.941368,14.312696,19.629860,1.700523,False,4,5.903828
8,transfer_main_last_block_freq_head_lr5e5,9,12503,6.006488,8.527051,19.395618,55.362469,5.965114,3.844353,14.245091,19.138489,1.570243,False,4,5.903828



Main transfer decision row:


...


Main transfer role comparison:


,window_role,rows_candidate,target_hr_mean_candidate,pred_hr_mean_candidate,mae_mean_candidate,mae_median_candidate,mae_p90_candidate,mae_p95_candidate,signed_error_mean_candidate,abs_signed_bias_candidate,...,abs_signed_bias_frozen,severe_error_15bpm_rate_frozen,high_hr_ge_100_rows_frozen,delta_mae_mean,delta_mae_median,delta_mae_p90,delta_mae_p95,delta_signed_error_mean,delta_abs_signed_bias,delta_severe_error_15bpm_rate
0,finetune_test,2828,79.782204,80.754799,5.6142,3.4709,12.9307,19.0050,0.9726,0.9726,...,1.4174,0.0831,244,-0.1554,-0.1565,-0.7677,-0.0411,-0.4448,-0.4448,-0.0014
1,finetune_train,12503,80.237000,81.241501,5.5929,3.6000,13.1742,18.1172,1.0044,1.0044,...,1.5658,0.0846,1129,-0.2307,-0.1437,-0.5450,-0.8254,-0.5614,-0.5614,-0.0072
2,finetune_val,2839,80.161697,81.006699,5.9038,3.8122,14.0703,18.8673,0.8450,0.8450,...,1.4724,0.0891,168,-0.1237,-0.0411,0.0002,-0.3646,-0.6274,-0.6274,-0.0014



Main transfer dataset-role comparison:


,dataset,window_role,rows_candidate,target_hr_mean_candidate,pred_hr_mean_candidate,mae_mean_candidate,mae_median_candidate,mae_p90_candidate,mae_p95_candidate,signed_error_mean_candidate,...,abs_signed_bias_frozen,severe_error_15bpm_rate_frozen,high_hr_ge_100_rows_frozen,delta_mae_mean,delta_mae_median,delta_mae_p90,delta_mae_p95,delta_signed_error_mean,delta_abs_signed_bias,delta_severe_error_15bpm_rate
0,mcd_rppg,finetune_test,2094,79.853302,80.351700,4.7241,2.9564,10.5270,15.1855,0.4984,...,0.9242,0.0525,192,-0.1606,-0.1938,-0.4430,-0.3635,-0.4258,-0.4258,-0.0004
3,ubfc_phys,finetune_test,658,77.420601,80.052803,8.6923,5.5300,19.6061,24.9449,2.6322,...,3.1980,0.1900,7,-0.1661,-0.1435,-0.6801,-0.7284,-0.5658,-0.5658,-0.0046
6,ubfc_rppg,finetune_test,76,98.269302,97.938599,3.4893,3.1737,6.0822,7.1456,-0.3307,...,0.4102,0.0000,45,0.0808,0.0284,-0.0013,0.3062,0.0795,-0.0795,0.0000
1,mcd_rppg,finetune_train,7446,79.064796,79.801598,4.3928,2.9781,9.7303,13.3038,0.7368,...,1.2645,0.0431,569,-0.2948,-0.1858,-0.6419,-0.6931,-0.5277,-0.5277,-0.0067
4,ubfc_phys,finetune_train,4595,80.305496,81.991798,7.6263,5.3416,18.1973,22.7008,1.6863,...,2.3192,0.1567,367,-0.1556,-0.0832,-0.3955,-0.7834,-0.6329,-0.6329,-0.0085
7,ubfc_rppg,finetune_train,462,98.449203,96.986099,4.7107,3.4183,9.9607,13.0830,-1.4631,...,1.0723,0.0368,193,0.0559,0.2736,0.0418,-0.0857,-0.3908,0.3908,-0.0022
2,mcd_rppg,finetune_val,1743,79.108597,79.686600,4.9307,3.3634,11.1311,15.1778,0.5780,...,1.2656,0.0545,79,-0.2018,-0.0112,-0.4057,-0.3006,-0.6876,-0.6876,-0.0029
5,ubfc_phys,finetune_val,986,80.002403,81.708603,7.7480,5.0549,18.1335,23.6791,1.7062,...,2.2826,0.1531,19,-0.0149,0.2088,-0.1390,-0.5865,-0.5764,-0.5764,0.0011
8,ubfc_rppg,finetune_val,110,98.277199,95.632896,4.7925,2.5206,13.4386,16.6231,-2.6443,...,2.5126,0.0636,70,0.1392,0.0768,1.0612,1.2214,-0.1317,0.1317,0.0000



Saved main transfer history: /kaggle/working/crvse_nb13_app_relevant_without_ecg_fitness/nb13_transfer_main_history.csv
Saved main transfer predictions: /kaggle/working/crvse_nb13_app_relevant_without_ecg_fitness/nb13_transfer_main_predictions.csv
Saved main transfer decision: /kaggle/working/crvse_nb13_app_relevant_without_ecg_fitness/nb13_transfer_main_decision.csv
Saved main transfer dataset comparison: /kaggle/working/crvse_nb13_app_relevant_without_ecg_fitness/nb13_transfer_main_dataset_role_compare.csv
Saved main transfer checkpoint: /kaggle/working/crvse_nb13_app_relevant_without_ecg_fitness/nb13_transfer_main_best_state.pt

PASS: Main transfer-learning experiment completed.


### NB13 broader transfer branch: all transformer layers with conservative learning rate

This notebook now tests whether the previous transfer-learning branch was too constrained by updating only the final transformer block. The new branch keeps the input encoder frozen, but allows the frequency projection, all transformer layers, and regression head to adapt to the app-relevant source-FPS tensors.

This experiment remains scoped to still/seated webcam rPPG datasets only: MCD-rPPG, UBFC-rPPG, and UBFC-Phys. ECG-Fitness remains excluded from model selection, so this branch does not test exercise, high-motion, or exercise-related high-HR robustness.

The candidate should only be considered stronger than the frozen baseline if it improves validation MAE by at least the predefined threshold, improves held-out test MAE, and does not create unacceptable per-dataset regressions, p90-error regressions, severe-error increases, or signed-bias increases.

In [11]:
TRANSFER_ALL_EXPERIMENT = "transfer_main_all_transformer_freq_head_lr2e5"

TRANSFER_ALL_CONFIG = {
    "seed": 42,
    "train_batch_size": 256,
    "eval_batch_size": 512,
    "lr": 2e-5,
    "weight_decay": 1e-4,
    "huber_beta": 6.0,
    "grad_clip": 5.0,
    "max_epochs": 20,
    "early_stopping_patience": 5,
    "train_encoder": False,
    "train_freq_proj": True,
    "train_all_transformer_layers": True,
    "train_head": True,
}


def get_device_object() -> torch.device:
    """Return DEVICE as a torch.device object."""
    if isinstance(DEVICE, torch.device):
        return DEVICE
    return torch.device(str(DEVICE))


DEVICE_OBJ = get_device_object()
PIN_MEMORY = DEVICE_OBJ.type == "cuda"


def seed_everything(seed: int) -> None:
    """Seed Python, NumPy, and PyTorch for a reproducible notebook branch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_training_default(name: str, fallback):
    """Read a prior notebook training default when it exists."""
    config = globals().get("NB13_TRAINING_CONFIG", {})

    if isinstance(config, dict):
        return config.get(name, fallback)

    return fallback


TRANSFER_ALL_CONFIG["weight_decay"] = get_training_default(
    "default_weight_decay",
    TRANSFER_ALL_CONFIG["weight_decay"],
)
TRANSFER_ALL_CONFIG["huber_beta"] = get_training_default(
    "default_huber_beta",
    TRANSFER_ALL_CONFIG["huber_beta"],
)
TRANSFER_ALL_CONFIG["grad_clip"] = get_training_default(
    "default_grad_clip",
    TRANSFER_ALL_CONFIG["grad_clip"],
)
TRANSFER_ALL_CONFIG["max_epochs"] = get_training_default(
    "max_epochs_main",
    TRANSFER_ALL_CONFIG["max_epochs"],
)
TRANSFER_ALL_CONFIG["early_stopping_patience"] = get_training_default(
    "early_stopping_patience",
    TRANSFER_ALL_CONFIG["early_stopping_patience"],
)


def build_fresh_transfer_model() -> nn.Module:
    """Build a fresh model initialized from the frozen checkpoint."""
    if "build_transfer_model_from_frozen" in globals():
        model = build_transfer_model_from_frozen()
    elif "build_physformer_from_checkpoint" in globals() and "FROZEN_CHECKPOINT" in globals():
        model = build_physformer_from_checkpoint(FROZEN_CHECKPOINT)
    elif (
        "build_physformer_from_checkpoint" in globals()
        and "load_checkpoint_file" in globals()
        and "CHECKPOINT_PATH" in globals()
    ):
        model = build_physformer_from_checkpoint(load_checkpoint_file(CHECKPOINT_PATH))
    else:
        raise NameError(
            "Could not build a fresh transfer model. Expected "
            "build_transfer_model_from_frozen() or build_physformer_from_checkpoint()."
        )

    return model.to(DEVICE_OBJ)


def set_all_transformer_freq_head_trainable(model: nn.Module) -> nn.Module:
    """Freeze the encoder and train freq_proj, all transformer layers, and head."""
    for param in model.parameters():
        param.requires_grad = False

    for module_name in ["freq_proj", "transformer_layers", "head"]:
        module = getattr(model, module_name, None)

        if module is None:
            raise AttributeError(f"Model is missing expected module: {module_name}")

        for param in module.parameters():
            param.requires_grad = True

    return model


def summarize_major_module_trainability(model: nn.Module) -> pd.DataFrame:
    """Summarize trainable parameters by major PhysFormer module."""
    rows = []

    for module_name in ["encoder", "freq_proj", "transformer_layers", "head"]:
        module = getattr(model, module_name, None)

        if module is None:
            continue

        total_params = sum(param.numel() for param in module.parameters())
        trainable_params = sum(
            param.numel() for param in module.parameters() if param.requires_grad
        )

        rows.append(
            {
                "module": module_name,
                "total_params": int(total_params),
                "trainable_params": int(trainable_params),
                "trainable_pct": round(
                    100.0 * trainable_params / total_params,
                    4,
                )
                if total_params
                else 0.0,
            }
        )

    return pd.DataFrame(rows)


def make_nb13_loader(role: str, batch_size: int, shuffle: bool) -> DataLoader:
    """Create a DataLoader for one NB13 tensor dataset role."""
    if "NB13_DATASETS" not in globals():
        raise NameError("NB13_DATASETS is not defined. Run the dataset/DataLoader cell first.")

    if role not in NB13_DATASETS:
        raise KeyError(f"NB13_DATASETS is missing role: {role}")

    generator = None

    if shuffle:
        generator = torch.Generator()
        generator.manual_seed(int(TRANSFER_ALL_CONFIG["seed"]))

    return DataLoader(
        NB13_DATASETS[role],
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=PIN_MEMORY,
        generator=generator,
    )


def summarize_prediction_side(
    table: pd.DataFrame,
    pred_col: str,
    error_col: str,
    group_cols: list[str],
    severe_threshold_bpm: float = 15.0,
    high_hr_threshold_bpm: float = 100.0,
) -> pd.DataFrame:
    """Summarize one prediction table by role or dataset-role."""
    summary = (
        table
        .groupby(group_cols, dropna=False)
        .agg(
            rows=("target_hr_from_tensor", "size"),
            target_hr_mean=("target_hr_from_tensor", "mean"),
            pred_hr_mean=(pred_col, "mean"),
            mae_mean=(error_col, "mean"),
            mae_median=(error_col, "median"),
            mae_p90=(error_col, lambda values: float(np.quantile(values, 0.90))),
            mae_p95=(error_col, lambda values: float(np.quantile(values, 0.95))),
            signed_error_mean=(
                pred_col,
                lambda values: float(
                    np.mean(values.to_numpy() - table.loc[values.index, "target_hr_from_tensor"].to_numpy())
                ),
            ),
            severe_error_15bpm_rate=(
                error_col,
                lambda values: float((values >= severe_threshold_bpm).mean()),
            ),
            high_hr_ge_100_rows=(
                "target_hr_from_tensor",
                lambda values: int((values >= high_hr_threshold_bpm).sum()),
            ),
        )
        .reset_index()
    )

    summary["abs_signed_bias"] = summary["signed_error_mean"].abs()
    return summary


def evaluate_model_loader(
    model: nn.Module,
    role: str,
    loader: DataLoader,
    experiment_name: str,
) -> pd.DataFrame:
    """Run one model over one NB13 role and return row-level predictions."""
    dataset = loader.dataset
    model.eval()

    output_parts = []
    fallback_start_index = 0

    with torch.no_grad():
        for batch in loader:
            if len(batch) >= 3:
                x_batch, y_batch, index_batch = batch[:3]
                index_np = index_batch.cpu().numpy().astype(int)
            else:
                x_batch, y_batch = batch[:2]
                batch_size = int(y_batch.shape[0])
                index_np = np.arange(fallback_start_index, fallback_start_index + batch_size)
                fallback_start_index += batch_size

            x_batch = x_batch.to(DEVICE_OBJ, non_blocking=PIN_MEMORY)
            pred_batch = model(x_batch).detach().cpu().reshape(-1).numpy().astype(np.float32)
            target_batch = y_batch.detach().cpu().reshape(-1).numpy().astype(np.float32)

            metadata = dataset.metadata.iloc[index_np].reset_index(drop=True).copy()

            if "window_role" not in metadata.columns:
                metadata["window_role"] = role

            metadata["experiment"] = experiment_name
            metadata["role"] = role
            metadata["target_hr_from_tensor"] = target_batch
            metadata["candidate_model_hr"] = pred_batch
            metadata["candidate_model_abs_error"] = np.abs(pred_batch - target_batch)

            output_parts.append(metadata)

    return pd.concat(output_parts, ignore_index=True)


def normalize_prediction_keys(table: pd.DataFrame) -> pd.DataFrame:
    """Normalize row identity columns before joining prediction tables."""
    output = table.copy()

    join_keys = [
        "dataset",
        "window_role",
        "subject_id",
        "recording_id",
        "start_frame",
        "end_frame",
    ]

    missing = [col for col in join_keys if col not in output.columns]

    if missing:
        raise KeyError(f"Prediction table is missing join key columns: {missing}")

    for col in ["dataset", "window_role", "subject_id", "recording_id"]:
        output[col] = output[col].astype(str)

    for col in ["start_frame", "end_frame"]:
        output[col] = pd.to_numeric(output[col], errors="raise").astype(int)

    return output


def find_baseline_predictions_table() -> pd.DataFrame:
    """Find the frozen source-FPS baseline prediction table from prior cells or disk."""
    for variable_name in [
        "sourcefps_app",
        "SOURCEFPS_APP",
        "FROZEN_NB13_BASELINE_PREDICTIONS",
        "baseline_predictions",
    ]:
        if variable_name in globals() and isinstance(globals()[variable_name], pd.DataFrame):
            return globals()[variable_name].copy()

    for path_name in [
        "BASELINE_PREDICTIONS_PATH",
        "baseline_predictions_path",
    ]:
        if path_name in globals():
            candidate_path = Path(globals()[path_name])

            if candidate_path.exists():
                return pd.read_csv(candidate_path)

    if "LIVE_COMPATIBLE_DIR" in globals():
        candidate_path = (
            Path(LIVE_COMPATIBLE_DIR)
            / "live_finetune_frozen_baseline_predictions.csv"
        )

        if candidate_path.exists():
            return pd.read_csv(candidate_path)

    raise FileNotFoundError(
        "Could not find the frozen source-FPS baseline predictions table."
    )


def get_app_dataset_values() -> list[str]:
    """Read the app-relevant dataset names from the loaded NB13 datasets."""
    values = []

    for dataset in NB13_DATASETS.values():
        if hasattr(dataset, "metadata") and "dataset" in dataset.metadata.columns:
            values.extend(dataset.metadata["dataset"].astype(str).dropna().tolist())

    return sorted(set(values))


def compare_candidate_to_frozen_baseline(
    candidate_predictions: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Compare candidate predictions against the frozen baseline row by row."""
    join_keys = [
        "dataset",
        "window_role",
        "subject_id",
        "recording_id",
        "start_frame",
        "end_frame",
    ]

    baseline = find_baseline_predictions_table()
    app_dataset_values = get_app_dataset_values()

    if app_dataset_values and "dataset" in baseline.columns:
        baseline = baseline[
            baseline["dataset"].astype(str).isin(app_dataset_values)
        ].copy()

    baseline_prediction_column = None

    for candidate_column in [
        "model_hr",
        "baseline_model_hr",
        "notebook_model_hr",
        "pred_hr",
        "predicted_hr",
    ]:
        if candidate_column in baseline.columns:
            baseline_prediction_column = candidate_column
            break

    if baseline_prediction_column is None:
        raise KeyError(
            "Could not identify the frozen baseline prediction column. "
            f"Available columns: {sorted(baseline.columns.tolist())}"
        )

    candidate_join = normalize_prediction_keys(candidate_predictions)
    baseline_join = normalize_prediction_keys(baseline)

    baseline_keep = baseline_join[
        join_keys + [baseline_prediction_column]
    ].rename(
        columns={baseline_prediction_column: "frozen_model_hr"}
    )

    comparison = candidate_join.merge(
        baseline_keep,
        on=join_keys,
        how="left",
        validate="one_to_one",
    )

    comparison["frozen_model_hr"] = pd.to_numeric(
        comparison["frozen_model_hr"],
        errors="coerce",
    )

    missing_baseline_rows = int(comparison["frozen_model_hr"].isna().sum())

    if missing_baseline_rows:
        raise AssertionError(
            f"Candidate predictions are missing frozen baseline matches: {missing_baseline_rows}"
        )

    comparison["candidate_model_hr"] = pd.to_numeric(
        comparison["candidate_model_hr"],
        errors="raise",
    )
    comparison["target_hr_from_tensor"] = pd.to_numeric(
        comparison["target_hr_from_tensor"],
        errors="raise",
    )

    comparison["candidate_model_abs_error"] = (
        comparison["candidate_model_hr"] - comparison["target_hr_from_tensor"]
    ).abs()
    comparison["frozen_model_abs_error"] = (
        comparison["frozen_model_hr"] - comparison["target_hr_from_tensor"]
    ).abs()

    candidate_role = summarize_prediction_side(
        comparison,
        pred_col="candidate_model_hr",
        error_col="candidate_model_abs_error",
        group_cols=["window_role"],
    )
    frozen_role = summarize_prediction_side(
        comparison,
        pred_col="frozen_model_hr",
        error_col="frozen_model_abs_error",
        group_cols=["window_role"],
    )

    candidate_dataset_role = summarize_prediction_side(
        comparison,
        pred_col="candidate_model_hr",
        error_col="candidate_model_abs_error",
        group_cols=["dataset", "window_role"],
    )
    frozen_dataset_role = summarize_prediction_side(
        comparison,
        pred_col="frozen_model_hr",
        error_col="frozen_model_abs_error",
        group_cols=["dataset", "window_role"],
    )

    role_compare = merge_candidate_and_frozen_summaries(
        candidate_role,
        frozen_role,
        group_cols=["window_role"],
    )
    dataset_role_compare = merge_candidate_and_frozen_summaries(
        candidate_dataset_role,
        frozen_dataset_role,
        group_cols=["dataset", "window_role"],
    )

    return comparison, role_compare, dataset_role_compare


def merge_candidate_and_frozen_summaries(
    candidate_summary: pd.DataFrame,
    frozen_summary: pd.DataFrame,
    group_cols: list[str],
) -> pd.DataFrame:
    """Merge candidate and frozen summary tables and add delta columns."""
    candidate_renamed = candidate_summary.rename(
        columns={
            col: f"{col}_candidate"
            for col in candidate_summary.columns
            if col not in group_cols
        }
    )
    frozen_renamed = frozen_summary.rename(
        columns={
            col: f"{col}_frozen"
            for col in frozen_summary.columns
            if col not in group_cols
        }
    )

    merged = candidate_renamed.merge(
        frozen_renamed,
        on=group_cols,
        how="left",
        validate="one_to_one",
    )

    for metric in [
        "mae_mean",
        "mae_median",
        "mae_p90",
        "mae_p95",
        "signed_error_mean",
        "abs_signed_bias",
        "severe_error_15bpm_rate",
    ]:
        merged[f"delta_{metric}"] = (
            merged[f"{metric}_candidate"] - merged[f"{metric}_frozen"]
        )

    return merged


def make_nb13_candidate_decision(
    experiment_name: str,
    role_compare: pd.DataFrame,
    dataset_role_compare: pd.DataFrame,
) -> pd.DataFrame:
    """Apply the predefined NB13 candidate decision thresholds."""
    thresholds = {
        "minimum_val_gain_bpm": 0.20,
        "minimum_test_gain_bpm": 0.10,
        "maximum_dataset_test_regression_bpm": 0.30,
        "maximum_ubfc_phys_test_regression_bpm": 0.10,
        "maximum_p90_regression_bpm": 0.25,
        "maximum_severe_error_rate_increase": 0.01,
        "maximum_abs_signed_bias_increase_bpm": 0.25,
    }

    role_index = role_compare.set_index("window_role")

    val_delta_mae_mean = float(role_index.loc["finetune_val", "delta_mae_mean"])
    test_delta_mae_mean = float(role_index.loc["finetune_test", "delta_mae_mean"])
    val_delta_mae_p90 = float(role_index.loc["finetune_val", "delta_mae_p90"])
    test_delta_mae_p90 = float(role_index.loc["finetune_test", "delta_mae_p90"])
    test_delta_severe_error_rate = float(
        role_index.loc["finetune_test", "delta_severe_error_15bpm_rate"]
    )
    test_delta_abs_signed_bias = float(
        role_index.loc["finetune_test", "delta_abs_signed_bias"]
    )

    test_dataset_rows = dataset_role_compare[
        dataset_role_compare["window_role"] == "finetune_test"
    ].copy()

    max_dataset_test_delta_mae = float(test_dataset_rows["delta_mae_mean"].max())

    ubfc_phys_rows = test_dataset_rows[test_dataset_rows["dataset"] == "ubfc_phys"]

    if len(ubfc_phys_rows) == 0:
        ubfc_phys_test_delta_mae = np.nan
    else:
        ubfc_phys_test_delta_mae = float(ubfc_phys_rows["delta_mae_mean"].iloc[0])

    gains_ok = (
        val_delta_mae_mean <= -thresholds["minimum_val_gain_bpm"]
        and test_delta_mae_mean <= -thresholds["minimum_test_gain_bpm"]
    )

    guardrails_ok = (
        np.isfinite(max_dataset_test_delta_mae)
        and np.isfinite(ubfc_phys_test_delta_mae)
        and max_dataset_test_delta_mae
        <= thresholds["maximum_dataset_test_regression_bpm"]
        and ubfc_phys_test_delta_mae
        <= thresholds["maximum_ubfc_phys_test_regression_bpm"]
        and test_delta_mae_p90 <= thresholds["maximum_p90_regression_bpm"]
        and test_delta_severe_error_rate
        <= thresholds["maximum_severe_error_rate_increase"]
        and test_delta_abs_signed_bias
        <= thresholds["maximum_abs_signed_bias_increase_bpm"]
    )

    if gains_ok and guardrails_ok:
        decision_status = "passes_nb13_candidate_policy"
    elif guardrails_ok and (val_delta_mae_mean < 0 or test_delta_mae_mean < 0):
        decision_status = "weak_or_incomplete_candidate"
    else:
        decision_status = "reject"

    decision = pd.DataFrame(
        [
            {
                "candidate": experiment_name,
                "val_delta_mae_mean": val_delta_mae_mean,
                "test_delta_mae_mean": test_delta_mae_mean,
                "val_delta_mae_p90": val_delta_mae_p90,
                "test_delta_mae_p90": test_delta_mae_p90,
                "test_delta_severe_error_rate": test_delta_severe_error_rate,
                "test_delta_abs_signed_bias": test_delta_abs_signed_bias,
                "max_dataset_test_delta_mae": max_dataset_test_delta_mae,
                "ubfc_phys_test_delta_mae": ubfc_phys_test_delta_mae,
                "decision_status": decision_status,
            }
        ]
    )

    numeric_cols = decision.select_dtypes(include=[np.number]).columns
    decision[numeric_cols] = decision[numeric_cols].round(4)

    return decision


def train_candidate_model(
    model: nn.Module,
    experiment_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    config: dict,
) -> tuple[pd.DataFrame, dict, float, int]:
    """Train one NB13 candidate and return history plus the best state."""
    trainable_params = [
        param for param in model.parameters() if param.requires_grad
    ]

    if not trainable_params:
        raise ValueError("No trainable parameters are enabled for this candidate.")

    optimizer = torch.optim.AdamW(
        trainable_params,
        lr=float(config["lr"]),
        weight_decay=float(config["weight_decay"]),
    )
    loss_fn = nn.SmoothL1Loss(beta=float(config["huber_beta"]))

    best_val_mae = float("inf")
    best_epoch = -1
    best_state = None
    epochs_without_improvement = 0
    history_rows = []

    for epoch in range(1, int(config["max_epochs"]) + 1):
        model.train()

        total_loss = 0.0
        total_abs_error = 0.0
        total_rows = 0
        grad_norms = []

        for x_batch, y_batch, *_ in train_loader:
            x_batch = x_batch.to(DEVICE_OBJ, non_blocking=PIN_MEMORY)
            y_batch = y_batch.to(DEVICE_OBJ, non_blocking=PIN_MEMORY).reshape(-1)

            optimizer.zero_grad(set_to_none=True)

            pred_batch = model(x_batch).reshape(-1)
            loss = loss_fn(pred_batch, y_batch)

            loss.backward()

            grad_norm = torch.nn.utils.clip_grad_norm_(
                trainable_params,
                max_norm=float(config["grad_clip"]),
            )
            grad_norms.append(float(grad_norm.detach().cpu()))

            optimizer.step()

            batch_rows = int(y_batch.shape[0])
            total_rows += batch_rows
            total_loss += float(loss.detach().cpu()) * batch_rows
            total_abs_error += float(
                torch.abs(pred_batch.detach() - y_batch).sum().detach().cpu()
            )

        val_predictions = evaluate_model_loader(
            model=model,
            role="finetune_val",
            loader=val_loader,
            experiment_name=experiment_name,
        )
        val_summary = summarize_prediction_side(
            val_predictions,
            pred_col="candidate_model_hr",
            error_col="candidate_model_abs_error",
            group_cols=["window_role"],
        )
        val_row = val_summary.iloc[0]

        train_loss = total_loss / max(total_rows, 1)
        train_mae = total_abs_error / max(total_rows, 1)
        val_mae = float(val_row["mae_mean"])

        improved = val_mae < (best_val_mae - 1e-8)

        if improved:
            best_val_mae = val_mae
            best_epoch = epoch
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        history_rows.append(
            {
                "experiment": experiment_name,
                "epoch": epoch,
                "train_rows": int(total_rows),
                "train_loss": train_loss,
                "train_mae": train_mae,
                "grad_norm_mean": float(np.mean(grad_norms)) if grad_norms else np.nan,
                "grad_norm_max": float(np.max(grad_norms)) if grad_norms else np.nan,
                "val_mae": val_mae,
                "val_median": float(val_row["mae_median"]),
                "val_p90": float(val_row["mae_p90"]),
                "val_p95": float(val_row["mae_p95"]),
                "val_signed_error_mean": float(val_row["signed_error_mean"]),
                "improved": bool(improved),
                "best_epoch_so_far": int(best_epoch),
                "best_val_mae_so_far": float(best_val_mae),
            }
        )

        print(
            f"{experiment_name} | epoch {epoch:02d} | "
            f"train MAE {train_mae:.4f} | val MAE {val_mae:.4f} | "
            f"best {best_val_mae:.4f} @ epoch {best_epoch}"
        )

        if epochs_without_improvement >= int(config["early_stopping_patience"]):
            print(f"Early stopping after epoch {epoch}.")
            break

    if best_state is None:
        raise RuntimeError("Training finished without a best checkpoint state.")

    history = pd.DataFrame(history_rows)
    numeric_cols = history.select_dtypes(include=[np.number]).columns
    history[numeric_cols] = history[numeric_cols].round(6)

    return history, best_state, best_val_mae, best_epoch


seed_everything(int(TRANSFER_ALL_CONFIG["seed"]))

if DEVICE_OBJ.type == "cuda":
    torch.cuda.empty_cache()

TRANSFER_ALL_TRAIN_LOADER = make_nb13_loader(
    role="finetune_train",
    batch_size=int(TRANSFER_ALL_CONFIG["train_batch_size"]),
    shuffle=True,
)
TRANSFER_ALL_EVAL_LOADERS_BY_ROLE = {
    role: make_nb13_loader(
        role=role,
        batch_size=int(TRANSFER_ALL_CONFIG["eval_batch_size"]),
        shuffle=False,
    )
    for role in ["finetune_train", "finetune_val", "finetune_test"]
}

TRANSFER_ALL_MODEL = build_fresh_transfer_model()
TRANSFER_ALL_MODEL = set_all_transformer_freq_head_trainable(TRANSFER_ALL_MODEL)

TRANSFER_ALL_PARAM_REPORT = summarize_major_module_trainability(TRANSFER_ALL_MODEL)

print("Main all-transformer transfer trainable parameter report:")
display(TRANSFER_ALL_PARAM_REPORT)

TRANSFER_ALL_HISTORY, TRANSFER_ALL_BEST_STATE, TRANSFER_ALL_BEST_VAL_MAE, TRANSFER_ALL_BEST_EPOCH = (
    train_candidate_model(
        model=TRANSFER_ALL_MODEL,
        experiment_name=TRANSFER_ALL_EXPERIMENT,
        train_loader=TRANSFER_ALL_TRAIN_LOADER,
        val_loader=TRANSFER_ALL_EVAL_LOADERS_BY_ROLE["finetune_val"],
        config=TRANSFER_ALL_CONFIG,
    )
)

TRANSFER_ALL_MODEL.load_state_dict(TRANSFER_ALL_BEST_STATE)
TRANSFER_ALL_MODEL.to(DEVICE_OBJ)
TRANSFER_ALL_MODEL.eval()

TRANSFER_ALL_PREDICTIONS = pd.concat(
    [
        evaluate_model_loader(
            model=TRANSFER_ALL_MODEL,
            role=role,
            loader=loader,
            experiment_name=TRANSFER_ALL_EXPERIMENT,
        )
        for role, loader in TRANSFER_ALL_EVAL_LOADERS_BY_ROLE.items()
    ],
    ignore_index=True,
)

(
    TRANSFER_ALL_COMPARE,
    TRANSFER_ALL_ROLE_COMPARE,
    TRANSFER_ALL_DATASET_ROLE_COMPARE,
) = compare_candidate_to_frozen_baseline(TRANSFER_ALL_PREDICTIONS)

TRANSFER_ALL_DECISION = make_nb13_candidate_decision(
    experiment_name=TRANSFER_ALL_EXPERIMENT,
    role_compare=TRANSFER_ALL_ROLE_COMPARE,
    dataset_role_compare=TRANSFER_ALL_DATASET_ROLE_COMPARE,
)

for table in [
    TRANSFER_ALL_ROLE_COMPARE,
    TRANSFER_ALL_DATASET_ROLE_COMPARE,
]:
    numeric_cols = table.select_dtypes(include=[np.number]).columns
    table[numeric_cols] = table[numeric_cols].round(4)

OUTPUT_DIR = Path(NB13_WORKING_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

history_path = OUTPUT_DIR / "nb13_transfer_main_all_transformer_history.csv"
predictions_path = OUTPUT_DIR / "nb13_transfer_main_all_transformer_predictions.csv"
decision_path = OUTPUT_DIR / "nb13_transfer_main_all_transformer_decision.csv"
dataset_compare_path = OUTPUT_DIR / "nb13_transfer_main_all_transformer_dataset_role_compare.csv"
checkpoint_path = OUTPUT_DIR / "nb13_transfer_main_all_transformer_best_state.pt"

TRANSFER_ALL_HISTORY.to_csv(history_path, index=False)
TRANSFER_ALL_PREDICTIONS.to_csv(predictions_path, index=False)
TRANSFER_ALL_DECISION.to_csv(decision_path, index=False)
TRANSFER_ALL_DATASET_ROLE_COMPARE.to_csv(dataset_compare_path, index=False)

torch.save(
    {
        "experiment": TRANSFER_ALL_EXPERIMENT,
        "model_state": TRANSFER_ALL_BEST_STATE,
        "best_val_mae": float(TRANSFER_ALL_BEST_VAL_MAE),
        "best_epoch": int(TRANSFER_ALL_BEST_EPOCH),
        "training_config": TRANSFER_ALL_CONFIG,
        "decision": TRANSFER_ALL_DECISION.to_dict(orient="records")[0],
        "source_checkpoint_path": str(globals().get("CHECKPOINT_PATH", "")),
        "architecture_note": (
            "NB13 broader transfer branch initialized from the frozen source-FPS "
            "checkpoint; encoder frozen; freq_proj, all transformer layers, and "
            "head trainable."
        ),
    },
    checkpoint_path,
)

print("\nMain all-transformer transfer training history:")
display(TRANSFER_ALL_HISTORY)

print("\nMain all-transformer transfer decision row:")
display(TRANSFER_ALL_DECISION)

print("\nMain all-transformer transfer role comparison:")
display(TRANSFER_ALL_ROLE_COMPARE)

print("\nMain all-transformer transfer dataset-role comparison:")
display(
    TRANSFER_ALL_DATASET_ROLE_COMPARE.sort_values(
        ["window_role", "dataset"]
    )
)

print(f"\nSaved main all-transformer transfer history: {history_path}")
print(f"Saved main all-transformer transfer predictions: {predictions_path}")
print(f"Saved main all-transformer transfer decision: {decision_path}")
print(f"Saved main all-transformer transfer dataset comparison: {dataset_compare_path}")
print(f"Saved main all-transformer transfer checkpoint: {checkpoint_path}")
print("\nPASS: Main all-transformer transfer experiment completed.")

Main all-transformer transfer trainable parameter report:


,module,total_params,trainable_params,trainable_pct
0,encoder,1696,0,0.0
1,freq_proj,47680,47680,100.0
2,transformer_layers,270144,270144,100.0
3,head,2625,2625,100.0


transfer_main_all_transformer_freq_head_lr2e5 | epoch 01 | train MAE 8.5648 | val MAE 5.9779 | best 5.9779 @ epoch 1
transfer_main_all_transformer_freq_head_lr2e5 | epoch 02 | train MAE 8.6103 | val MAE 5.8920 | best 5.8920 @ epoch 2
transfer_main_all_transformer_freq_head_lr2e5 | epoch 03 | train MAE 8.5316 | val MAE 5.8722 | best 5.8722 @ epoch 3
transfer_main_all_transformer_freq_head_lr2e5 | epoch 04 | train MAE 8.6344 | val MAE 5.9297 | best 5.8722 @ epoch 3
transfer_main_all_transformer_freq_head_lr2e5 | epoch 05 | train MAE 8.5187 | val MAE 5.9355 | best 5.8722 @ epoch 3
transfer_main_all_transformer_freq_head_lr2e5 | epoch 06 | train MAE 8.4856 | val MAE 5.8933 | best 5.8722 @ epoch 3
transfer_main_all_transformer_freq_head_lr2e5 | epoch 07 | train MAE 8.4444 | val MAE 5.8905 | best 5.8722 @ epoch 3
transfer_main_all_transformer_freq_head_lr2e5 | epoch 08 | train MAE 8.5153 | val MAE 5.9118 | best 5.8722 @ epoch 3
Early stopping after epoch 8.

Main all-transformer transfer tra

,experiment,epoch,train_rows,train_loss,train_mae,grad_norm_mean,grad_norm_max,val_mae,val_median,val_p90,val_p95,val_signed_error_mean,improved,best_epoch_so_far,best_val_mae_so_far
0,transfer_main_all_transformer_freq_head_lr2e5,1,12503,6.053246,8.564845,15.820180,26.549070,5.977870,3.808510,14.179719,19.090441,1.296045,True,1,5.977870
1,transfer_main_all_transformer_freq_head_lr2e5,2,12503,6.082604,8.610312,17.965402,41.048710,5.891997,3.743874,14.096118,18.980366,0.766246,True,2,5.891997
2,transfer_main_all_transformer_freq_head_lr2e5,3,12503,6.017231,8.531647,17.108434,40.536922,5.872171,3.769150,14.189542,18.800768,0.898261,True,3,5.872171
3,transfer_main_all_transformer_freq_head_lr2e5,4,12503,6.104746,8.634445,18.535385,34.393360,5.929651,3.808739,14.035525,18.958639,1.195284,False,3,5.872171
4,transfer_main_all_transformer_freq_head_lr2e5,5,12503,6.011399,8.518730,17.017686,35.257397,5.935483,3.789757,13.997926,18.960627,1.071706,False,3,5.872171
5,transfer_main_all_transformer_freq_head_lr2e5,6,12503,5.967958,8.485608,17.086018,41.178967,5.893328,3.760849,14.070260,18.852535,0.873480,False,3,5.872171
6,transfer_main_all_transformer_freq_head_lr2e5,7,12503,5.939120,8.444418,17.651636,34.753269,5.890488,3.771332,13.982528,18.951033,1.107588,False,3,5.872171
7,transfer_main_all_transformer_freq_head_lr2e5,8,12503,5.987883,8.515271,18.762530,37.134003,5.911839,3.812080,14.061376,18.895136,1.134100,False,3,5.872171



Main all-transformer transfer decision row:


...


Main all-transformer transfer role comparison:


,window_role,rows_candidate,target_hr_mean_candidate,pred_hr_mean_candidate,mae_mean_candidate,mae_median_candidate,mae_p90_candidate,mae_p95_candidate,signed_error_mean_candidate,severe_error_15bpm_rate_candidate,...,severe_error_15bpm_rate_frozen,high_hr_ge_100_rows_frozen,abs_signed_bias_frozen,delta_mae_mean,delta_mae_median,delta_mae_p90,delta_mae_p95,delta_signed_error_mean,delta_abs_signed_bias,delta_severe_error_15bpm_rate
0,finetune_test,2828,79.782204,80.769302,5.6331,3.4531,13.0547,18.8363,0.9871,0.0827,...,0.0831,244,1.4174,-0.1366,-0.1743,-0.6437,-0.2097,-0.4302,-0.4302,-0.0004
1,finetune_train,12503,80.237000,81.296402,5.6244,3.6249,13.2659,18.2272,1.0593,0.0781,...,0.0846,1129,1.5658,-0.1992,-0.1188,-0.4533,-0.7154,-0.5065,-0.5065,-0.0065
2,finetune_val,2839,80.161697,81.059998,5.8722,3.7691,14.1895,18.8008,0.8983,0.0881,...,0.0891,168,1.4724,-0.1553,-0.0842,0.1194,-0.4312,-0.5742,-0.5742,-0.0011



Main all-transformer transfer dataset-role comparison:


,dataset,window_role,rows_candidate,target_hr_mean_candidate,pred_hr_mean_candidate,mae_mean_candidate,mae_median_candidate,mae_p90_candidate,mae_p95_candidate,signed_error_mean_candidate,...,severe_error_15bpm_rate_frozen,high_hr_ge_100_rows_frozen,abs_signed_bias_frozen,delta_mae_mean,delta_mae_median,delta_mae_p90,delta_mae_p95,delta_signed_error_mean,delta_abs_signed_bias,delta_severe_error_15bpm_rate
0,mcd_rppg,finetune_test,2094,79.853302,80.343697,4.7512,3.0184,10.7978,15.6601,0.4904,...,0.0525,192,0.9242,-0.1336,-0.1317,-0.1722,0.1112,-0.4337,-0.4337,0.0005
3,ubfc_phys,finetune_test,658,77.420601,80.168999,8.6899,5.6326,19.7648,25.0577,2.7484,...,0.1900,7,3.1980,-0.1685,-0.0409,-0.5214,-0.6157,-0.4495,-0.4495,-0.0030
6,ubfc_rppg,finetune_test,76,98.269302,97.692398,3.4667,3.0355,6.0567,7.0317,-0.5769,...,0.0000,45,0.4102,0.0582,-0.1098,-0.0268,0.1922,-0.1667,0.1667,0.0000
1,mcd_rppg,finetune_train,7446,79.064796,79.804298,4.4481,3.0513,9.8186,13.4743,0.7395,...,0.0431,569,1.2645,-0.2396,-0.1126,-0.5537,-0.5226,-0.5251,-0.5251,-0.0064
4,ubfc_phys,finetune_train,4595,80.305496,82.127602,7.6344,5.3353,18.2191,22.8601,1.8221,...,0.1567,367,2.3192,-0.1475,-0.0895,-0.3737,-0.6241,-0.4971,-0.4971,-0.0067
7,ubfc_rppg,finetune_train,462,98.449203,97.077202,4.5932,3.2445,9.9444,13.0848,-1.3720,...,0.0368,193,1.0723,-0.0616,0.0998,0.0255,-0.0838,-0.2997,0.2997,-0.0043
2,mcd_rppg,finetune_val,1743,79.108597,79.656403,4.9061,3.2740,11.1710,15.2177,0.5478,...,0.0545,79,1.2656,-0.2264,-0.1007,-0.3658,-0.2607,-0.7178,-0.7178,-0.0029
5,ubfc_phys,finetune_val,986,80.002403,81.924103,7.7050,4.9464,18.1132,23.8245,1.9217,...,0.1531,19,2.2826,-0.0580,0.1003,-0.1593,-0.4411,-0.3609,-0.3609,0.0020
8,ubfc_rppg,finetune_val,110,98.277199,95.555298,4.7509,2.6177,13.1312,16.3050,-2.7219,...,0.0636,70,2.5126,0.0976,0.1739,0.7538,0.9033,-0.2094,0.2094,0.0000



Saved main all-transformer transfer history: /kaggle/working/crvse_nb13_app_relevant_without_ecg_fitness/nb13_transfer_main_all_transformer_history.csv
Saved main all-transformer transfer predictions: /kaggle/working/crvse_nb13_app_relevant_without_ecg_fitness/nb13_transfer_main_all_transformer_predictions.csv
Saved main all-transformer transfer decision: /kaggle/working/crvse_nb13_app_relevant_without_ecg_fitness/nb13_transfer_main_all_transformer_decision.csv
Saved main all-transformer transfer dataset comparison: /kaggle/working/crvse_nb13_app_relevant_without_ecg_fitness/nb13_transfer_main_all_transformer_dataset_role_compare.csv
Saved main all-transformer transfer checkpoint: /kaggle/working/crvse_nb13_app_relevant_without_ecg_fitness/nb13_transfer_main_all_transformer_best_state.pt

PASS: Main all-transformer transfer experiment completed.


### NB13 scratch-training branch with target-mean regression-head initialization

This notebook now tests a true scratch-training branch. The model architecture is kept app-aligned, but the weights are not initialized from the frozen checkpoint. Only the final regression head bias is initialized near the training-set mean heart rate so that early evaluation does not collapse to the lower 40 BPM output clamp.

This remains a scratch experiment because no pretrained checkpoint weights are transferred into the encoder, frequency projection, transformer layers, or head weights. The mean-bias initialization only gives the regression output a sensible starting scale for BPM prediction.

The experiment still belongs to the app-relevant still/seated webcam rPPG scope. It does not test exercise, high-motion, or ECG-Fitness robustness. The candidate must be judged against the same NB13 policy: validation gain, held-out test gain, p90 behavior, severe-error rate, signed bias, and per-dataset regressions.

In [12]:
SCRATCH_MAIN_EXPERIMENT = "scratch_main_physformer_mean_bias_lr3e4"

SCRATCH_MAIN_CONFIG = {
    "seed": 42,
    "train_batch_size": 256,
    "eval_batch_size": 512,
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "huber_beta": 6.0,
    "grad_clip": 5.0,
    "max_epochs": 35,
    "early_stopping_patience": 8,
    "head_bias_init": "train_target_mean",
    "head_weight_init_std": 1e-3,
}


def as_device_object(device_value) -> torch.device:
    """Return a torch.device object from the notebook DEVICE value."""
    if isinstance(device_value, torch.device):
        return device_value
    return torch.device(str(device_value))


DEVICE_OBJ = as_device_object(DEVICE)
PIN_MEMORY = DEVICE_OBJ.type == "cuda"


def seed_everything(seed: int) -> None:
    """Seed Python, NumPy, and PyTorch for a reproducible notebook branch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_scratch_loader(role: str, batch_size: int, shuffle: bool) -> DataLoader:
    """Create a DataLoader for one NB13 role."""
    generator = None

    if shuffle:
        generator = torch.Generator()
        generator.manual_seed(int(SCRATCH_MAIN_CONFIG["seed"]))

    return DataLoader(
        NB13_DATASETS[role],
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=PIN_MEMORY,
        generator=generator,
    )


def get_train_target_mean() -> float:
    """Read the training target mean from the verified NB13 tensor cache."""
    if "NB13_TENSOR_CACHE" in globals():
        return float(np.mean(NB13_TENSOR_CACHE["finetune_train"]["y"]))

    dataset = NB13_DATASETS["finetune_train"]

    if hasattr(dataset, "y"):
        y_values = dataset.y
        if torch.is_tensor(y_values):
            y_values = y_values.detach().cpu().numpy()
        return float(np.mean(y_values))

    if hasattr(dataset, "metadata") and "target_hr_mean" in dataset.metadata.columns:
        return float(pd.to_numeric(dataset.metadata["target_hr_mean"]).mean())

    raise ValueError("Could not infer training target mean for scratch head initialization.")


def initialize_final_regression_head(
    model: nn.Module,
    bias_value: float,
    weight_std: float,
) -> None:
    """Initialize the final regression layer near the training-set HR mean."""
    linear_layers = [module for module in model.head.modules() if isinstance(module, nn.Linear)]

    if not linear_layers:
        raise ValueError("Model head does not contain a Linear layer.")

    final_linear = linear_layers[-1]

    with torch.no_grad():
        final_linear.weight.normal_(mean=0.0, std=float(weight_std))
        final_linear.bias.fill_(float(bias_value))


def summarize_major_module_trainability(model: nn.Module) -> pd.DataFrame:
    """Summarize trainable parameters by major PhysFormer module."""
    rows = []

    for module_name in ["encoder", "freq_proj", "transformer_layers", "head"]:
        module = getattr(model, module_name, None)

        if module is None:
            continue

        total_params = sum(param.numel() for param in module.parameters())
        trainable_params = sum(
            param.numel() for param in module.parameters() if param.requires_grad
        )

        rows.append(
            {
                "module": module_name,
                "total_params": int(total_params),
                "trainable_params": int(trainable_params),
                "trainable_pct": round(100.0 * trainable_params / total_params, 4),
            }
        )

    return pd.DataFrame(rows)


def summarize_prediction_floor_behavior(predictions: pd.DataFrame) -> pd.DataFrame:
    """Summarize whether a candidate is collapsing to output clamps."""
    summary = (
        predictions
        .groupby("window_role", dropna=False)
        .agg(
            rows=("candidate_model_hr", "size"),
            pred_min=("candidate_model_hr", "min"),
            pred_mean=("candidate_model_hr", "mean"),
            pred_max=("candidate_model_hr", "max"),
            floor_40bpm_rate=("candidate_model_hr", lambda values: float((values <= 40.0001).mean())),
            ceiling_180bpm_rate=("candidate_model_hr", lambda values: float((values >= 179.9999).mean())),
        )
        .reset_index()
    )

    numeric_cols = summary.select_dtypes(include=[np.number]).columns
    summary[numeric_cols] = summary[numeric_cols].round(6)

    return summary


required_helpers = [
    "build_scratch_model_like_checkpoint",
    "train_candidate_model",
    "evaluate_model_loader",
    "compare_candidate_to_frozen_baseline",
    "make_nb13_candidate_decision",
]

missing_helpers = [name for name in required_helpers if name not in globals()]

if missing_helpers:
    raise NameError(
        "Run the previous NB13 model/helper cells first. Missing helpers: "
        + ", ".join(missing_helpers)
    )

seed_everything(int(SCRATCH_MAIN_CONFIG["seed"]))

if DEVICE_OBJ.type == "cuda":
    torch.cuda.empty_cache()

scratch_train_target_mean = get_train_target_mean()

SCRATCH_MAIN_TRAIN_LOADER = make_scratch_loader(
    role="finetune_train",
    batch_size=int(SCRATCH_MAIN_CONFIG["train_batch_size"]),
    shuffle=True,
)

SCRATCH_MAIN_EVAL_LOADERS_BY_ROLE = {
    role: make_scratch_loader(
        role=role,
        batch_size=int(SCRATCH_MAIN_CONFIG["eval_batch_size"]),
        shuffle=False,
    )
    for role in ["finetune_train", "finetune_val", "finetune_test"]
}

SCRATCH_MAIN_MODEL = build_scratch_model_like_checkpoint().to(DEVICE_OBJ)

for param in SCRATCH_MAIN_MODEL.parameters():
    param.requires_grad = True

initialize_final_regression_head(
    model=SCRATCH_MAIN_MODEL,
    bias_value=scratch_train_target_mean,
    weight_std=float(SCRATCH_MAIN_CONFIG["head_weight_init_std"]),
)

SCRATCH_MAIN_PARAM_REPORT = summarize_major_module_trainability(SCRATCH_MAIN_MODEL)

print("Scratch main trainable parameter report:")
display(SCRATCH_MAIN_PARAM_REPORT)

print(
    "\nScratch main initialization:",
    json.dumps(
        {
            "train_target_mean_bias_bpm": round(float(scratch_train_target_mean), 6),
            "head_weight_init_std": SCRATCH_MAIN_CONFIG["head_weight_init_std"],
            "lr": SCRATCH_MAIN_CONFIG["lr"],
            "max_epochs": SCRATCH_MAIN_CONFIG["max_epochs"],
            "early_stopping_patience": SCRATCH_MAIN_CONFIG["early_stopping_patience"],
        },
        indent=2,
    ),
)

SCRATCH_MAIN_HISTORY, SCRATCH_MAIN_BEST_STATE, SCRATCH_MAIN_BEST_VAL_MAE, SCRATCH_MAIN_BEST_EPOCH = (
    train_candidate_model(
        model=SCRATCH_MAIN_MODEL,
        experiment_name=SCRATCH_MAIN_EXPERIMENT,
        train_loader=SCRATCH_MAIN_TRAIN_LOADER,
        val_loader=SCRATCH_MAIN_EVAL_LOADERS_BY_ROLE["finetune_val"],
        config=SCRATCH_MAIN_CONFIG,
    )
)

SCRATCH_MAIN_MODEL.load_state_dict(SCRATCH_MAIN_BEST_STATE)
SCRATCH_MAIN_MODEL.to(DEVICE_OBJ)
SCRATCH_MAIN_MODEL.eval()

SCRATCH_MAIN_PREDICTIONS = pd.concat(
    [
        evaluate_model_loader(
            model=SCRATCH_MAIN_MODEL,
            role=role,
            loader=loader,
            experiment_name=SCRATCH_MAIN_EXPERIMENT,
        )
        for role, loader in SCRATCH_MAIN_EVAL_LOADERS_BY_ROLE.items()
    ],
    ignore_index=True,
)

(
    SCRATCH_MAIN_COMPARE,
    SCRATCH_MAIN_ROLE_COMPARE,
    SCRATCH_MAIN_DATASET_ROLE_COMPARE,
) = compare_candidate_to_frozen_baseline(SCRATCH_MAIN_PREDICTIONS)

SCRATCH_MAIN_DECISION = make_nb13_candidate_decision(
    experiment_name=SCRATCH_MAIN_EXPERIMENT,
    role_compare=SCRATCH_MAIN_ROLE_COMPARE,
    dataset_role_compare=SCRATCH_MAIN_DATASET_ROLE_COMPARE,
)

SCRATCH_MAIN_FLOOR_REPORT = summarize_prediction_floor_behavior(SCRATCH_MAIN_PREDICTIONS)

for table in [
    SCRATCH_MAIN_ROLE_COMPARE,
    SCRATCH_MAIN_DATASET_ROLE_COMPARE,
]:
    numeric_cols = table.select_dtypes(include=[np.number]).columns
    table[numeric_cols] = table[numeric_cols].round(4)

OUTPUT_DIR = Path(NB13_WORKING_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

history_path = OUTPUT_DIR / "nb13_scratch_main_history.csv"
predictions_path = OUTPUT_DIR / "nb13_scratch_main_predictions.csv"
decision_path = OUTPUT_DIR / "nb13_scratch_main_decision.csv"
dataset_compare_path = OUTPUT_DIR / "nb13_scratch_main_dataset_role_compare.csv"
floor_report_path = OUTPUT_DIR / "nb13_scratch_main_floor_report.csv"
checkpoint_path = OUTPUT_DIR / "nb13_scratch_main_best_state.pt"

SCRATCH_MAIN_HISTORY.to_csv(history_path, index=False)
SCRATCH_MAIN_PREDICTIONS.to_csv(predictions_path, index=False)
SCRATCH_MAIN_DECISION.to_csv(decision_path, index=False)
SCRATCH_MAIN_DATASET_ROLE_COMPARE.to_csv(dataset_compare_path, index=False)
SCRATCH_MAIN_FLOOR_REPORT.to_csv(floor_report_path, index=False)

torch.save(
    {
        "experiment": SCRATCH_MAIN_EXPERIMENT,
        "model_state": SCRATCH_MAIN_BEST_STATE,
        "best_val_mae": float(SCRATCH_MAIN_BEST_VAL_MAE),
        "best_epoch": int(SCRATCH_MAIN_BEST_EPOCH),
        "training_config": SCRATCH_MAIN_CONFIG,
        "train_target_mean_bias_bpm": float(scratch_train_target_mean),
        "decision": SCRATCH_MAIN_DECISION.to_dict(orient="records")[0],
        "architecture_note": (
            "NB13 scratch branch with app-aligned PhysFormer architecture. "
            "No pretrained checkpoint weights are used; only the final regression "
            "bias is initialized to the training target mean."
        ),
    },
    checkpoint_path,
)

print("\nScratch main training history:")
display(SCRATCH_MAIN_HISTORY)

print("\nScratch main prediction floor report:")
display(SCRATCH_MAIN_FLOOR_REPORT)

print("\nScratch main decision row:")
display(SCRATCH_MAIN_DECISION)

print("\nScratch main role comparison:")
display(SCRATCH_MAIN_ROLE_COMPARE)

print("\nScratch main dataset-role comparison:")
display(
    SCRATCH_MAIN_DATASET_ROLE_COMPARE.sort_values(
        ["window_role", "dataset"]
    )
)

print(f"\nSaved scratch main history: {history_path}")
print(f"Saved scratch main predictions: {predictions_path}")
print(f"Saved scratch main decision: {decision_path}")
print(f"Saved scratch main dataset comparison: {dataset_compare_path}")
print(f"Saved scratch main floor report: {floor_report_path}")
print(f"Saved scratch main checkpoint: {checkpoint_path}")
print("\nPASS: Scratch main experiment completed.")

Scratch main trainable parameter report:


,module,total_params,trainable_params,trainable_pct
0,encoder,1696,1696,100.0
1,freq_proj,47680,47680,100.0
2,transformer_layers,270144,270144,100.0
3,head,2625,2625,100.0



Scratch main initialization: {
  "train_target_mean_bias_bpm": 80.237045,
  "head_weight_init_std": 0.001,
  "lr": 0.0003,
  "max_epochs": 35,
  "early_stopping_patience": 8
}
scratch_main_physformer_mean_bias_lr3e4 | epoch 01 | train MAE 10.3863 | val MAE 9.4092 | best 9.4092 @ epoch 1
scratch_main_physformer_mean_bias_lr3e4 | epoch 02 | train MAE 9.0624 | val MAE 8.8454 | best 8.8454 @ epoch 2
scratch_main_physformer_mean_bias_lr3e4 | epoch 03 | train MAE 8.3647 | val MAE 8.2489 | best 8.2489 @ epoch 3
scratch_main_physformer_mean_bias_lr3e4 | epoch 04 | train MAE 7.7884 | val MAE 8.4478 | best 8.2489 @ epoch 3
scratch_main_physformer_mean_bias_lr3e4 | epoch 05 | train MAE 7.4666 | val MAE 7.7601 | best 7.7601 @ epoch 5
scratch_main_physformer_mean_bias_lr3e4 | epoch 06 | train MAE 7.2811 | val MAE 7.7579 | best 7.7579 @ epoch 6
scratch_main_physformer_mean_bias_lr3e4 | epoch 07 | train MAE 7.0970 | val MAE 7.4276 | best 7.4276 @ epoch 7
scratch_main_physformer_mean_bias_lr3e4 | epo

,experiment,epoch,train_rows,train_loss,train_mae,grad_norm_mean,grad_norm_max,val_mae,val_median,val_p90,val_p95,val_signed_error_mean,improved,best_epoch_so_far,best_val_mae_so_far
0,scratch_main_physformer_mean_bias_lr3e4,1,12503,7.784871,10.386345,5.645662,16.094536,9.409205,7.947067,19.594082,24.149481,-1.299288,True,1,9.409205
1,scratch_main_physformer_mean_bias_lr3e4,2,12503,6.519761,9.062376,10.003096,22.698557,8.845352,6.835091,18.781750,23.214256,-0.727165,True,2,8.845352
2,scratch_main_physformer_mean_bias_lr3e4,3,12503,5.905071,8.364744,8.770503,25.545053,8.248921,6.200630,18.244398,23.051834,-2.755337,True,3,8.248921
3,scratch_main_physformer_mean_bias_lr3e4,4,12503,5.368436,7.788411,12.357534,33.091663,8.447800,6.377602,18.400263,23.704716,-4.581531,False,3,8.248921
4,scratch_main_physformer_mean_bias_lr3e4,5,12503,5.084036,7.466626,13.696677,35.908325,7.760132,6.035713,17.193071,21.775352,-2.958822,True,5,7.760132
5,scratch_main_physformer_mean_bias_lr3e4,6,12503,4.911158,7.281129,17.255624,36.612301,7.757948,5.790787,17.384510,22.452408,-3.438695,True,6,7.757948
6,scratch_main_physformer_mean_bias_lr3e4,7,12503,4.747591,7.096957,15.722061,35.748253,7.427588,5.421928,16.762608,22.215511,-2.746399,True,7,7.427588
7,scratch_main_physformer_mean_bias_lr3e4,8,12503,4.645861,6.975461,16.717825,37.073231,7.255719,5.310066,16.535900,21.788975,-1.601888,True,8,7.255719
8,scratch_main_physformer_mean_bias_lr3e4,9,12503,4.537094,6.858564,18.250666,43.318092,7.474891,5.478241,16.661343,21.715420,-2.849258,False,8,7.255719
9,scratch_main_physformer_mean_bias_lr3e4,10,12503,4.482035,6.785810,21.380453,49.622032,7.012410,4.990181,16.057474,21.528896,-1.045767,True,10,7.012410



Scratch main prediction floor report:


...


Scratch main role comparison:


,window_role,rows_candidate,target_hr_mean_candidate,pred_hr_mean_candidate,mae_mean_candidate,mae_median_candidate,mae_p90_candidate,mae_p95_candidate,signed_error_mean_candidate,severe_error_15bpm_rate_candidate,...,severe_error_15bpm_rate_frozen,high_hr_ge_100_rows_frozen,abs_signed_bias_frozen,delta_mae_mean,delta_mae_median,delta_mae_p90,delta_mae_p95,delta_signed_error_mean,delta_abs_signed_bias,delta_severe_error_15bpm_rate
0,finetune_test,2828,79.782204,77.971199,6.0196,3.8689,14.3723,18.8798,-1.8110,0.0870,...,0.0831,244,1.4174,0.2500,0.2415,0.6739,-0.1663,-3.2283,0.3936,0.0039
1,finetune_train,12503,80.237000,78.777397,5.8280,3.7145,13.9146,18.6996,-1.4596,0.0848,...,0.0846,1129,1.5658,0.0044,-0.0292,0.1954,-0.2431,-3.0254,-0.1062,0.0002
2,finetune_val,2839,80.161697,78.588097,6.6083,4.4082,15.5966,21.0845,-1.5736,0.1081,...,0.0891,168,1.4724,0.5808,0.5549,1.5265,1.8525,-3.0461,0.1012,0.0190



Scratch main dataset-role comparison:


,dataset,window_role,rows_candidate,target_hr_mean_candidate,pred_hr_mean_candidate,mae_mean_candidate,mae_median_candidate,mae_p90_candidate,mae_p95_candidate,signed_error_mean_candidate,...,severe_error_15bpm_rate_frozen,high_hr_ge_100_rows_frozen,abs_signed_bias_frozen,delta_mae_mean,delta_mae_median,delta_mae_p90,delta_mae_p95,delta_signed_error_mean,delta_abs_signed_bias,delta_severe_error_15bpm_rate
0,mcd_rppg,finetune_test,2094,79.853302,77.427299,5.5292,3.5755,12.6553,17.6697,-2.4260,...,0.0525,192,0.9242,0.6445,0.4253,1.6853,2.1207,-3.3502,1.5018,0.0143
3,ubfc_phys,finetune_test,658,77.420601,77.412903,7.8198,5.7114,17.4970,21.4587,-0.0077,...,0.1900,7,3.1980,-1.0386,0.0379,-2.7891,-4.2146,-3.2056,-3.1903,-0.0304
6,ubfc_rppg,finetune_test,76,98.269302,97.791702,3.9454,3.2629,7.1539,8.8499,-0.4776,...,0.0000,45,0.4102,0.5370,0.1175,1.0703,2.0104,-0.0674,0.0674,0.0132
1,mcd_rppg,finetune_train,7446,79.064796,77.245201,4.6327,2.9535,10.6775,14.5657,-1.8196,...,0.0431,569,1.2645,-0.0549,-0.2104,0.3053,0.5688,-3.0842,0.5551,0.0027
4,ubfc_phys,finetune_train,4595,80.305496,79.597702,7.8178,5.7493,17.9542,22.4012,-0.7077,...,0.1567,367,2.3192,0.0359,0.3244,-0.6387,-1.0831,-3.0269,-1.6114,-0.0061
7,ubfc_rppg,finetune_train,462,98.449203,95.314201,5.3013,3.7605,11.8054,17.0591,-3.1350,...,0.0368,193,1.0723,0.6465,0.6159,1.8864,3.8904,-2.0627,2.0627,0.0216
2,mcd_rppg,finetune_val,1743,79.108597,76.948601,5.8014,3.8882,13.0551,18.9888,-2.1600,...,0.0545,79,1.2656,0.6689,0.5135,1.5183,3.5103,-3.4256,0.8944,0.0252
5,ubfc_phys,finetune_val,986,80.002403,79.664597,8.0672,5.7666,18.5448,23.4416,-0.3378,...,0.1531,19,2.2826,0.3042,0.9205,0.2723,-0.8241,-2.6203,-1.9448,0.0030
8,ubfc_rppg,finetune_val,110,98.277199,94.917099,6.3175,4.2927,16.3242,19.7989,-3.3602,...,0.0636,70,2.5126,1.6642,1.8489,3.9468,4.3972,-0.8476,0.8476,0.0636



Saved scratch main history: /kaggle/working/crvse_nb13_app_relevant_without_ecg_fitness/nb13_scratch_main_history.csv
Saved scratch main predictions: /kaggle/working/crvse_nb13_app_relevant_without_ecg_fitness/nb13_scratch_main_predictions.csv
Saved scratch main decision: /kaggle/working/crvse_nb13_app_relevant_without_ecg_fitness/nb13_scratch_main_decision.csv
Saved scratch main dataset comparison: /kaggle/working/crvse_nb13_app_relevant_without_ecg_fitness/nb13_scratch_main_dataset_role_compare.csv
Saved scratch main floor report: /kaggle/working/crvse_nb13_app_relevant_without_ecg_fitness/nb13_scratch_main_floor_report.csv
Saved scratch main checkpoint: /kaggle/working/crvse_nb13_app_relevant_without_ecg_fitness/nb13_scratch_main_best_state.pt

PASS: Scratch main experiment completed.


### NB13 candidate synthesis before further training

This notebook now consolidates the NB13 transfer-learning and scratch-training candidates against the predefined app-relevant decision policy. The purpose is to avoid selecting a checkpoint from a single encouraging metric and instead compare each candidate by validation gain, held-out test gain, p90 error, severe-error rate, signed bias, and per-dataset regressions.

This synthesis keeps the scientific scope narrow. The candidates are evaluated only for still/seated webcam rPPG on MCD-rPPG, UBFC-rPPG, and UBFC-Phys. ECG-Fitness remains excluded from NB13 model selection, so these results do not support claims about exercise, high-motion, or exercise-related high-HR robustness.

In [13]:
NB13_SYNTHESIS_DIR = Path(NB13_WORKING_DIR)

NB13_POLICY_THRESHOLDS = {
    "minimum_val_gain_bpm": 0.20,
    "minimum_test_gain_bpm": 0.10,
    "maximum_dataset_test_regression_bpm": 0.30,
    "maximum_ubfc_phys_test_regression_bpm": 0.10,
    "maximum_p90_regression_bpm": 0.25,
    "maximum_severe_error_rate_increase": 0.01,
    "maximum_abs_signed_bias_increase_bpm": 0.25,
}

NB13_CANDIDATE_ARTIFACTS = [
    {
        "candidate_label": "transfer_smoke_last_block_freq_head",
        "branch_family": "transfer",
        "run_kind": "smoke",
        "decision_file": "nb13_transfer_smoke_decision.csv",
        "dataset_compare_file": "nb13_transfer_smoke_dataset_role_compare.csv",
    },
    {
        "candidate_label": "scratch_smoke_physformer",
        "branch_family": "scratch",
        "run_kind": "smoke",
        "decision_file": "nb13_scratch_smoke_decision.csv",
        "dataset_compare_file": "nb13_scratch_smoke_dataset_role_compare.csv",
    },
    {
        "candidate_label": "transfer_main_last_block_freq_head_lr5e5",
        "branch_family": "transfer",
        "run_kind": "main",
        "decision_file": "nb13_transfer_main_decision.csv",
        "dataset_compare_file": "nb13_transfer_main_dataset_role_compare.csv",
    },
    {
        "candidate_label": "transfer_main_all_transformer_freq_head_lr2e5",
        "branch_family": "transfer",
        "run_kind": "main",
        "decision_file": "nb13_transfer_main_all_transformer_decision.csv",
        "dataset_compare_file": "nb13_transfer_main_all_transformer_dataset_role_compare.csv",
    },
    {
        "candidate_label": "scratch_main_physformer_mean_bias_lr3e4",
        "branch_family": "scratch",
        "run_kind": "main",
        "decision_file": "nb13_scratch_main_decision.csv",
        "dataset_compare_file": "nb13_scratch_main_dataset_role_compare.csv",
    },
]


def clean_json_value(value):
    """Convert NumPy/Pandas values into JSON-safe Python values."""
    if pd.isna(value):
        return None

    if isinstance(value, np.generic):
        return value.item()

    return value


def compact_candidate_record(row: pd.Series) -> dict:
    """Build a compact JSON-safe candidate summary record."""
    keys = [
        "candidate",
        "branch_family",
        "run_kind",
        "val_delta_mae_mean",
        "test_delta_mae_mean",
        "val_delta_mae_p90",
        "test_delta_mae_p90",
        "test_delta_severe_error_rate",
        "test_delta_abs_signed_bias",
        "max_dataset_test_delta_mae",
        "ubfc_phys_test_delta_mae",
        "decision_status",
        "adoptable_by_nb13_policy",
        "scientific_label",
    ]

    return {key: clean_json_value(row.get(key)) for key in keys}


leaderboard_rows = []
missing_decision_files = []

for artifact in NB13_CANDIDATE_ARTIFACTS:
    decision_path = NB13_SYNTHESIS_DIR / artifact["decision_file"]

    if not decision_path.exists():
        missing_decision_files.append(str(decision_path))
        continue

    decision = pd.read_csv(decision_path)

    if len(decision) == 0:
        missing_decision_files.append(f"{decision_path} is empty")
        continue

    row = decision.iloc[0].to_dict()
    row["candidate"] = row.get("candidate", artifact["candidate_label"])
    row["candidate_label"] = artifact["candidate_label"]
    row["branch_family"] = artifact["branch_family"]
    row["run_kind"] = artifact["run_kind"]
    row["decision_path"] = str(decision_path)
    row["dataset_compare_path"] = str(NB13_SYNTHESIS_DIR / artifact["dataset_compare_file"])
    leaderboard_rows.append(row)

if not leaderboard_rows:
    raise FileNotFoundError("No NB13 candidate decision files were found.")

NB13_CANDIDATE_LEADERBOARD = pd.DataFrame(leaderboard_rows)

numeric_cols = [
    "val_delta_mae_mean",
    "test_delta_mae_mean",
    "val_delta_mae_p90",
    "test_delta_mae_p90",
    "test_delta_severe_error_rate",
    "test_delta_abs_signed_bias",
    "max_dataset_test_delta_mae",
    "ubfc_phys_test_delta_mae",
]

for col in numeric_cols:
    if col in NB13_CANDIDATE_LEADERBOARD.columns:
        NB13_CANDIDATE_LEADERBOARD[col] = pd.to_numeric(
            NB13_CANDIDATE_LEADERBOARD[col],
            errors="coerce",
        )

NB13_CANDIDATE_LEADERBOARD["passes_val_gain"] = (
    NB13_CANDIDATE_LEADERBOARD["val_delta_mae_mean"]
    <= -NB13_POLICY_THRESHOLDS["minimum_val_gain_bpm"]
)
NB13_CANDIDATE_LEADERBOARD["passes_test_gain"] = (
    NB13_CANDIDATE_LEADERBOARD["test_delta_mae_mean"]
    <= -NB13_POLICY_THRESHOLDS["minimum_test_gain_bpm"]
)
NB13_CANDIDATE_LEADERBOARD["passes_dataset_guardrail"] = (
    NB13_CANDIDATE_LEADERBOARD["max_dataset_test_delta_mae"]
    <= NB13_POLICY_THRESHOLDS["maximum_dataset_test_regression_bpm"]
)
NB13_CANDIDATE_LEADERBOARD["passes_ubfc_phys_guardrail"] = (
    NB13_CANDIDATE_LEADERBOARD["ubfc_phys_test_delta_mae"]
    <= NB13_POLICY_THRESHOLDS["maximum_ubfc_phys_test_regression_bpm"]
)
NB13_CANDIDATE_LEADERBOARD["passes_p90_guardrail"] = (
    NB13_CANDIDATE_LEADERBOARD["test_delta_mae_p90"]
    <= NB13_POLICY_THRESHOLDS["maximum_p90_regression_bpm"]
)
NB13_CANDIDATE_LEADERBOARD["passes_severe_guardrail"] = (
    NB13_CANDIDATE_LEADERBOARD["test_delta_severe_error_rate"]
    <= NB13_POLICY_THRESHOLDS["maximum_severe_error_rate_increase"]
)
NB13_CANDIDATE_LEADERBOARD["passes_bias_guardrail"] = (
    NB13_CANDIDATE_LEADERBOARD["test_delta_abs_signed_bias"]
    <= NB13_POLICY_THRESHOLDS["maximum_abs_signed_bias_increase_bpm"]
)

NB13_CANDIDATE_LEADERBOARD["adoptable_by_nb13_policy"] = (
    NB13_CANDIDATE_LEADERBOARD["passes_val_gain"]
    & NB13_CANDIDATE_LEADERBOARD["passes_test_gain"]
    & NB13_CANDIDATE_LEADERBOARD["passes_dataset_guardrail"]
    & NB13_CANDIDATE_LEADERBOARD["passes_ubfc_phys_guardrail"]
    & NB13_CANDIDATE_LEADERBOARD["passes_p90_guardrail"]
    & NB13_CANDIDATE_LEADERBOARD["passes_severe_guardrail"]
    & NB13_CANDIDATE_LEADERBOARD["passes_bias_guardrail"]
)

NB13_CANDIDATE_LEADERBOARD["scientific_label"] = np.select(
    [
        NB13_CANDIDATE_LEADERBOARD["run_kind"].eq("smoke"),
        NB13_CANDIDATE_LEADERBOARD["adoptable_by_nb13_policy"],
        NB13_CANDIDATE_LEADERBOARD["decision_status"].eq("reject"),
    ],
    [
        "smoke_test_only",
        "adoption_candidate_by_policy",
        "rejected_candidate",
    ],
    default="analysis_only_weak_candidate",
)

NB13_CANDIDATE_LEADERBOARD["run_order"] = NB13_CANDIDATE_LEADERBOARD["run_kind"].map(
    {"main": 0, "smoke": 1}
).fillna(2)

NB13_CANDIDATE_LEADERBOARD = (
    NB13_CANDIDATE_LEADERBOARD
    .sort_values(["run_order", "val_delta_mae_mean", "test_delta_mae_mean"])
    .reset_index(drop=True)
)

dataset_delta_rows = []

for artifact in NB13_CANDIDATE_ARTIFACTS:
    dataset_path = NB13_SYNTHESIS_DIR / artifact["dataset_compare_file"]

    if not dataset_path.exists():
        continue

    dataset_compare = pd.read_csv(dataset_path)
    dataset_compare = dataset_compare[
        dataset_compare["window_role"].astype(str).eq("finetune_test")
    ].copy()

    dataset_compare["candidate"] = artifact["candidate_label"]
    dataset_compare["branch_family"] = artifact["branch_family"]
    dataset_compare["run_kind"] = artifact["run_kind"]

    keep_cols = [
        "candidate",
        "branch_family",
        "run_kind",
        "dataset",
        "rows_candidate",
        "mae_mean_candidate",
        "mae_mean_frozen",
        "delta_mae_mean",
        "delta_mae_p90",
        "delta_mae_p95",
        "delta_abs_signed_bias",
        "delta_severe_error_15bpm_rate",
    ]

    dataset_delta_rows.append(
        dataset_compare[[col for col in keep_cols if col in dataset_compare.columns]]
    )

NB13_TEST_DATASET_DELTA_TABLE = (
    pd.concat(dataset_delta_rows, ignore_index=True)
    if dataset_delta_rows
    else pd.DataFrame()
)

main_candidates = NB13_CANDIDATE_LEADERBOARD[
    NB13_CANDIDATE_LEADERBOARD["run_kind"].eq("main")
].copy()

adoptable_main_candidates = main_candidates[
    main_candidates["adoptable_by_nb13_policy"]
].copy()

best_main_by_validation = (
    main_candidates.sort_values("val_delta_mae_mean").iloc[0]
    if len(main_candidates)
    else pd.Series(dtype=object)
)
best_main_by_test = (
    main_candidates.sort_values("test_delta_mae_mean").iloc[0]
    if len(main_candidates)
    else pd.Series(dtype=object)
)

recommended_checkpoint_action = (
    "consider_best_policy_passing_candidate"
    if len(adoptable_main_candidates)
    else "keep_frozen_sourcefps_checkpoint_for_app"
)

NB13_MODEL_SELECTION_SUMMARY = {
    "scope": {
        "included_datasets": ["mcd_rppg", "ubfc_rppg", "ubfc_phys"],
        "excluded_dataset": "ecg_fitness",
        "app_claim": "still/seated webcam rPPG only",
        "not_supported_by_nb13": "exercise, high-motion, or ECG-Fitness/high-HR robustness",
    },
    "policy_thresholds": NB13_POLICY_THRESHOLDS,
    "main_candidate_count": int(len(main_candidates)),
    "adoptable_main_candidate_count": int(len(adoptable_main_candidates)),
    "best_main_by_validation": compact_candidate_record(best_main_by_validation),
    "best_main_by_test": compact_candidate_record(best_main_by_test),
    "adoptable_main_candidates": [
        compact_candidate_record(row)
        for _, row in adoptable_main_candidates.iterrows()
    ],
    "recommended_checkpoint_action": recommended_checkpoint_action,
    "interpretation": (
        "No main NB13 candidate currently passes the predefined adoption policy. "
        "Transfer learning gives modest app-relevant gains but does not clear the "
        "validation-gain threshold. Scratch training learns a real model after "
        "mean-bias initialization, but remains worse than the frozen baseline overall."
    ),
    "missing_decision_files": missing_decision_files,
}

leaderboard_path = NB13_SYNTHESIS_DIR / "nb13_candidate_leaderboard.csv"
dataset_delta_path = NB13_SYNTHESIS_DIR / "nb13_test_dataset_delta_table.csv"
selection_summary_path = NB13_SYNTHESIS_DIR / "nb13_model_selection_summary.json"

NB13_CANDIDATE_LEADERBOARD.drop(columns=["run_order"]).to_csv(
    leaderboard_path,
    index=False,
)
NB13_TEST_DATASET_DELTA_TABLE.to_csv(dataset_delta_path, index=False)
selection_summary_path.write_text(
    json.dumps(NB13_MODEL_SELECTION_SUMMARY, indent=2),
    encoding="utf-8",
)

display_cols = [
    "candidate",
    "branch_family",
    "run_kind",
    "val_delta_mae_mean",
    "test_delta_mae_mean",
    "val_delta_mae_p90",
    "test_delta_mae_p90",
    "test_delta_severe_error_rate",
    "test_delta_abs_signed_bias",
    "max_dataset_test_delta_mae",
    "ubfc_phys_test_delta_mae",
    "decision_status",
    "scientific_label",
    "adoptable_by_nb13_policy",
]

rounded_leaderboard = NB13_CANDIDATE_LEADERBOARD[display_cols].copy()
num_cols = rounded_leaderboard.select_dtypes(include=[np.number]).columns
rounded_leaderboard[num_cols] = rounded_leaderboard[num_cols].round(4)

print("NB13 candidate leaderboard:")
display(rounded_leaderboard)

print("\nNB13 held-out test dataset delta table:")
display(
    NB13_TEST_DATASET_DELTA_TABLE.sort_values(
        ["run_kind", "candidate", "dataset"]
    )
)

print("\nNB13 model selection summary:")
print(json.dumps(NB13_MODEL_SELECTION_SUMMARY, indent=2))

print(f"\nSaved NB13 candidate leaderboard: {leaderboard_path}")
print(f"Saved NB13 test dataset delta table: {dataset_delta_path}")
print(f"Saved NB13 model selection summary: {selection_summary_path}")

if len(adoptable_main_candidates):
    print("\nATTENTION: At least one main candidate passes the NB13 policy.")
else:
    print("\nPASS: No main NB13 candidate passes the adoption policy; frozen checkpoint remains the app checkpoint.")

NB13 candidate leaderboard:


,candidate,branch_family,run_kind,val_delta_mae_mean,test_delta_mae_mean,val_delta_mae_p90,test_delta_mae_p90,test_delta_severe_error_rate,test_delta_abs_signed_bias,max_dataset_test_delta_mae,ubfc_phys_test_delta_mae,decision_status,scientific_label,adoptable_by_nb13_policy
0,transfer_main_all_transformer_freq_head_lr2e5,transfer,main,-0.1553,-0.1366,0.1194,-0.6437,-0.0004,-0.4302,0.0582,-0.1685,weak_or_incomplete_candidate,analysis_only_weak_candidate,False
1,transfer_main_last_block_freq_head_lr5e5,transfer,main,-0.1237,-0.1554,0.0002,-0.7677,-0.0014,-0.4448,0.0808,-0.1661,weak_or_incomplete_candidate,analysis_only_weak_candidate,False
2,scratch_main_physformer_mean_bias_lr3e4,scratch,main,0.5808,0.2500,1.5265,0.6739,0.0039,0.3936,0.6445,-1.0386,reject,rejected_candidate,False
3,transfer_smoke_last_block_freq_head,transfer,smoke,-0.0917,-0.1230,0.0950,-0.6578,-0.0018,-0.5321,0.1326,-0.1582,weak_or_incomplete_candidate,smoke_test_only,False
4,scratch_smoke_physformer,scratch,smoke,34.1342,34.0126,42.4071,45.1928,0.9006,38.3648,54.8608,28.5621,reject,smoke_test_only,False



NB13 held-out test dataset delta table:


,candidate,branch_family,run_kind,dataset,rows_candidate,mae_mean_candidate,mae_mean_frozen,delta_mae_mean,delta_mae_p90,delta_mae_p95,delta_abs_signed_bias,delta_severe_error_15bpm_rate
12,scratch_main_physformer_mean_bias_lr3e4,scratch,main,mcd_rppg,2094,5.5292,4.8847,0.6445,1.6853,2.1207,1.5018,0.0143
13,scratch_main_physformer_mean_bias_lr3e4,scratch,main,ubfc_phys,658,7.8198,8.8584,-1.0386,-2.7891,-4.2146,-3.1903,-0.0304
14,scratch_main_physformer_mean_bias_lr3e4,scratch,main,ubfc_rppg,76,3.9454,3.4085,0.5370,1.0703,2.0104,0.0674,0.0132
9,transfer_main_all_transformer_freq_head_lr2e5,transfer,main,mcd_rppg,2094,4.7512,4.8847,-0.1336,-0.1722,0.1112,-0.4337,0.0005
10,transfer_main_all_transformer_freq_head_lr2e5,transfer,main,ubfc_phys,658,8.6899,8.8584,-0.1685,-0.5214,-0.6157,-0.4495,-0.0030
11,transfer_main_all_transformer_freq_head_lr2e5,transfer,main,ubfc_rppg,76,3.4667,3.4085,0.0582,-0.0268,0.1922,0.1667,0.0000
6,transfer_main_last_block_freq_head_lr5e5,transfer,main,mcd_rppg,2094,4.7241,4.8847,-0.1606,-0.4430,-0.3635,-0.4258,-0.0004
7,transfer_main_last_block_freq_head_lr5e5,transfer,main,ubfc_phys,658,8.6923,8.8584,-0.1661,-0.6801,-0.7284,-0.5658,-0.0046
8,transfer_main_last_block_freq_head_lr5e5,transfer,main,ubfc_rppg,76,3.4893,3.4085,0.0808,-0.0013,0.3062,-0.0795,0.0000
3,scratch_smoke_physformer,scratch,smoke,mcd_rppg,2094,39.8533,4.8847,34.9686,48.3313,48.1243,38.9291,0.9317



NB13 model selection summary:
{
  "scope": {
    "included_datasets": [
      "mcd_rppg",
      "ubfc_rppg",
      "ubfc_phys"
    ],
    "excluded_dataset": "ecg_fitness",
    "app_claim": "still/seated webcam rPPG only",
    "not_supported_by_nb13": "exercise, high-motion, or ECG-Fitness/high-HR robustness"
  },
  "policy_thresholds": {
    "minimum_val_gain_bpm": 0.2,
    "minimum_test_gain_bpm": 0.1,
    "maximum_dataset_test_regression_bpm": 0.3,
    "maximum_ubfc_phys_test_regression_bpm": 0.1,
    "maximum_p90_regression_bpm": 0.25,
    "maximum_severe_error_rate_increase": 0.01,
    "maximum_abs_signed_bias_increase_bpm": 0.25
  },
  "main_candidate_count": 3,
  "adoptable_main_candidate_count": 0,
  "best_main_by_validation": {
    "candidate": "transfer_main_all_transformer_freq_head_lr2e5",
    "branch_family": "transfer",
    "run_kind": "main",
    "val_delta_mae_mean": -0.1553,
    "test_delta_mae_mean": -0.1366,
    "val_delta_mae_p90": 0.1194,
    "test_delta_mae_p90":

### NB13 conclusion: no app checkpoint replacement

This notebook tested whether excluding ECG-Fitness from model selection would produce a stronger app-relevant CRVSE PhysFormer candidate for still/seated webcam rPPG. The app-relevant scope included MCD-rPPG, UBFC-rPPG, and UBFC-Phys only. ECG-Fitness was intentionally excluded because NB13 does not claim exercise, high-motion, or exercise-related high-HR robustness.

The transfer-learning branches produced modest app-relevant gains compared with the frozen source-FPS checkpoint. The best validation candidate was the all-transformer transfer branch, but its validation MAE gain did not reach the predefined minimum improvement threshold. The best held-out test candidate was the last-block transfer branch, but it also failed the validation-gain threshold. Both transfer candidates are therefore analysis-only weak candidates, not app checkpoint replacements.

The scratch-training branch became a real training result after target-mean regression-head initialization, because its predictions no longer collapsed to the 40 BPM clamp. However, it remained worse than the frozen checkpoint on overall validation and held-out test performance. It improved UBFC-Phys test performance, but harmed MCD-rPPG and UBFC-rPPG enough that it was rejected by the NB13 policy.

The NB13 model-selection result is therefore conservative: no trained NB13 candidate should replace the frozen source-FPS checkpoint in the live app. The live demo should continue to present spectral consensus HR as the primary estimate, with CRVSE model HR treated as experimental. NB13 supports a narrower scientific claim: app-relevant transfer learning can produce small improvements on still/seated rPPG datasets, but the improvement was not large or stable enough to justify checkpoint adoption.

In [14]:
NB13_FINAL_DIR = Path(NB13_WORKING_DIR)

leaderboard_path = NB13_FINAL_DIR / "nb13_candidate_leaderboard.csv"
dataset_delta_path = NB13_FINAL_DIR / "nb13_test_dataset_delta_table.csv"
selection_summary_path = NB13_FINAL_DIR / "nb13_model_selection_summary.json"

if not leaderboard_path.exists():
    raise FileNotFoundError(f"Missing candidate leaderboard: {leaderboard_path}")

if not dataset_delta_path.exists():
    raise FileNotFoundError(f"Missing dataset delta table: {dataset_delta_path}")

if not selection_summary_path.exists():
    raise FileNotFoundError(f"Missing model selection summary: {selection_summary_path}")

nb13_leaderboard = pd.read_csv(leaderboard_path)
nb13_dataset_delta = pd.read_csv(dataset_delta_path)
nb13_selection_summary = json.loads(selection_summary_path.read_text(encoding="utf-8"))

main_candidates = nb13_leaderboard[
    nb13_leaderboard["run_kind"].astype(str).eq("main")
].copy()

adoptable_candidates = main_candidates[
    main_candidates["adoptable_by_nb13_policy"].astype(bool)
].copy()

best_by_validation = (
    main_candidates.sort_values("val_delta_mae_mean").iloc[0].to_dict()
    if len(main_candidates)
    else {}
)

best_by_test = (
    main_candidates.sort_values("test_delta_mae_mean").iloc[0].to_dict()
    if len(main_candidates)
    else {}
)

final_recommendation = (
    "keep_frozen_sourcefps_checkpoint_for_app"
    if len(adoptable_candidates) == 0
    else "review_policy_passing_candidate_before_app_adoption"
)

final_status = (
    "no_nb13_candidate_adopted"
    if len(adoptable_candidates) == 0
    else "policy_passing_candidate_requires_manual_review"
)

nb13_final_conclusion = {
    "notebook": "NB13 app-relevant CRVSE training without ECG-Fitness",
    "final_status": final_status,
    "final_recommendation": final_recommendation,
    "scope": {
        "included_datasets": ["mcd_rppg", "ubfc_rppg", "ubfc_phys"],
        "excluded_dataset": "ecg_fitness",
        "supported_app_scope": "still/seated webcam rPPG research demo",
        "not_supported_by_nb13": [
            "exercise robustness",
            "high-motion robustness",
            "ECG-Fitness robustness",
            "general high-HR robustness",
            "medical-device validation",
        ],
    },
    "baseline_checkpoint_action": "retain frozen source-FPS CRVSE PhysFormer checkpoint",
    "primary_app_estimate": "spectral consensus HR",
    "model_app_role": "experimental secondary estimate",
    "main_candidate_count": int(len(main_candidates)),
    "adoptable_main_candidate_count": int(len(adoptable_candidates)),
    "best_main_by_validation": {
        "candidate": best_by_validation.get("candidate"),
        "val_delta_mae_mean": best_by_validation.get("val_delta_mae_mean"),
        "test_delta_mae_mean": best_by_validation.get("test_delta_mae_mean"),
        "decision_status": best_by_validation.get("decision_status"),
        "scientific_label": best_by_validation.get("scientific_label"),
    },
    "best_main_by_test": {
        "candidate": best_by_test.get("candidate"),
        "val_delta_mae_mean": best_by_test.get("val_delta_mae_mean"),
        "test_delta_mae_mean": best_by_test.get("test_delta_mae_mean"),
        "decision_status": best_by_test.get("decision_status"),
        "scientific_label": best_by_test.get("scientific_label"),
    },
    "scientific_interpretation": (
        "Transfer learning produced modest app-relevant gains, but no main candidate "
        "passed the predefined NB13 adoption policy. Scratch training became a real "
        "training result after target-mean head-bias initialization, but remained worse "
        "than the frozen baseline overall. The frozen source-FPS checkpoint should "
        "remain the app checkpoint."
    ),
}

final_json_path = NB13_FINAL_DIR / "nb13_final_conclusion.json"
final_markdown_path = NB13_FINAL_DIR / "nb13_final_conclusion.md"

final_json_path.write_text(
    json.dumps(nb13_final_conclusion, indent=2),
    encoding="utf-8",
)

final_markdown = f"""# NB13 final conclusion

## Final recommendation

{final_recommendation}

## Scope

NB13 evaluated app-relevant still/seated webcam rPPG only.

Included datasets:

- MCD-rPPG
- UBFC-rPPG
- UBFC-Phys

Excluded dataset:

- ECG-Fitness

NB13 does not support claims about exercise robustness, high-motion robustness, ECG-Fitness robustness, general high-HR robustness, or medical-device validation.

## Model-selection result

Main candidates evaluated: {len(main_candidates)}
Adoptable main candidates: {len(adoptable_candidates)}

Best main candidate by validation:

- Candidate: {best_by_validation.get("candidate")}
- Validation MAE delta: {best_by_validation.get("val_delta_mae_mean")}
- Test MAE delta: {best_by_validation.get("test_delta_mae_mean")}
- Decision status: {best_by_validation.get("decision_status")}

Best main candidate by held-out test:

- Candidate: {best_by_test.get("candidate")}
- Validation MAE delta: {best_by_test.get("val_delta_mae_mean")}
- Test MAE delta: {best_by_test.get("test_delta_mae_mean")}
- Decision status: {best_by_test.get("decision_status")}

## App checkpoint decision

No NB13 candidate should replace the frozen source-FPS CRVSE PhysFormer checkpoint.

The live app should continue to use spectral consensus HR as the primary estimate. The CRVSE model output should remain an experimental secondary estimate.
"""

final_markdown_path.write_text(final_markdown, encoding="utf-8")

display_cols = [
    "candidate",
    "branch_family",
    "run_kind",
    "val_delta_mae_mean",
    "test_delta_mae_mean",
    "max_dataset_test_delta_mae",
    "ubfc_phys_test_delta_mae",
    "decision_status",
    "scientific_label",
    "adoptable_by_nb13_policy",
]

print("NB13 final candidate summary:")
display(nb13_leaderboard[display_cols])

print("\nNB13 final conclusion:")
print(json.dumps(nb13_final_conclusion, indent=2))

print(f"\nSaved final NB13 conclusion JSON: {final_json_path}")
print(f"Saved final NB13 conclusion Markdown: {final_markdown_path}")

if len(adoptable_candidates) == 0:
    print("\nPASS: NB13 is complete. Frozen source-FPS checkpoint remains the app checkpoint.")
else:
    print("\nATTENTION: A candidate passed policy gates and needs manual review before adoption.")

NB13 final candidate summary:


,candidate,branch_family,run_kind,val_delta_mae_mean,test_delta_mae_mean,max_dataset_test_delta_mae,ubfc_phys_test_delta_mae,decision_status,scientific_label,adoptable_by_nb13_policy
0,transfer_main_all_transformer_freq_head_lr2e5,transfer,main,-0.1553,-0.1366,0.0582,-0.1685,weak_or_incomplete_candidate,analysis_only_weak_candidate,False
1,transfer_main_last_block_freq_head_lr5e5,transfer,main,-0.1237,-0.1554,0.0808,-0.1661,weak_or_incomplete_candidate,analysis_only_weak_candidate,False
2,scratch_main_physformer_mean_bias_lr3e4,scratch,main,0.5808,0.2500,0.6445,-1.0386,reject,rejected_candidate,False
3,transfer_smoke_last_block_freq_head,transfer,smoke,-0.0917,-0.1230,0.1326,-0.1582,weak_or_incomplete_candidate,smoke_test_only,False
4,scratch_smoke_physformer,scratch,smoke,34.1342,34.0126,54.8608,28.5621,reject,smoke_test_only,False



NB13 final conclusion:
{
  "notebook": "NB13 app-relevant CRVSE training without ECG-Fitness",
  "final_status": "no_nb13_candidate_adopted",
  "final_recommendation": "keep_frozen_sourcefps_checkpoint_for_app",
  "scope": {
    "included_datasets": [
      "mcd_rppg",
      "ubfc_rppg",
      "ubfc_phys"
    ],
    "excluded_dataset": "ecg_fitness",
    "supported_app_scope": "still/seated webcam rPPG research demo",
    "not_supported_by_nb13": [
      "exercise robustness",
      "high-motion robustness",
      "ECG-Fitness robustness",
      "general high-HR robustness",
      "medical-device validation"
    ]
  },
  "baseline_checkpoint_action": "retain frozen source-FPS CRVSE PhysFormer checkpoint",
  "primary_app_estimate": "spectral consensus HR",
  "model_app_role": "experimental secondary estimate",
  "main_candidate_count": 3,
  "adoptable_main_candidate_count": 0,
  "best_main_by_validation": {
    "candidate": "transfer_main_all_transformer_freq_head_lr2e5",
    "val_delt